# [dbo].[ActualizarCamposEnProcedimientos]

In [0]:
USE [PETSHOMEDB]
GO


In [0]:
/****** Object:  StoredProcedure [dbo].[ActualizarCamposEnProcedimientos]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
-- =============================================
-- Procedimiento para actualizar campos en múltiples procedimientos
-- SQL Server
-- =============================================

CREATE   PROCEDURE [dbo].[ActualizarCamposEnProcedimientos]
    @patron NVARCHAR(100),
    @campo_antiguo NVARCHAR(100),
    @campo_nuevo NVARCHAR(100),
    @ejecutar BIT = 0  -- 0 = solo genera script, 1 = ejecuta cambios
AS
BEGIN
    SET NOCOUNT ON;
    
    DECLARE @proc_name NVARCHAR(255);
    DECLARE @proc_definition NVARCHAR(MAX);
    DECLARE @nueva_definition NVARCHAR(MAX);
    DECLARE @sql NVARCHAR(MAX);
    DECLARE @count INT = 0;
    DECLARE @schema_name NVARCHAR(128);
    
    -- Tabla temporal para almacenar scripts
    IF OBJECT_ID('tempdb..#ScriptsActualizacion') IS NOT NULL
        DROP TABLE #ScriptsActualizacion;
    
    CREATE TABLE #ScriptsActualizacion (
        id INT IDENTITY(1,1) PRIMARY KEY,
        schema_name NVARCHAR(128),
        procedure_name NVARCHAR(255),
        script_completo NVARCHAR(MAX),
        contiene_campo BIT,
        ejecutado BIT DEFAULT 0,
        mensaje NVARCHAR(500)
    );
    
    -- Cursor para recorrer procedimientos que contengan el campo
    DECLARE cur_procedures CURSOR FOR
        SELECT 
            SCHEMA_NAME(p.schema_id) AS schema_name,
            p.name AS proc_name,
            m.definition
        FROM sys.procedures p
        INNER JOIN sys.sql_modules m ON p.object_id = m.object_id
        WHERE p.name LIKE @patron
          AND m.definition LIKE '%' + @campo_antiguo + '%' COLLATE SQL_Latin1_General_CP1_CI_AS
        ORDER BY p.name;
    
    OPEN cur_procedures;
    
    FETCH NEXT FROM cur_procedures INTO @schema_name, @proc_name, @proc_definition;
    
    WHILE @@FETCH_STATUS = 0
    BEGIN
        -- Reemplazar el campo antiguo por el nuevo
        SET @nueva_definition = REPLACE(@proc_definition, @campo_antiguo, @campo_nuevo);
        
        -- Limpiar la definición (quitar CREATE PROCEDURE inicial si existe)
        -- porque vamos a recrearlo
        IF @nueva_definition LIKE '%CREATE%PROCEDURE%'
        BEGIN
            SET @nueva_definition = SUBSTRING(
                @nueva_definition, 
                CHARINDEX('PROCEDURE', @nueva_definition),
                LEN(@nueva_definition)
            );
            SET @nueva_definition = 'CREATE ' + @nueva_definition;
        END
        
        -- Generar script completo
        SET @sql = 
            '-- ============================================' + CHAR(13) + CHAR(10) +
            '-- Procedimiento: ' + @schema_name + '.' + @proc_name + CHAR(13) + CHAR(10) +
            '-- Reemplazando: ' + @campo_antiguo + ' -> ' + @campo_nuevo + CHAR(13) + CHAR(10) +
            '-- ============================================' + CHAR(13) + CHAR(10) +
            'DROP PROCEDURE IF EXISTS [' + @schema_name + '].[' + @proc_name + '];' + CHAR(13) + CHAR(10) +
            'GO' + CHAR(13) + CHAR(10) +
            @nueva_definition + CHAR(13) + CHAR(10) +
            'GO' + CHAR(13) + CHAR(10) + CHAR(13) + CHAR(10);
        
        INSERT INTO #ScriptsActualizacion (schema_name, procedure_name, script_completo, contiene_campo)
        VALUES (@schema_name, @proc_name, @sql, 1);
        
        -- Si se solicita ejecutar, hacerlo
        IF @ejecutar = 1
        BEGIN
            BEGIN TRY
                -- Eliminar procedimiento existente
                SET @sql = 'DROP PROCEDURE [' + @schema_name + '].[' + @proc_name + ']';
                EXEC sp_executesql @sql;
                
                -- Crear procedimiento modificado
                EXEC sp_executesql @nueva_definition;
                
                UPDATE #ScriptsActualizacion 
                SET ejecutado = 1, mensaje = 'Actualizado correctamente'
                WHERE procedure_name = @proc_name AND schema_name = @schema_name;
                
                PRINT 'Procedimiento actualizado: ' + @schema_name + '.' + @proc_name;
                SET @count = @count + 1;
            END TRY
            BEGIN CATCH
                UPDATE #ScriptsActualizacion 
                SET ejecutado = 0, mensaje = 'ERROR: ' + ERROR_MESSAGE()
                WHERE procedure_name = @proc_name AND schema_name = @schema_name;
                
                PRINT 'ERROR en ' + @schema_name + '.' + @proc_name + ': ' + ERROR_MESSAGE();
            END CATCH
        END
        
        FETCH NEXT FROM cur_procedures INTO @schema_name, @proc_name, @proc_definition;
    END
    
    CLOSE cur_procedures;
    DEALLOCATE cur_procedures;
    
    -- Mostrar resultados
    SELECT 
        schema_name + '.' + procedure_name AS [Procedimiento],
        CASE WHEN contiene_campo = 1 THEN 'SÍ' ELSE 'NO' END AS [Contiene Campo],
        CASE WHEN ejecutado = 1 THEN 'SÍ' ELSE 'NO' END AS [Ejecutado],
        mensaje AS [Resultado],
        script_completo AS [Script Generado]
    FROM #ScriptsActualizacion
    ORDER BY id;
    
    -- Resumen
    SELECT 
        COUNT(*) AS [Total Procedimientos con el Campo],
        SUM(CASE WHEN ejecutado = 1 THEN 1 ELSE 0 END) AS [Procedimientos Actualizados Exitosamente],
        SUM(CASE WHEN ejecutado = 0 AND @ejecutar = 1 THEN 1 ELSE 0 END) AS [Procedimientos con Error]
    FROM #ScriptsActualizacion;
    
    -- Si solo se generó el script, informar
    IF @ejecutar = 0
    BEGIN
        PRINT '';
        PRINT '================================================';
        PRINT 'MODO VISTA PREVIA - No se realizaron cambios';
        PRINT 'Revisa los scripts generados arriba';
        PRINT 'Para ejecutar los cambios, usa: @ejecutar = 1';
        PRINT '================================================';
    END
    ELSE
    BEGIN
        PRINT '';
        PRINT '================================================';
        PRINT 'CAMBIOS EJECUTADOS';
        PRINT 'Procedimientos actualizados: ' + CAST(@count AS NVARCHAR(10));
        PRINT '================================================';
    END
    
    DROP TABLE #ScriptsActualizacion;
END
GO


# [dbo].[Generador]

In [0]:
/****** Object:  StoredProcedure [dbo].[Generador]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [dbo].[Generador]
  @tableName AS VARCHAR(100)  
AS  
  
--CAPITALIZE TABLENAME  
SET @tableName = UPPER(LEFT(@tableName,1)) + RIGHT(@tableName, LEN(@tableName) -1)	
  
--SALTO DE LÍNEA  
DECLARE @nl AS CHAR  
SET @nl = CHAR(10) + CHAR(13)   
  
--CABECERA  
DECLARE @spHeaders AS VARCHAR(1000)    
SET @spHeaders = ''
--'SET ANSI_NULLS ON' + @nl +  
--'GO' + @nl +  
--'SET QUOTED_IDENTIFIER ON' + @nl +  
--'GO' + @nl
  
DECLARE @table AS VARCHAR(MAX)  
DECLARE @column AS VARCHAR(MAX)  
DECLARE @data_type AS VARCHAR(MAX)  
DECLARE @length AS INT  
DECLARE @precision AS INT  
DECLARE @scale AS INT  
  
--PARÁMETROS  
DECLARE @spParameters AS VARCHAR(MAX) SET @spParameters = ''  
  
--LISTA DE CAMPOS  
DECLARE @fieldList AS VARCHAR(MAX) SET @fieldList = ''  
  
--LISTA DE CAMPOS PARA EL SET DEL UPDATE  
DECLARE @fieldSetList AS VARCHAR(MAX) SET @fieldSetList = ''  
  
--LISTA DE PARÁMETROS PARA EL INSERT  
DECLARE @insertParameters AS VARCHAR(MAX) SET @insertParameters = ''  



DECLARE @sinId AS VARCHAR(MAX) SET @sinId = ''  

--DECLARE @spParametersFirst AS VARCHAR(MAX) SET @spParametersFirst = ''  
  
--CONDICIONES  
DECLARE @spConditions AS VARCHAR(MAX) SET @spConditions = ''  
DECLARE @Id AS VARCHAR(MAX) SET @Id = ''  
  
DECLARE c CURSOR STATIC FOR  
select table_name, column_name, data_type, character_maximum_length,numeric_precision, numeric_scale from information_schema.columns where table_name = @tableName order by ordinal_position  
OPEN c FETCH NEXT FROM c INTO @table, @column, @data_type, @length, @precision, @scale  
WHILE @@FETCH_STATUS = 0 BEGIN  
  
 --SET @spParametersFirst = @spParametersFirst + 
 
 --(CASE WHEN LEN(@spParametersFirst) > 0 
 --THEN @nl + ' ,' ELSE '  ' END) +  @column + ' ' + UPPER(@data_type) + 
 
 --SET @spParameters = @spParameters + 
 --(CASE WHEN LEN(@spParameters) >0 
	--THEN @nl + ' ,' 
	--ELSE '  ' 
	--END) 
	--+ '@V' + @column + ' ' + UPPER(@data_type) + 
	--(CASE @data_type 
	--	WHEN 'VARCHAR' 
	--	THEN '('+CAST(@length AS VARCHAR)+')' WHEN 'DECIMAL' THEN '('+CAST(@precision AS VARCHAR)+', '+CAST(@scale AS VARCHAR)+')' ELSE '' END)-- + ' = NULL'  
 
 SET @spParameters = @spParameters + (CASE WHEN LEN(@spParameters) >0 THEN @nl + ' ,' ELSE '  ' END) + '@' + @column + ' ' + UPPER(@data_type) + (CASE @data_type WHEN 'VARCHAR' THEN '('+CAST(@length AS VARCHAR)+')' WHEN 'NVARCHAR' THEN '('+CAST(@length AS VARCHAR)+')' WHEN 'NCHAR' THEN '('+CAST(@length AS VARCHAR)+')' WHEN 'CHAR' THEN '('+CAST(@length AS VARCHAR)+')' WHEN 'VARBINARY' THEN '('+CAST(@length AS VARCHAR)+')'  WHEN 'DECIMAL' THEN '('+CAST(@precision AS VARCHAR)+', '+CAST(@scale AS VARCHAR)+')' ELSE '' END)-- + ' = NULL'  
 SET @fieldList = @fieldList + (CASE WHEN LEN(@fieldList) >0 THEN @nl + '            ,' ELSE '' END) + @column  
 SET @spConditions = @spConditions + (CASE WHEN LEN(@spConditions) <= 1 
 THEN @column + ' = @'+ @column  ELSE ''   END)

  SET @Id = @Id + (CASE WHEN LEN(@Id) <= 1 
 THEN @column ELSE ''   END)

   SET @sinId = @sinId + (CASE WHEN LEN(@sinId) > 1 
 THEN @column ELSE ''   END)

 SET @fieldSetList = @fieldSetList + (CASE WHEN LEN(@fieldSetList) > 0 THEN @nl + '     ,' ELSE '      ' END) + @column + ' = @' + @column  
 SET @sinId = @sinId + (CASE WHEN LEN(@sinId) > 2 THEN @nl + '     ,' ELSE '      ' END) + @column + ' = @' + @column  
 SET @insertParameters = @insertParameters + (CASE WHEN LEN(@insertParameters) >0 THEN @nl + '    ,' ELSE '' END) + '@' + @column  
  
 FETCH NEXT FROM c INTO @table, @column, @data_type, @length, @precision, @scale  
END  
CLOSE c DEALLOCATE c  

 DECLARE @PROCEDIMIENTOSTITLE AS VARCHAR(MAX)  
SET @PROCEDIMIENTOSTITLE = '/* ======== PROCEDIMIENTOS ' + UPPER(@tableName) + ' ======== */' + @nl 
PRINT + @PROCEDIMIENTOSTITLE
--********************************  
--*********** SEAR *************	
--********************************  
--DECLARE @SELECT AS VARCHAR(MAX)  
--SET @SELECT = @spHeaders + @nl  
--SET @SELECT = @SELECT + 'CREATE PROCEDURE UDP_' + SUBSTRING(@tableName,3,50) + '_Sear' + @nl  
--SET @SELECT = @SELECT + '  @Buscar nvarchar(100)' + @nl  
--SET @SELECT = @SELECT + 'AS' + @nl	
--SET @SELECT = @SELECT + 'BEGIN' + @nl  
--SET @SELECT = @SELECT + '    SELECT ' + @fieldList + @nl  
--SET @SELECT = @SELECT + '    FROM ' + @table + @nl  
--SET @SELECT = @SELECt + '    WHERE ' + @spConditions+ ' LIKE "%" + @Buscar + "%"' + @nl  
--SET @SELECt = @SELECt + 'END' + @nl  

--********************************  
--*********** UPDATE *************  
--********************************  
DECLARE @UPDATE AS VARCHAR(MAX)  
SET @UPDATE = @spHeaders + @nl  
SET @UPDATE = @UPDATE + 'CREATE PROCEDURE UDP_' + SUBSTRING(@tableName,3,50) + '_Update' + @nl  
SET @UPDATE = @UPDATE + @spParameters + @nl  
SET @UPDATE = @UPDATE + 'AS' + @nl 
SET @UPDATE = @UPDATE + 'BEGIN' + @nl 
SET @UPDATE = @UPDATE + '		BEGIN TRANSACTION' + @nl  
SET @UPDATE = @UPDATE + '			BEGIN TRY' + @nl
SET @UPDATE = @UPDATE + '			UPDATE ' + @table + @nl  
SET @UPDATE = @UPDATE +	'			SET ' + @fieldSetList + @nl  
SET @UPDATE = @UPDATE + '    WHERE ' + @spConditions + @nl  
SET @UPDATE = @UPDATE + '			COMMIT TRANSACTION' + @nl
SET @UPDATE = @UPDATE + '		END TRY' + @nl
SET @UPDATE = @UPDATE + '		BEGIN CATCH' + @nl
SET @UPDATE = @UPDATE + '			ROLLBACK TRANSACTION' + @nl
SET @UPDATE = @UPDATE + '		END CATCH' + @nl
SET @UPDATE = @UPDATE + 'END' + @nl  

--********************************  
--*********** DELETE *************  
--********************************  
DECLARE @DELETE AS VARCHAR(MAX)  
SET @DELETE = @spHeaders + @nl  
SET @DELETE = @DELETE + 'CREATE PROCEDURE UDP_' + SUBSTRING(@tableName,3,50) + '_Delete' + @nl  
SET @DELETE = @DELETE + '@' + @Id+' INT' + @nl
SET @DELETE = @DELETE + 'AS' + @nl  
SET @DELETE = @DELETE + 'BEGIN' + @nl  
SET @DELETE = @DELETE + '		BEGIN TRANSACTION' + @nl  
SET @DELETE = @DELETE + '			BEGIN TRY' + @nl  
SET @DELETE = @DELETE + '				UPDATE ' + @table  + @nl  
SET @DELETE = @DELETE + '				 SET	esEliminado = 1' + @nl  
SET @DELETE = @DELETE + '				WHERE ' + @spConditions + @nl  
SET @DELETE = @DELETE + '			COMMIT TRANSACTION' + @nl
SET @DELETE = @DELETE + '		END TRY' + @nl
SET @DELETE = @DELETE + '		BEGIN CATCH' + @nl
SET @DELETE = @DELETE + '			ROLLBACK TRANSACTION' + @nl
SET @DELETE = @DELETE + '		END CATCH' + @nl
--SET @DELETE = @DELETE + 'WHERE       ' + @fieldSetList + ' = @Delete' + @nl 
SET @DELETE = @DELETE + 'END' + @nl  
  
--********************************  
--*********** INSERT *************  
--********************************  
DECLARE @INSERT AS VARCHAR(MAX)  
SET @INSERT = @spHeaders + @nl  
SET @INSERT = @INSERT + 'CREATE PROCEDURE UDP_' + SUBSTRING(@tableName,3,50) + '_Insert' + @nl  
SET @INSERT = @INSERT + @spParameters + @nl  
SET @INSERT = @INSERT + 'AS' + @nl 
SET @INSERT = @INSERT + 'BEGIN' + @nl  
SET @INSERT = @INSERT + '		BEGIN TRANSACTION' + @nl  
SET @INSERT = @INSERT + '			BEGIN TRY' + @nl  
SET @INSERT = @INSERT + '			INSERT INTO ' + @table + @nl  --+ '(' + @nl  
SET @INSERT = @INSERT + '            (' + @nl  
SET @INSERT = @INSERT + '            ' + @fieldList + @nl  
SET @INSERT = @INSERT + '            )' + @nl + '      VALUES(' + @nl + '     ' + @insertParameters + @nl  
SET @INSERT = @INSERT + '            )' + @nl  
SET @INSERT = @INSERT + '			COMMIT TRANSACTION' + @nl
SET @INSERT = @INSERT + '		END TRY' + @nl
SET @INSERT = @INSERT + '		BEGIN CATCH' + @nl
SET @INSERT = @INSERT + '			ROLLBACK TRANSACTION' + @nl
SET @INSERT = @INSERT + '		END CATCH' + @nl
SET @INSERT = @INSERT + 'END' + @nl   

--********************************  
--*********** EXISTS *************  
--********************************  
--DECLARE @EXISTS AS VARCHAR(MAX)  
--SET @EXISTS = @spHeaders + @nl  
--SET @EXISTS = @EXISTS + 'CREATE PROCEDURE ' + @tableName + '_Exists' + @nl  
--SET @EXISTS = @EXISTS + @spParameters + @nl  
--SET @EXISTS = @EXISTS + ' ,@exists BIT OUT' + @nl  
--SET @EXISTS = @EXISTS + 'AS' + @nl
--SET @EXISTS = @EXISTS + '    IF EXISTS (' + @nl + ' SELECT ' + LEFT(@fieldList,CHARINDEX(@nl,@fieldList))  
--SET @EXISTS = @EXISTS + '    FROM ' + @table + @nl  
--SET @EXISTS = @EXISTS + '    WHERE ' + @spConditions + @nl + ' )' + @nl  
--SET @EXISTS = @EXISTS + ' SET @exists = 1' + @nl + ' ELSE SET @exists = 0'  
  
--MOSTRAR GENERADOS  
--PRINT + '-----------------• SEAR' + @nl + @SELECT  
PRINT + '-----------------• INSERT' + @nl + @INSERT
PRINT + '-----------------• UPDATE' + @nl + @UPDATE  
PRINT + '-----------------• DELETE' + @nl + @DELETE  
--PRINT + '-----------------• EXISTS' + @nl + @EXISTS  
GO


# [dbo].[PR_Seguridad_GetUserPantallas]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_GetUserPantallas]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


-- =============================================
-- Stored Procedure: PR_Seguridad_GetUserPantallas  
-- Descripción: Obtiene solo las pantallas permitidas para un usuario (compatible con sistema actual)
-- =============================================

CREATE   PROCEDURE [dbo].[PR_Seguridad_GetUserPantallas]
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT DISTINCT
        mp.modpt_Descripcion AS Pantalla
    FROM seguridad.tbUsuarios u
    INNER JOIN seguridad.tbRolModulosPantallas rmp ON u.Rol_Id = rmp.rol_Id
    INNER JOIN seguridad.tbModulosPantallas mp ON rmp.modpt_Id = mp.modpt_Id
    WHERE u.usu_Id = @usu_Id
        AND u.Usu_EsActivo = 1
        AND ISNULL(u.Usu_Suspendido, 0) = 0
        AND ISNULL(u.Usu_EsEliminado, 0) = 0
        AND mp.modpt_EsActivo = 1
    ORDER BY mp.modpt_Descripcion;
END
GO


# [dbo].[PR_Seguridad_GetUserPermissions]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_GetUserPermissions]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
-- =============================================
-- Stored Procedure: PR_Seguridad_GetUserPermissions
-- Descripción: Obtiene todos los permisos de un usuario por pantalla para almacenar en sesión
-- =============================================

CREATE   PROCEDURE [dbo].[PR_Seguridad_GetUserPermissions]
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        mp.modpt_Descripcion AS Pantalla,
        p.Per_Nombre AS Permiso,
        m.Mod_Nombre AS Modulo,
        mp.modpt_Id
    FROM seguridad.tbUsuarios u
    INNER JOIN seguridad.tbRolModulosPantallas rmp ON u.Rol_Id = rmp.rol_Id
    INNER JOIN seguridad.tbModulosPantallas mp ON rmp.modpt_Id = mp.modpt_Id
    INNER JOIN seguridad.tbModulos m ON mp.mod_Id = m.Mod_Id
    INNER JOIN seguridad.tbRolModuloPermisos rmp2 ON u.Rol_Id = rmp2.Rol_Id AND m.Mod_Id = rmp2.Mod_Id
    INNER JOIN seguridad.tbPermisos p ON rmp2.Per_Id = p.Per_Id
    WHERE u.usu_Id = @usu_Id
        AND u.Usu_EsActivo = 1
        AND ISNULL(u.Usu_Suspendido, 0) = 0
        AND ISNULL(u.Usu_EsEliminado, 0) = 0
        AND mp.modpt_EsActivo = 1
        AND m.Mod_EsActivo = 1
        AND p.Per_EsActivo = 1
    ORDER BY m.Mod_Orden, mp.modpt_Descripcion, p.Per_Nombre;
END
GO


# [dbo].[PR_Seguridad_MenuUsuario_List]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_MenuUsuario_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Obtener menú para usuario específico
CREATE   PROCEDURE [dbo].[PR_Seguridad_MenuUsuario_List]
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        u.usu_Id,
        u.Rol_Id,
        r.Rol_Descripcion,
        m.Mod_Id,
        m.Mod_Nombre,
        m.Mod_Descripcion,
        m.Mod_Icono,
        m.Mod_Url,
        m.Mod_Orden,
        ISNULL(STRING_AGG(p.Per_Nombre, ','), '') as Permisos
    FROM seguridad.tbUsuarios u
    INNER JOIN seguridad.tbRoles r ON u.Rol_Id = r.Rol_Id
    INNER JOIN seguridad.tbRolModulos rm ON u.Rol_Id = rm.Rol_Id
    INNER JOIN seguridad.tbModulos m ON rm.Mod_Id = m.Mod_Id
    LEFT JOIN seguridad.tbRolModuloPermisos rmp ON u.Rol_Id = rmp.Rol_Id AND m.Mod_Id = rmp.Mod_Id
    LEFT JOIN seguridad.tbPermisos p ON rmp.Per_Id = p.Per_Id
    WHERE u.usu_Id = @usu_Id
        AND u.Usu_EsActivo = 1
        AND u.Usu_EsEliminado = 0
        AND r.Rol_EsActivo = 1
        AND m.Mod_EsActivo = 1
    GROUP BY u.usu_Id, u.Rol_Id, r.Rol_Descripcion, m.Mod_Id, m.Mod_Nombre, 
             m.Mod_Descripcion, m.Mod_Icono, m.Mod_Url, m.Mod_Orden
    ORDER BY m.Mod_Orden, m.Mod_Nombre;
END
GO


# [dbo].[PR_Seguridad_MenuUsuarioCompleto_List]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_MenuUsuarioCompleto_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

-- Crear procedimiento mejorado para obtener menú completo con submódulos
CREATE   PROCEDURE [dbo].[PR_Seguridad_MenuUsuarioCompleto_List]
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    -- Obtener módulos principales
    SELECT 
        u.usu_Id,
        u.Rol_Id,
        r.Rol_Descripcion,
        m.Mod_Id,
        m.Mod_Nombre,
        m.Mod_Descripcion,
        m.Mod_Icono,
        m.Mod_Url,
        m.Mod_Orden,
        ISNULL(STRING_AGG(p.Per_Nombre, ','), '') as Permisos,
        'MODULE' as TipoItem,
        0 as Mod_Padre
    FROM seguridad.tbUsuarios u
    INNER JOIN seguridad.tbRoles r ON u.Rol_Id = r.Rol_Id
    INNER JOIN seguridad.tbRolModulos rm ON u.Rol_Id = rm.Rol_Id
    INNER JOIN seguridad.tbModulos m ON rm.Mod_Id = m.Mod_Id
    LEFT JOIN seguridad.tbRolModuloPermisos rmp ON u.Rol_Id = rmp.Rol_Id AND m.Mod_Id = rmp.Mod_Id
    LEFT JOIN seguridad.tbPermisos p ON rmp.Per_Id = p.Per_Id
    WHERE u.usu_Id = @usu_Id
        AND u.Usu_EsActivo = 1
        AND u.Usu_EsEliminado = 0
        AND r.Rol_EsActivo = 1
        AND m.Mod_EsActivo = 1
    GROUP BY u.usu_Id, u.Rol_Id, r.Rol_Descripcion, m.Mod_Id, m.Mod_Nombre, 
             m.Mod_Descripcion, m.Mod_Icono, m.Mod_Url, m.Mod_Orden
    
    UNION ALL
    
    -- Obtener submódulos/pantallas
    SELECT 
        u.usu_Id,
        u.Rol_Id,
        r.Rol_Descripcion,
        mp.modpt_Id as Mod_Id,
        mp.modpt_Descripcion as Mod_Nombre,
        mp.modpt_Descripcion as Mod_Descripcion,
        mp.modpt_Icono as Mod_Icono,
        mp.modpt_Url as Mod_Url,
        mp.modpt_Orden as Mod_Orden,
        ISNULL(STRING_AGG(p.Per_Nombre, ','), '') as Permisos,
        'SUBMODULE' as TipoItem,
        m.Mod_Id as Mod_Padre
    FROM seguridad.tbUsuarios u
    INNER JOIN seguridad.tbRoles r ON u.Rol_Id = r.Rol_Id
    INNER JOIN seguridad.tbRolModulos rm ON u.Rol_Id = rm.Rol_Id
    INNER JOIN seguridad.tbModulos m ON rm.Mod_Id = m.Mod_Id
    INNER JOIN seguridad.tbModulosPantallas mp ON m.Mod_Id = mp.mod_Id
    LEFT JOIN seguridad.tbRolModuloPermisos rmp ON u.Rol_Id = rmp.Rol_Id AND m.Mod_Id = rmp.Mod_Id
    LEFT JOIN seguridad.tbPermisos p ON rmp.Per_Id = p.Per_Id
    WHERE u.usu_Id = @usu_Id
        AND u.Usu_EsActivo = 1
        AND u.Usu_EsEliminado = 0
        AND r.Rol_EsActivo = 1
        AND m.Mod_EsActivo = 1
        AND mp.modpt_EsActivo = 1
    GROUP BY u.usu_Id, u.Rol_Id, r.Rol_Descripcion, mp.modpt_Id, mp.modpt_Descripcion, 
             mp.modpt_Icono, mp.modpt_Url, mp.modpt_Orden, m.Mod_Id
    
    ORDER BY Mod_Padre, Mod_Orden, Mod_Nombre;
END
GO


# [dbo].[PR_Seguridad_Modulos_Insert]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_Modulos_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Crear módulo
CREATE   PROCEDURE [dbo].[PR_Seguridad_Modulos_Insert]
    @Mod_Nombre NVARCHAR(50),
    @Mod_Descripcion NVARCHAR(200),
    @Mod_Icono NVARCHAR(50),
    @Mod_Url NVARCHAR(200),
    @Mod_Orden INT
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        -- Verificar si el módulo ya existe
        IF EXISTS (SELECT 1 FROM seguridad.tbModulos WHERE Mod_Nombre = @Mod_Nombre)
        BEGIN
            SELECT -1 AS CodeErrorInsert, 'El nombre del módulo ya existe' AS MsgErrorInsert;
            RETURN;
        END
        
        INSERT INTO seguridad.tbModulos (
            Mod_Nombre,
            Mod_Descripcion,
            Mod_Icono,
            Mod_Url,
            Mod_Orden,
            Mod_EsActivo,
            Mod_FechaCreacion
        )
        VALUES (
            @Mod_Nombre,
            @Mod_Descripcion,
            @Mod_Icono,
            @Mod_Url,
            @Mod_Orden,
            1,
            GETDATE()
        );
        
        SELECT SCOPE_IDENTITY() AS Mod_Id, 0 AS CodeErrorInsert, '' AS MsgErrorInsert;
    END TRY
    BEGIN CATCH
        SELECT -2 AS CodeErrorInsert, ERROR_MESSAGE() AS MsgErrorInsert;
    END CATCH
END
GO


# [dbo].[PR_Seguridad_Modulos_List]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_Modulos_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Listar todos los módulos
CREATE   PROCEDURE [dbo].[PR_Seguridad_Modulos_List]
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        Mod_Id,
        Mod_Nombre,
        Mod_Descripcion,
        Mod_Icono,
        Mod_Url,
        Mod_Orden,
        Mod_EsActivo,
        Mod_FechaCreacion
    FROM seguridad.tbModulos
    ORDER BY Mod_Orden, Mod_Nombre;
END
GO


# [dbo].[PR_Seguridad_Modulos_Update]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_Modulos_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Actualizar módulo
CREATE   PROCEDURE [dbo].[PR_Seguridad_Modulos_Update]
    @Mod_Id INT,
    @Mod_Nombre NVARCHAR(50),
    @Mod_Descripcion NVARCHAR(200),
    @Mod_Icono NVARCHAR(50),
    @Mod_Url NVARCHAR(200),
    @Mod_Orden INT,
    @Mod_EsActivo BIT
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        UPDATE seguridad.tbModulos 
        SET Mod_Nombre = @Mod_Nombre,
            Mod_Descripcion = @Mod_Descripcion,
            Mod_Icono = @Mod_Icono,
            Mod_Url = @Mod_Url,
            Mod_Orden = @Mod_Orden,
            Mod_EsActivo = @Mod_EsActivo
        WHERE Mod_Id = @Mod_Id;
        
        SELECT 1 AS Success, 'Módulo actualizado correctamente' AS Message;
    END TRY
    BEGIN CATCH
        SELECT 0 AS Success, ERROR_MESSAGE() AS Message;
    END CATCH
END
GO


# [dbo].[PR_Seguridad_Permisos_Insert]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_Permisos_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Crear permiso
CREATE   PROCEDURE [dbo].[PR_Seguridad_Permisos_Insert]
    @Per_Nombre NVARCHAR(50),
    @Per_Descripcion NVARCHAR(200)
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        -- Verificar si el permiso ya existe
        IF EXISTS (SELECT 1 FROM seguridad.tbPermisos WHERE Per_Nombre = @Per_Nombre)
        BEGIN
            SELECT -1 AS CodeErrorInsert, 'El nombre del permiso ya existe' AS MsgErrorInsert;
            RETURN;
        END
        
        INSERT INTO seguridad.tbPermisos (
            Per_Nombre,
            Per_Descripcion,
            Per_EsActivo
        )
        VALUES (
            @Per_Nombre,
            @Per_Descripcion,
            1
        );
        
        SELECT SCOPE_IDENTITY() AS Per_Id, 0 AS CodeErrorInsert, '' AS MsgErrorInsert;
    END TRY
    BEGIN CATCH
        SELECT -2 AS CodeErrorInsert, ERROR_MESSAGE() AS MsgErrorInsert;
    END CATCH
END
GO


# [dbo].[PR_Seguridad_Permisos_List]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_Permisos_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Listar todos los permisos
CREATE   PROCEDURE [dbo].[PR_Seguridad_Permisos_List]
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        Per_Id,
        Per_Nombre,
        Per_Descripcion,
        Per_EsActivo
    FROM seguridad.tbPermisos
    WHERE Per_EsActivo = 1
    ORDER BY Per_Nombre;
END
GO


# [dbo].[PR_Seguridad_RolModuloPermisos_Delete]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_RolModuloPermisos_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Remover permiso específico de rol-módulo
CREATE   PROCEDURE [dbo].[PR_Seguridad_RolModuloPermisos_Delete]
    @Rol_Id INT,
    @Mod_Id INT,
    @Per_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        DELETE FROM seguridad.tbRolModuloPermisos 
        WHERE Rol_Id = @Rol_Id AND Mod_Id = @Mod_Id AND Per_Id = @Per_Id;
        
        SELECT 1 AS Success, 'Permiso removido correctamente' AS Message;
    END TRY
    BEGIN CATCH
        SELECT 0 AS Success, ERROR_MESSAGE() AS Message;
    END CATCH
END
GO


# [dbo].[PR_Seguridad_RolModuloPermisos_Insert]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_RolModuloPermisos_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Asignar permiso específico a rol-módulo
CREATE   PROCEDURE [dbo].[PR_Seguridad_RolModuloPermisos_Insert]
    @Rol_Id INT,
    @Mod_Id INT,
    @Per_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        -- Verificar que el rol tenga acceso al módulo
        IF NOT EXISTS (SELECT 1 FROM seguridad.tbRolModulos WHERE Rol_Id = @Rol_Id AND Mod_Id = @Mod_Id)
        BEGIN
            SELECT 0 AS Success, 'El rol no tiene acceso a este módulo' AS Message;
            RETURN;
        END
        
        -- Verificar si ya existe el permiso
        IF EXISTS (SELECT 1 FROM seguridad.tbRolModuloPermisos WHERE Rol_Id = @Rol_Id AND Mod_Id = @Mod_Id AND Per_Id = @Per_Id)
        BEGIN
            SELECT 1 AS Success, 'El permiso ya está asignado' AS Message;
            RETURN;
        END
        
        INSERT INTO seguridad.tbRolModuloPermisos (Rol_Id, Mod_Id, Per_Id, RolModPer_FechaAsignacion)
        VALUES (@Rol_Id, @Mod_Id, @Per_Id, GETDATE());
        
        SELECT 1 AS Success, 'Permiso asignado correctamente' AS Message;
    END TRY
    BEGIN CATCH
        SELECT 0 AS Success, ERROR_MESSAGE() AS Message;
    END CATCH
END
GO


# [dbo].[PR_Seguridad_RolModulos_Delete]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_RolModulos_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Remover módulo de rol
CREATE   PROCEDURE [dbo].[PR_Seguridad_RolModulos_Delete]
    @Rol_Id INT,
    @Mod_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        -- Primero eliminar todos los permisos específicos del módulo
        DELETE FROM seguridad.tbRolModuloPermisos 
        WHERE Rol_Id = @Rol_Id AND Mod_Id = @Mod_Id;
        
        -- Luego eliminar la asignación del módulo
        DELETE FROM seguridad.tbRolModulos 
        WHERE Rol_Id = @Rol_Id AND Mod_Id = @Mod_Id;
        
        SELECT 1 AS Success, 'Módulo removido correctamente' AS Message;
    END TRY
    BEGIN CATCH
        SELECT 0 AS Success, ERROR_MESSAGE() AS Message;
    END CATCH
END
GO


# [dbo].[PR_Seguridad_RolModulos_Insert]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_RolModulos_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Asignar módulo a rol
CREATE   PROCEDURE [dbo].[PR_Seguridad_RolModulos_Insert]
    @Rol_Id INT,
    @Mod_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        -- Verificar si ya existe la asignación
        IF EXISTS (SELECT 1 FROM seguridad.tbRolModulos WHERE Rol_Id = @Rol_Id AND Mod_Id = @Mod_Id)
        BEGIN
            SELECT 1 AS Success, 'La asignación ya existe' AS Message;
            RETURN;
        END
        
        INSERT INTO seguridad.tbRolModulos (Rol_Id, Mod_Id, RolMod_FechaAsignacion)
        VALUES (@Rol_Id, @Mod_Id, GETDATE());
        
        SELECT 1 AS Success, 'Módulo asignado correctamente' AS Message;
    END TRY
    BEGIN CATCH
        SELECT 0 AS Success, ERROR_MESSAGE() AS Message;
    END CATCH
END
GO


# [dbo].[PR_Seguridad_RolModulosCompleto_List]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_RolModulosCompleto_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Obtener permisos completos de un rol
CREATE   PROCEDURE [dbo].[PR_Seguridad_RolModulosCompleto_List]
    @Rol_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        r.Rol_Id,
        r.Rol_Descripcion,
        m.Mod_Id,
        m.Mod_Nombre,
        m.Mod_Descripcion,
        m.Mod_Icono,
        m.Mod_Url,
        m.Mod_Orden,
        CASE 
            WHEN rm.Rol_Id IS NOT NULL THEN 1 
            ELSE 0 
        END as TieneAcceso,
        ISNULL(STRING_AGG(p.Per_Nombre, ','), '') as Permisos
    FROM seguridad.tbRoles r
    CROSS JOIN seguridad.tbModulos m
    LEFT JOIN seguridad.tbRolModulos rm ON r.Rol_Id = rm.Rol_Id AND m.Mod_Id = rm.Mod_Id
    LEFT JOIN seguridad.tbRolModuloPermisos rmp ON r.Rol_Id = rmp.Rol_Id AND m.Mod_Id = rmp.Mod_Id
    LEFT JOIN seguridad.tbPermisos p ON rmp.Per_Id = p.Per_Id
    WHERE r.Rol_Id = @Rol_Id
        AND r.Rol_EsActivo = 1
        AND m.Mod_EsActivo = 1
    GROUP BY r.Rol_Id, r.Rol_Descripcion, m.Mod_Id, m.Mod_Nombre, m.Mod_Descripcion, 
             m.Mod_Icono, m.Mod_Url, m.Mod_Orden, rm.Rol_Id
    ORDER BY m.Mod_Orden, m.Mod_Nombre;
END
GO


# [dbo].[PR_Seguridad_Usuarios_Login]

In [0]:
/****** Object:  StoredProcedure [dbo].[PR_Seguridad_Usuarios_Login]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- Procedimiento para login de usuario
CREATE   PROCEDURE [dbo].[PR_Seguridad_Usuarios_Login] 
    @Usu_Nombre NVARCHAR(50),
    @Usu_PasswordHash NVARCHAR(255)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        u.usu_Id,
        u.Emp_Id,
        u.Usu_Nombre,
        e.emp_Codigo,
        u.Rol_Id,
        r.Rol_Descripcion,
        --r.Rol_Pantallas,
        u.Usu_EsActivo,
        u.Usu_Suspendido,
        u.Usu_PasswordHash,
        u.Usu_PasswordSalt
    FROM seguridad.tbUsuarios u
    INNER JOIN refugio.tbEmpleados e ON u.Emp_Id = e.emp_Id
    INNER JOIN seguridad.tbRoles r ON u.Rol_Id = r.Rol_Id
    WHERE u.Usu_Nombre = @Usu_Nombre 
        AND u.Usu_EsActivo = 1 
        AND ISNULL(u.Usu_Suspendido, 0) = 0
        AND ISNULL(u.Usu_EsEliminado, 0) = 0
        AND u.Usu_PasswordHash = @Usu_PasswordHash
        AND r.Rol_EsActivo = 1
        AND e.emp_EsActivo = 1;
END
GO


# [General].[PR_General_Departamentos_Delete]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Departamentos_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


CREATE PROCEDURE [General].[PR_General_Departamentos_Delete]
	@depto_Id	INT
AS
BEGIN
	UPDATE	[General].[tbDepartamentos]
	SET		[depto_EsEliminado]	= 1
	WHERE	[depto_Id]	= @depto_Id
END
GO


# [General].[PR_General_Departamentos_Detail]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Departamentos_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [General].[PR_General_Departamentos_Detail]
@depto_Id INT
AS
BEGIN
	SELECT	[departamento].[depto_Id],
			[departamento].[depto_Codigo],
			[departamento].[depto_Descripcion],
			[departamento].[depto_Capital],
			[departamento].[depto_Poblacion],
			[departamento].[depto_AreaKm2],
			[usuarioCrea].[usu_Nombre] AS [UsuarioCreacion],
			[departamento].[depto_FechaCrea],
			[usuarioModifica].[usu_Nombre] AS [UsuarioModificacion],
			[departamento].[depto_FechaModifica]
	FROM	[General].[tbDepartamentos] departamento 
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		[departamento].[depto_UsuarioCrea] = [usuarioCrea].[usu_Id] 
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		[departamento].[depto_UsuarioModifica] = [usuarioModifica].[usu_Id]
	WHERE	[depto_EsEliminado] = 0
	AND		[departamento].depto_Id = @depto_Id
END
GO


# [General].[PR_General_Departamentos_Dropdown]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Departamentos_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [General].[PR_General_Departamentos_Dropdown]
AS
BEGIN
	SELECT	depto_Id,
			depto_Descripcion
    FROM	[General].[tbDepartamentos] AS departamento
	WHERE	departamento.depto_EsEliminado != 1
END
GO


# [General].[PR_General_Departamentos_Find]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Departamentos_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [General].[PR_General_Departamentos_Find]
@depto_Id INT
AS
BEGIN
	SELECT	[departamento].[depto_Id],
			[departamento].[depto_Codigo],
			[departamento].[depto_Descripcion],
			[departamento].depto_UsuarioCrea,
			[departamento].depto_FechaCrea,
			[departamento].depto_UsuarioModifica,
			[departamento].depto_FechaModifica
	FROM	[General].[tbDepartamentos] departamento  
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		departamento.depto_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		departamento.depto_UsuarioModifica = usuarioModifica.usu_Id
	WHERE	[depto_EsEliminado] = 0
	AND		[departamento].depto_Id = @depto_Id
END
GO


# [General].[PR_General_Departamentos_Insert]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Departamentos_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [General].[PR_General_Departamentos_Insert]
	@depto_Codigo			 VARCHAR(2), 
	@depto_Descripcion		 NVARCHAR(100),
	@depto_UsuarioCrea		 INT
AS
BEGIN
	INSERT [General].[tbDepartamentos]
	(
		[depto_Codigo],
		[depto_Descripcion],
		[depto_UsuarioCrea], 
		[depto_FechaCrea]
	)
	VALUES 
	(
		@depto_Codigo, 
		@depto_Descripcion,
		@depto_UsuarioCrea, 
		GETDATE()
	)
	SELECT SCOPE_IDENTITY()
END
GO


# [General].[PR_General_Departamentos_List]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Departamentos_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [General].[PR_General_Departamentos_List]
AS
BEGIN
	SELECT	depto_Id, 
			depto_Codigo, 
			depto_Descripcion,
			[depto_Capital],
			[depto_Poblacion],
			[depto_AreaKm2]
	FROM 	[General].[tbDepartamentos]
	WHERE 	depto_EsEliminado != 1
END
GO


# [General].[PR_General_Departamentos_Update]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Departamentos_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [General].[PR_General_Departamentos_Update]
	@depto_Id				INT, 
	@depto_Codigo			VARCHAR(2), 
	@depto_Descripcion		NVARCHAR(100),
	@depto_UsuarioModifica	INT
AS
BEGIN
	UPDATE  [General].[tbDepartamentos]
	SET 	[depto_Codigo]			= @depto_Codigo,
			[depto_Descripcion]		= @depto_Descripcion,
			[depto_UsuarioModifica]	= @depto_UsuarioModifica, 
			[depto_FechaModifica]	= GETDATE()
	WHERE	[depto_Id]				= @depto_Id
END
GO


# [General].[PR_General_Municipios_Delete]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Municipios_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [General].[PR_General_Municipios_Delete]
	@mpio_Id INT
AS
BEGIN
	UPDATE	[General].[tbMunicipios]
	SET		[mpio_EsEliminado] = 1
	WHERE	[mpio_Id] = @mpio_Id
END
GO


# [General].[PR_General_Municipios_Detail]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Municipios_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [General].[PR_General_Municipios_Detail]
	@mpio_Id INT
AS
BEGIN
    SELECT  [mpio_Id],
            [mpio_Codigo],
            [mpio_Descripcion],
            SUBSTRING ( [mpio_Codigo] ,1 , 2 ) AS [codigoDept],
            SUBSTRING ( [mpio_Codigo] ,3 , 4 ) AS [codigo],
            [mpio_UsuarioCrea], 
            [mpio_FechaCrea], 
            [mpio_UsuarioModifica], 
            [mpio_FechaModifica]
    FROM    [General].[tbMunicipios] 
    WHERE   [mpio_Id] = @mpio_Id
	AND		[mpio_EsEliminado] != 1
END
GO


# [General].[PR_General_Municipios_Dropdown]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Municipios_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [General].[PR_General_Municipios_Dropdown]
AS
BEGIN
	SELECT	[mpio_Id],
			[mpio_Descripcion]
	FROM	[General].[tbMunicipios] AS municipio
	WHERE	municipio.mpio_EsEliminado != 1
END
GO


# [General].[PR_General_Municipios_Find]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Municipios_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [General].[PR_General_Municipios_Find]
	@mpio_Id INT
AS
BEGIN
    SELECT  [mpio_Id],
            [mpio_Codigo],
            [mpio_Descripcion],
            [depto_Id],
            SUBSTRING ( [mpio_Codigo] ,1 , 2 ) AS [codigoDept],
            SUBSTRING ( [mpio_Codigo] ,3 , 4 ) AS [codigo],
			usuarioCrea.Usu_Nombre AS [UsuarioCreacion],
			mpio_FechaCrea,
			usuarioCrea.Usu_Nombre,
			mpio_FechaModifica AS [UsuarioModificacion]
    FROM    [General].[tbMunicipios] AS municipios
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		municipios.mpio_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		municipios.mpio_UsuarioModifica = usuarioModifica.usu_Id
    WHERE   [mpio_EsEliminado] != 1
	AND		[mpio_Id] = @mpio_Id
END
GO


# [General].[PR_General_Municipios_Insert]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Municipios_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [General].[PR_General_Municipios_Insert]
	@mpio_Codigo		VARCHAR(4),
	@mpio_Descripcion	NVARCHAR(100),
	@depto_Id			INT,
	@mpio_UsuarioCrea	INT
AS
BEGIN
	INSERT INTO [General].[tbMunicipios]	
	(  [mpio_Codigo],
	   [mpio_Descripcion], 
	   [depto_Id],
	   [mpio_UsuarioCrea], 
       [mpio_FechaCrea]
	 )

	VALUES 
	(
	    @mpio_Codigo,
		@mpio_Descripcion,
		@depto_Id,  
		@mpio_UsuarioCrea,  
		GETDATE()
	)
END
GO


# [General].[PR_General_Municipios_List]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Municipios_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [General].[PR_General_Municipios_List]
AS
BEGIN
    SELECT  [mpio_Id],
            [mpio_Codigo],
            [mpio_Descripcion]
    FROM    [General].[tbMunicipios] 
    WHERE   [mpio_EsEliminado]    = 0
END
GO


# [General].[PR_General_Municipios_SelectbyDepartamento]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Municipios_SelectbyDepartamento]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [General].[PR_General_Municipios_SelectbyDepartamento]
	@depto_Id	INT
AS
BEGIN
 
     SELECT [mpio].[mpio_Id],
            [mpio].[mpio_Codigo],
            [mpio].[mpio_Descripcion],
            [mpio].[depto_Id],
            [depto].[depto_Descripcion],
			usuarioCrea.Usu_Nombre AS [UsuarioCreacion],
			mpio_FechaCrea,
			usuarioCrea.Usu_Nombre,
			mpio_FechaModifica AS [UsuarioModificacion]
     FROM   [General].[tbMunicipios]    AS  [mpio]		INNER JOIN
			[General].[tbDepartamentos] AS  [depto] ON  [mpio].[depto_Id] = [depto].[depto_Id]
			LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
			ON		[mpio].mpio_UsuarioCrea = usuarioCrea.usu_Id
			LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
			ON		[mpio].mpio_UsuarioModifica = usuarioModifica.usu_Id
     WHERE  [mpio_EsEliminado] = 0 AND [depto].[depto_Id] =@depto_Id

END
GO


# [General].[PR_General_Municipios_Update]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Municipios_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [General].[PR_General_Municipios_Update]
	@mpio_Id				INT,
	@mpio_Codigo			VARCHAR(4),
	@mpio_Descripcion		NVARCHAR(100),
	@mpio_UsuarioModifica	INT
AS
BEGIN
	UPDATE	[General].[tbMunicipios]
	SET		[mpio_Codigo] = @mpio_Codigo,
			[mpio_Descripcion] = @mpio_Descripcion,
			[mpio_UsuarioModifica] = @mpio_UsuarioModifica,
			[mpio_FechaModifica] = GETDATE()
	WHERE	[mpio_Id] = @mpio_Id
END
GO


# [General].[PR_General_Municipios_ValidacionUnique]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Municipios_ValidacionUnique]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [General].[PR_General_Municipios_ValidacionUnique]
    @mpio_Codigo        VARCHAR(4)
AS
BEGIN

    SELECT    [mpio_Id],
              [mpio_Codigo]
    FROM      [General].[tbMunicipios] AS mun
    WHERE     [mun].[mpio_Codigo] = @mpio_Codigo 
    AND       [mpio_EsEliminado] != 1
END
GO


# [General].[PR_General_Personas_Delete]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Personas_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


-----------------> DELETE
CREATE PROCEDURE [General].[PR_General_Personas_Delete] 
	@per_Id INT
AS
  BEGIN
          UPDATE [General].[tbPersonas]
          SET    per_EsEliminado	= 1
          WHERE  per_Id			= @per_Id
  END
GO


# [General].[PR_General_Personas_Insert]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Personas_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [General].[PR_General_Personas_Insert] 
    @per_Identidad VARCHAR(13),
	@per_PrimerNombre nvarchar(50),
	@per_SegundoNombre nvarchar(50),
	@per_ApellidoPaterno nvarchar(50),
	@per_ApellidoMaterno nvarchar(50),
    @per_FechaNacimiento DATE,
    @per_Domicilio NVARCHAR(100),
    @per_Telefono VARCHAR(8),
    @per_Correo NVARCHAR(150),
    @per_UsuarioCrea INT
AS BEGIN
INSERT INTO [General].[tbPersonas] (
		per_Identidad,
		per_PrimerNombre,
		per_SegundoNombre,
		per_ApellidoPaterno,
		per_ApellidoMaterno,
		per_FechaNacimiento,
		per_Domicilio,
		per_Telefono,
		per_Correo,
		per_UsuarioCrea,
		per_FechaCrea
    )
VALUES
    (
		@per_Identidad,
		@per_PrimerNombre,
		@per_SegundoNombre,
		@per_ApellidoPaterno,
		@per_ApellidoMaterno,
		@per_FechaNacimiento,
		@per_Domicilio,
		@per_Telefono,
		@per_Correo,
		@per_UsuarioCrea,
		GETDATE()
    )
END
GO


# [General].[PR_General_Personas_Update]

In [0]:
/****** Object:  StoredProcedure [General].[PR_General_Personas_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-----------------> UPDATE
CREATE PROCEDURE [General].[PR_General_Personas_Update] 
@per_Id	int,
@per_Identidad	varchar(13),
@per_PrimerNombre	nvarchar(50),
@per_SegundoNombre	nvarchar(50),
@per_ApellidoPaterno	nvarchar(50),
@per_ApellidoMaterno	nvarchar(50),
@per_FechaNacimiento	date,
@per_Domicilio	nvarchar(100),
@per_Telefono	varchar(8),
@per_Correo	varchar(150),
@per_UsuarioModifica	int
AS BEGIN
UPDATE [General].[tbPersonas]
SET per_Identidad = @per_Identidad,
	per_PrimerNombre = @per_PrimerNombre,
	per_SegundoNombre = @per_SegundoNombre,
	per_ApellidoPaterno = @per_ApellidoPaterno,
	per_ApellidoMaterno = @per_ApellidoMaterno,
	per_FechaNacimiento = @per_FechaNacimiento,
	per_Domicilio = @per_Domicilio,
	per_Telefono = @per_Telefono,
	per_Correo = @per_Correo,
	per_UsuarioModifica = @per_UsuarioModifica,
	per_FechaModifica = Getdate()
WHERE per_Id = @per_Id
END 

GO


# [Inventario].[PR_Inventario_Categorias_Delete]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Categorias_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO




CREATE PROCEDURE [Inventario].[PR_Inventario_Categorias_Delete] 
	@cat_Id INT
AS
  BEGIN
          UPDATE [Inventario].[tbCategorias]
          SET    cat_EsEliminado	= 1
          WHERE  cat_Id			= @cat_Id
  END
GO


# [Inventario].[PR_Inventario_Categorias_Detail]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Categorias_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbCategorias
CREATE PROCEDURE [Inventario].[PR_Inventario_Categorias_Detail]
AS
BEGIN
    SELECT 	cat_Id,
			cat_Descripcion,
			usuarioCrea.usu_Nombre AS UsuarioCreacion,
			cat_FechaCrea,
			usuarioModifica.usu_Nombre AS UsuarioModificacion,
			cat_FechaModifica
    FROM 	[Inventario].[tbCategorias] AS categorias
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		usuarioCrea.usu_Id = categorias.cat_UsuarioCrea
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		usuarioModifica.usu_Id= categorias.cat_UsuarioModifica
    WHERE   cat_EsEliminado != 1
END
GO


# [Inventario].[PR_Inventario_Categorias_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Categorias_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
------------------> tbCategorias
create PROCEDURE [Inventario].[PR_Inventario_Categorias_Dropdown]
AS
BEGIN
    SELECT 	cat_Id,
			cat_Descripcion
    FROM 	[Inventario].[tbCategorias]
    WHERE   cat_EsEliminado != 1
END
GO


# [Inventario].[PR_Inventario_Categorias_Find]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Categorias_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Inventario].[PR_Inventario_Categorias_Find]
@cat_Id INT
AS
BEGIN
    SELECT cat_Id, cat_Descripcion, cat_EsActivo,
           cat_UsuarioCrea,
           usuarioCrea.Usu_Nombre AS usuarioCrea,
           cat_FechaCrea,
           cat_UsuarioModifica,
           usuarioModifica.Usu_Nombre AS usuarioModifica,
           cat_FechaModifica
    FROM [Inventario].[tbCategorias] AS c
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON c.cat_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON c.cat_UsuarioModifica = usuarioModifica.usu_Id
    WHERE c.cat_EsEliminado != 1 AND c.cat_Id = @cat_Id
END
GO


# [Inventario].[PR_Inventario_Categorias_Insert]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Categorias_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Inventario].[PR_Inventario_Categorias_Insert]
    @cat_Descripcion VARCHAR(100),
    @cat_EsActivo BIT = 1,
    @cat_UsuarioCrea INT
AS
BEGIN
    INSERT INTO [Inventario].[tbCategorias] (cat_Descripcion, cat_EsActivo, cat_UsuarioCrea, cat_FechaCrea)
    VALUES (@cat_Descripcion, @cat_EsActivo, @cat_UsuarioCrea, GETDATE())
END
GO


# [Inventario].[PR_Inventario_Categorias_List]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Categorias_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Inventario].[PR_Inventario_Categorias_List]
AS
BEGIN
    SELECT cat_Id, cat_Descripcion,
           CASE WHEN cat_EsActivo = 1 THEN 'Activo' ELSE 'Inactivo' END AS cat_EsActivo
    FROM [Inventario].[tbCategorias]
    WHERE cat_EsEliminado != 1
END

GO


# [Inventario].[PR_Inventario_Categorias_Update]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Categorias_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Inventario].[PR_Inventario_Categorias_Update]
    @cat_Id INT,
    @cat_Descripcion VARCHAR(100),
    @cat_EsActivo BIT = 1,
    @cat_UsuarioModifica INT
AS
BEGIN
    UPDATE [Inventario].[tbCategorias]
    SET cat_Descripcion = @cat_Descripcion,
        cat_EsActivo = @cat_EsActivo,
        cat_UsuarioModifica = @cat_UsuarioModifica,
        cat_FechaModifica = GETDATE()
    WHERE cat_Id = @cat_Id
END
GO


# [Inventario].[PR_Inventario_Entradas_	List]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Entradas_	List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
------------------> tbInventarios
create PROCEDURE [Inventario].[PR_Inventario_Entradas_	List]
@ent_Id INT
AS
BEGIN
    SELECT 	entradas.ent_Id, 
			ent_Descripcion, 
			refugios.refg_Nombre, 
			ent_Fecha
    FROM 	[Inventario].[tbEntradas] AS entradas
	INNER JOIN [Refugio].[tbRefugios] AS refugios
	ON		entradas.refg_Id = refugios.refg_Id
    WHERE  	ent_EsEliminado != 1
	AND		entradas.ent_Id = @ent_Id
END

GO


# [Inventario].[PR_Inventario_Entradas_Delete]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Entradas_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
 




-----------------> DELETE

CREATE PROCEDURE [Inventario].[PR_Inventario_Entradas_Delete] 
	@ent_Id INT
AS
  BEGIN
          UPDATE [Inventario].[tbEntradas]
          SET    ent_EsEliminado	= 1
          WHERE  ent_Id			= @ent_Id
  END
GO


# [Inventario].[PR_Inventario_Entradas_Detail]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Entradas_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbInventarios
CREATE PROCEDURE [Inventario].[PR_Inventario_Entradas_Detail]
@ent_Id INT
AS
BEGIN
    SELECT 	entradas.ent_Id, 
			ent_Descripcion, 
			refugios.refg_Nombre, 
			SUM(entradasDetalles.entdet_Cantidad * items.itm_Precio) AS ent_SumaTotal,
			ent_Fecha, 
			usuarioCrea.usu_Nombre AS UsuarioCreacion, 
			ent_FechaCrea, 
			usuarioModifica.usu_Nombre AS UsuarioModificacion, 
			ent_FechaModifica 
    FROM 	[Inventario].[tbEntradas] AS entradas
	INNER JOIN [Refugio].[tbRefugios] AS refugios
	ON		entradas.refg_Id = refugios.refg_Id
	INNER JOIN [Inventario].[tbEntradasDetalles] AS entradasDetalles
	ON		entradasDetalles.ent_Id = entradas.ent_Id
	INNER JOIN [Inventario].[tbItems] AS items
	ON entradasDetalles.itm_Id = items.itm_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		entradas.ent_UsuarioCrea = usuarioCrea.usu_Id 
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		entradas.ent_UsuarioModifica = usuarioModifica.usu_Id
    WHERE  	ent_EsEliminado != 1
	AND		entradas.ent_Id = @ent_Id 
	GROUP BY entradas.ent_Id, 
			ent_Descripcion, 
			refugios.refg_Nombre,
			ent_Fecha, 
			usuarioCrea.usu_Nombre, 
			ent_FechaCrea, 
			usuarioModifica.usu_Nombre, 
			ent_FechaModifica 
END
GO


# [Inventario].[PR_Inventario_Entradas_Find]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Entradas_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbInventarios
CREATE PROCEDURE [Inventario].[PR_Inventario_Entradas_Find]
@ent_Id INT
AS
BEGIN
    SELECT 	entradas.ent_Id, 
			ent_Descripcion, 
			entradas.refg_Id,
			refugios.refg_Nombre, 
			ent_Fecha, 
			ent_UsuarioCrea,
			usuarioCrea.Usu_Nombre AS usuarioCrea, 
			ent_FechaCrea, 
			ent_UsuarioModifica,
			usuarioModifica.Usu_Nombre AS usuarioModifica, 
			ent_FechaModifica 
    FROM 	[Inventario].[tbEntradas] AS entradas
	INNER JOIN [Refugio].[tbRefugios] AS refugios
	ON		entradas.refg_Id = refugios.refg_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		entradas.ent_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		entradas.ent_UsuarioModifica = usuarioModifica.usu_Id
    WHERE  	ent_EsEliminado != 1
	AND		entradas.ent_Id = @ent_Id 
END
GO


# [Inventario].[PR_Inventario_Entradas_Insert]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Entradas_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Inventario].[PR_Inventario_Entradas_Insert]
@ent_Descripcion	nvarchar(250),
@refg_Id	int,
@ent_Fecha	datetime,
@ent_UsuarioCrea	int
	
AS
BEGIN
    INSERT INTO [Inventario].[tbEntradas]
    (
        ent_Descripcion,
        refg_Id,
        ent_Fecha,
        ent_UsuarioCrea,
        ent_FechaCrea
    )
	VALUES
    (
        @ent_Descripcion,
        @refg_Id,
        @ent_Fecha,
        @ent_UsuarioCrea,
        GETDATE()
    )
END
GO


# [Inventario].[PR_Inventario_Entradas_List]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Entradas_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Inventario].[PR_Inventario_Entradas_List]
AS
BEGIN
    SELECT 	entradas.ent_Id, 
			ent_Descripcion, 
			refugios.refg_Nombre, 
			ent_Fecha
    FROM 	[Inventario].[tbEntradas] AS entradas
	INNER JOIN [Refugio].[tbRefugios] AS refugios
	ON		entradas.refg_Id = refugios.refg_Id
    WHERE  	ent_EsEliminado != 1
END
GO


# [Inventario].[PR_Inventario_Entradas_Update]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Entradas_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Inventario].[PR_Inventario_Entradas_Update]
@ent_Id	int,
@ent_Descripcion	nvarchar(250),
@refg_Id	int,
@ent_Fecha	datetime,
@ent_UsuarioModifica	int
AS
BEGIN
    UPDATE [Inventario].[tbEntradas]
    SET ent_Descripcion = @ent_Descripcion,
        refg_Id = @refg_Id,
        ent_Fecha = @ent_Fecha,
        ent_UsuarioModifica = @ent_UsuarioModifica
    WHERE ent_Id = @ent_Id
END
GO


# [Inventario].[PR_Inventario_EntradasDetalles_Delete]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_EntradasDetalles_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
create PROCEDURE [Inventario].[PR_Inventario_EntradasDetalles_Delete]
@entdet_Id	int
AS
BEGIN
    UPDATE [Inventario].[tbEntradasDetalles]
    SET entdet_Cantidad = 1
END
GO


# [Inventario].[PR_Inventario_EntradasDetalles_Detail]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_EntradasDetalles_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


CREATE PROCEDURE [Inventario].[PR_Inventario_EntradasDetalles_Detail]
@entdet_Id INT
AS
BEGIN
    SELECT 	entdet_Id, 
			entradasDetalles.ent_Id,
			entradas.ent_Descripcion,
			entradasDetalles.itm_Id,
			items.itm_Descripcion,
			entdet_Cantidad, 
			entdet_EsEliminado, 
			usuarioCrea.usu_Nombre AS UsuarioCreacion, 
			entdet_FechaCrea, 
			usuarioModifica.usu_Nombre AS UsuarioModificacion, 
			entdet_FechaModifica
    FROM 	[Inventario].[tbEntradasDetalles] AS entradasDetalles
	INNER JOIN [Inventario].[tbEntradas] AS entradas
	ON		entradasDetalles.ent_Id = entradas.ent_Id
	INNER JOIN [Inventario].[tbItems] AS items
	ON		entradasDetalles.itm_Id = items.itm_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		entradasDetalles.entdet_UsuarioCrea = usuarioCrea.usu_Id 
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		entradasDetalles.entdet_UsuarioModifica = usuarioModifica.usu_Id
    WHERE  	entdet_EsEliminado != 1
END
GO


# [Inventario].[PR_Inventario_EntradasDetalles_Find]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_EntradasDetalles_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Inventario].[PR_Inventario_EntradasDetalles_Find]
@entdet_Id INT
AS
BEGIN
    SELECT 	entradas.ent_Id,
			entdet_Id,
			entradasDetalles.itm_Id,
			items.itm_Descripcion,
			entdet_Cantidad, 
			entdet_EsEliminado, 
			entdet_UsuarioCrea,
			usuarioCrea.Usu_Nombre AS usuarioCrea, 
			entdet_FechaCrea, 
			entdet_UsuarioModifica,
			usuarioModifica.Usu_Nombre AS usuarioModifica, 
			entdet_FechaModifica
    FROM 	[Inventario].[tbEntradasDetalles] AS entradasDetalles
	INNER JOIN [Inventario].[tbEntradas] AS entradas
	ON		entradasDetalles.ent_Id = entradas.ent_Id
	INNER JOIN [Inventario].[tbItems] AS items
	ON		entradasDetalles.itm_Id = items.itm_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		entradasDetalles.entdet_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		entradasDetalles.entdet_UsuarioModifica = usuarioModifica.usu_Id
    WHERE  	entdet_EsEliminado != 1
	AND		entdet_Id = 1
END
GO


# [Inventario].[PR_Inventario_EntradasDetalles_Insert]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_EntradasDetalles_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
create PROCEDURE [Inventario].[PR_Inventario_EntradasDetalles_Insert]
@ent_Id	int,
@itm_Id	int,
@entdet_Cantidad	int,
@entdet_UsuarioCrea	int
AS
BEGIN
    INSERT INTO [Inventario].[tbEntradasDetalles]
    (
        ent_Id, 
        itm_Id, 
        entdet_Cantidad, 
        entdet_UsuarioCrea, 
        entdet_FechaCrea
    )
	VALUES
    (
        @ent_Id, 
        @itm_Id, 
        @entdet_Cantidad, 
        @entdet_UsuarioCrea,
        GETDATE()
    )
END
GO


# [Inventario].[PR_Inventario_EntradasDetalles_List]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_EntradasDetalles_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

create PROCEDURE [Inventario].[PR_Inventario_EntradasDetalles_List]
AS
BEGIN
    SELECT 	entdet_Id, 
			entradas.ent_Descripcion,
			items.itm_Descripcion,
			entdet_Cantidad
    FROM 	[Inventario].[tbEntradasDetalles] AS entradasDetalles
	INNER JOIN [Inventario].[tbEntradas] AS entradas
	ON		entradasDetalles.ent_Id = entradas.ent_Id
	INNER JOIN [Inventario].[tbItems] AS items
	ON		entradasDetalles.itm_Id = items.itm_Id
    WHERE  	entdet_EsEliminado != 1
END
GO


# [Inventario].[PR_Inventario_EntradasDetalles_SelectByEntrada]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_EntradasDetalles_SelectByEntrada]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Inventario].[PR_Inventario_EntradasDetalles_SelectByEntrada]
@ent_Id INT
AS
BEGIN
    SELECT 	entdet_Id, 
			items.itm_Descripcion,
			entdet_Cantidad
    FROM 	[Inventario].[tbEntradasDetalles] AS entradasDetalles
	INNER JOIN [Inventario].[tbEntradas] AS entradas
	ON		entradasDetalles.ent_Id = entradas.ent_Id
	INNER JOIN [Inventario].[tbItems] AS items
	ON		entradasDetalles.itm_Id = items.itm_Id
    WHERE  	entdet_EsEliminado != 1
	AND		entradas.ent_Id = @ent_Id
END
GO


# [Inventario].[PR_Inventario_EntradasDetalles_Update]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_EntradasDetalles_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

create PROCEDURE [Inventario].[PR_Inventario_EntradasDetalles_Update]
@entdet_Id	int,
@ent_Id	int,
@itm_Id	int,
@entdet_Cantidad	int,
@entdet_EsEliminado	bit,
@entdet_UsuarioCrea	int,
@entdet_FechaCrea	datetime,
@entdet_UsuarioModifica	int,
@entdet_FechaModifica	datetime
AS
BEGIN
    UPDATE [Inventario].[tbEntradasDetalles]
    SET ent_Id = @ent_Id,
        itm_Id = @itm_Id,
        entdet_Cantidad = @entdet_Cantidad,
        entdet_EsEliminado = @entdet_EsEliminado,
        entdet_UsuarioCrea = @entdet_UsuarioCrea,
        entdet_FechaCrea = @entdet_FechaCrea,
        entdet_UsuarioModifica = @entdet_UsuarioModifica,
        entdet_FechaModifica = @entdet_FechaModifica
    WHERE entdet_Id = @entdet_Id
END
GO


# [Inventario].[PR_Inventario_Items_Delete]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Items_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


-----------------> DELETE

CREATE PROCEDURE [Inventario].[PR_Inventario_Items_Delete] 
@itm_Id INT
AS
  BEGIN
          UPDATE [Inventario].[tbItems]
          SET    itm_EsEliminado	= 1
          WHERE  itm_Id			= @itm_Id
  END
GO


# [Inventario].[PR_Inventario_Items_Detail]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Items_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Inventario].[PR_Inventario_Items_Detail]
@itm_Id INT
AS BEGIN
	SELECT	itm_Id,
			itm_Codigo,
			itm_Descripcion,
			categorias.cat_Descripcion
			itm_Precio,
			usuarioCrea.usu_Nombre AS UsuarioCreacion,
			itm_FechaCrea,
			usuarioModifica.usu_Nombre AS UsuarioModificacion,
			itm_FechaModifica
	FROM [Inventario].[tbItems] AS items
	INNER JOIN	[Inventario].[tbCategorias] AS categorias
	ON		 items.cat_Id = categorias.cat_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		items.itm_UsuarioCrea = usuarioCrea.usu_Id 
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		items.itm_UsuarioModifica = usuarioModifica.usu_Id
	WHERE itm_EsEliminado != 1 
	AND		itm_Id = @itm_Id
END
GO


# [Inventario].[PR_Inventario_Items_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Items_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Inventario].[PR_Inventario_Items_Dropdown]
AS BEGIN
	SELECT	itm_Id,
			itm_Codigo,
			itm_Descripcion
	FROM [Inventario].[tbItems] AS items
	WHERE itm_EsEliminado != 1 
END
GO


# [Inventario].[PR_Inventario_Items_Find]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Items_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Inventario].[PR_Inventario_Items_Find] 
@itm_Id INT
AS BEGIN
	SELECT	itm_Id,
			itm_Codigo,
			itm_Descripcion,
			items.cat_Id,
			categorias.cat_Descripcion,
			itm_Precio,
			itm_UsuarioCrea,
			usuarioCrea.Usu_Nombre AS usuarioCrea,
			itm_FechaCrea,
			itm_UsuarioModifica,
			usuarioModifica.Usu_Nombre AS usuarioModifica,
			itm_FechaModifica
	FROM [Inventario].[tbItems] AS items
	INNER JOIN	[Inventario].[tbCategorias] AS categorias
	ON		 items.cat_Id = categorias.cat_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		items.itm_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		items.itm_UsuarioModifica = usuarioModifica.usu_Id
	WHERE itm_EsEliminado != 1 
	AND		itm_Id = @itm_Id
END
GO


# [Inventario].[PR_Inventario_Items_Insert]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Items_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Inventario].[PR_Inventario_Items_Insert] 
@itm_Codigo	varchar(50),
@itm_Descripcion	varchar(50),
@cat_Id	int,
@itm_Precio	decimal(10, 2),
@itm_UsuarioCrea	int
	
AS BEGIN
INSERT INTO [Inventario].[tbItems] 
	(
        itm_Codigo,
		itm_Descripcion,
		cat_Id,
		itm_Precio,
		itm_UsuarioCrea,
		itm_FechaCrea
    )
VALUES
    (
        @itm_Codigo,
		@itm_Descripcion,
		@cat_Id,
		@itm_Precio,
		@itm_UsuarioCrea,
		GETDATE()
    )
END


GO


# [Inventario].[PR_Inventario_Items_List]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Items_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Inventario].[PR_Inventario_Items_List]
--@itm_Id INT
AS BEGIN
	SELECT	itm_Id,
			itm_Codigo,
			itm_Descripcion,
			categorias.cat_Descripcion,
			itm_Precio
	FROM [Inventario].[tbItems] AS items
	INNER JOIN	[Inventario].[tbCategorias] AS categorias
	ON		 items.cat_Id = categorias.cat_Id
	WHERE itm_EsEliminado != 1 
	--AND		itm_Id = @itm_Id
END
GO


# [Inventario].[PR_Inventario_Items_Update]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Items_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
-----------------> UPDATE
CREATE PROCEDURE [Inventario].[PR_Inventario_Items_Update] 
@itm_Id	int,
@itm_Codigo	varchar(50),
@itm_Descripcion	varchar(50),
@cat_Id	int,
@itm_Precio decimal(10, 2),
@itm_UsuarioModifica	int
AS BEGIN
UPDATE [Inventario].[tbItems]
SET itm_Codigo = @itm_Codigo,
	itm_Descripcion = @itm_Descripcion,
	cat_Id = @cat_Id,
	itm_Precio = @itm_Precio,
	itm_UsuarioModifica = @itm_UsuarioModifica,
	itm_FechaModifica = Getdate()
WHERE itm_Id = @itm_Id
END 
GO


# [Inventario].[PR_Inventario_ProcesarRecepcionInventario]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_ProcesarRecepcionInventario]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =====================================================
-- PASO 11: SP PARA PROCESAR RECEPCIÓN Y ACTUALIZAR EXISTENCIAS
-- =====================================================
CREATE PROCEDURE [Inventario].[PR_Inventario_ProcesarRecepcionInventario]
   @recep_Id INT
AS
BEGIN
   BEGIN TRANSACTION
   BEGIN TRY
       -- Actualizar existencias basado en los detalles de la recepción
       MERGE [Inventario].[tbExistencias] AS target
       USING (
           SELECT 
               d.itm_Id,
               r.refg_Id,
               SUM(d.recdet_Cantidad) as Cantidad
           FROM [Inventario].[tbRecepcionesDetalles] d
           INNER JOIN [Inventario].[tbRecepcionesMercancia] r ON d.recep_Id = r.recep_Id
           WHERE d.recep_Id = @recep_Id
               AND d.recdet_EsEliminado = 0
           GROUP BY d.itm_Id, r.refg_Id
       ) AS source ON (target.itm_Id = source.itm_Id AND target.refg_Id = source.refg_Id)
       WHEN MATCHED THEN
           UPDATE SET 
               exi_Cantidad = exi_Cantidad + source.Cantidad,
               exi_UltimaActualizacion = GETDATE()
       WHEN NOT MATCHED THEN
           INSERT (itm_Id, refg_Id, exi_Cantidad)
           VALUES (source.itm_Id, source.refg_Id, source.Cantidad);
       
       COMMIT TRANSACTION
   END TRY
   BEGIN CATCH
       ROLLBACK TRANSACTION
       THROW
   END CATCH
END
GO


# [Inventario].[PR_Inventario_ProcesarSalidaInventario]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_ProcesarSalidaInventario]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Procesar Salida y Actualizar Existencias
CREATE PROCEDURE [Inventario].[PR_Inventario_ProcesarSalidaInventario]
   @sal_Id INT
AS
BEGIN
   BEGIN TRANSACTION
   BEGIN TRY
       DECLARE @refg_Id INT
       SELECT @refg_Id = refg_Id FROM [Inventario].[tbSalidas] WHERE sal_Id = @sal_Id
       
       -- Actualizar existencias (restar cantidades)
       UPDATE e
       SET e.exi_Cantidad = e.exi_Cantidad - d.saldet_Cantidad,
           e.exi_UltimaActualizacion = GETDATE()
       FROM [Inventario].[tbExistencias] e
       INNER JOIN [Inventario].[tbSalidasDetalles] d ON e.itm_Id = d.itm_Id
       WHERE d.sal_Id = @sal_Id
           AND e.refg_Id = @refg_Id
           AND d.saldet_EsEliminado = 0
       
       -- Verificar que no quedaron existencias negativas
       IF EXISTS (SELECT 1 FROM [Inventario].[tbExistencias] WHERE refg_Id = @refg_Id AND exi_Cantidad < 0)
       BEGIN
           RAISERROR('Error: La salida generaría existencias negativas', 16, 1)
           ROLLBACK TRANSACTION
           RETURN
       END
       
       COMMIT TRANSACTION
   END TRY
   BEGIN CATCH
       ROLLBACK TRANSACTION
       THROW
   END CATCH
END
GO


# [Inventario].[PR_Inventario_RecepcionesDetalles_ByRecepcion]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_RecepcionesDetalles_ByRecepcion]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Ver Detalle de una Recepción
CREATE PROCEDURE [Inventario].[PR_Inventario_RecepcionesDetalles_ByRecepcion]
   @recep_Id INT
AS
BEGIN
   SELECT 
       rd.recdet_Id,
       rd.itm_Id,
       i.itm_Codigo,
       i.itm_Descripcion,
       c.cat_Descripcion,
       rd.recdet_Cantidad,
       rd.recdet_PrecioUnitario,
       rd.recdet_FechaVencimiento,
       rd.recdet_NumeroLote,
       i.itm_Precio,
       ISNULL(rd.recdet_PrecioUnitario, i.itm_Precio) * rd.recdet_Cantidad as ValorTotal
   FROM [Inventario].[tbRecepcionesDetalles] rd
   INNER JOIN [Inventario].[tbItems] i ON rd.itm_Id = i.itm_Id
   INNER JOIN [Inventario].[tbCategorias] c ON i.cat_Id = c.cat_Id
   WHERE rd.recep_Id = @recep_Id
       AND rd.recdet_EsEliminado = 0
END
GO


# [Inventario].[PR_Inventario_RecepcionesDetalles_Delete]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_RecepcionesDetalles_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: Eliminar Detalle de Recepción
-- =============================================
CREATE   PROCEDURE [Inventario].[PR_Inventario_RecepcionesDetalles_Delete]
    @recdet_Id INT
AS
BEGIN
    SET NOCOUNT ON;

    BEGIN TRY
        BEGIN TRANSACTION;

        UPDATE [Inventario].[tbRecepcionesDetalles]
        SET
            recdet_EsEliminado = 1,
            recdet_UsuarioModifica = 1, -- Cambiar según usuario en sesión
            recdet_FechaModifica = GETDATE()
        WHERE
            recdet_Id = @recdet_Id;

        COMMIT TRANSACTION;
        SELECT 1 AS Result; -- Éxito
    END TRY
    BEGIN CATCH
        IF @@TRANCOUNT > 0
            ROLLBACK TRANSACTION;

        SELECT 0 AS Result; -- Error
    END CATCH
END
GO


# [Inventario].[PR_Inventario_RecepcionesDetalles_Find]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_RecepcionesDetalles_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: Buscar Detalle de Recepción por ID
-- =============================================
CREATE   PROCEDURE [Inventario].[PR_Inventario_RecepcionesDetalles_Find]
    @recdet_Id INT
AS
BEGIN
    SET NOCOUNT ON;

    SELECT
        recdet_Id,
        recep_Id,
        itm_Id,
        recdet_Cantidad,
        recdet_PrecioUnitario,
        recdet_FechaVencimiento,
        recdet_NumeroLote
    FROM
        [Inventario].[tbRecepcionesDetalles]
    WHERE
        recdet_Id = @recdet_Id
        AND recdet_EsEliminado = 0;
END
GO


# [Inventario].[PR_Inventario_RecepcionesDetalles_Insert]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_RecepcionesDetalles_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
-- Modificar el SP de insertar detalle de recepción
CREATE PROCEDURE [Inventario].[PR_Inventario_RecepcionesDetalles_Insert]
   @recep_Id INT,
   @itm_Id INT,
   @recdet_Cantidad INT,
   @recdet_PrecioUnitario DECIMAL(10,2) = NULL,
   @recdet_FechaVencimiento DATE = NULL,
   @recdet_NumeroLote VARCHAR(50) = NULL,
   @recdet_UsuarioCrea INT
AS
BEGIN
   BEGIN TRANSACTION
   BEGIN TRY
       DECLARE @refg_Id INT
       SELECT @refg_Id = refg_Id FROM [Inventario].[tbRecepcionesMercancia] WHERE recep_Id = @recep_Id
       
       -- Insertar detalle
       INSERT INTO [Inventario].[tbRecepcionesDetalles]
       (recep_Id, itm_Id, recdet_Cantidad, recdet_PrecioUnitario, 
        recdet_FechaVencimiento, recdet_NumeroLote, recdet_EsEliminado, 
        recdet_UsuarioCrea, recdet_FechaCrea)
       VALUES
       (@recep_Id, @itm_Id, @recdet_Cantidad, @recdet_PrecioUnitario, 
        @recdet_FechaVencimiento, @recdet_NumeroLote, 0, 
        @recdet_UsuarioCrea, GETDATE())
       
       -- Actualizar existencias inmediatamente
       MERGE [Inventario].[tbExistencias] AS target
       USING (SELECT @itm_Id AS itm_Id, @refg_Id AS refg_Id) AS source
       ON target.itm_Id = source.itm_Id AND target.refg_Id = source.refg_Id
       WHEN MATCHED THEN
           UPDATE SET 
               exi_Cantidad = exi_Cantidad + @recdet_Cantidad,
               exi_UltimaActualizacion = GETDATE()
       WHEN NOT MATCHED THEN
           INSERT (itm_Id, refg_Id, exi_Cantidad)
           VALUES (@itm_Id, @refg_Id, @recdet_Cantidad);
       
       -- Registrar movimiento
       INSERT INTO [Inventario].[tbMovimientos]
       (mov_TipoMovimiento, mov_DocumentoId, itm_Id, refg_Id, 
        mov_Cantidad, mov_SaldoAnterior, mov_SaldoActual, mov_UsuarioCrea)
       SELECT 
           'E', @recep_Id, @itm_Id, @refg_Id, @recdet_Cantidad,
           ISNULL(exi_Cantidad - @recdet_Cantidad, 0), exi_Cantidad, @recdet_UsuarioCrea
       FROM [Inventario].[tbExistencias]
       WHERE itm_Id = @itm_Id AND refg_Id = @refg_Id
       
       COMMIT TRANSACTION
   END TRY
   BEGIN CATCH
       ROLLBACK TRANSACTION
       THROW
   END CATCH
END
GO


# [Inventario].[PR_Inventario_RecepcionesDetalles_List]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_RecepcionesDetalles_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: Listado de Detalles por Recepción
-- =============================================
CREATE   PROCEDURE [Inventario].[PR_Inventario_RecepcionesDetalles_List]
    @recep_Id INT
AS
BEGIN
    SET NOCOUNT ON;

    SELECT
        rd.recdet_Id,
        rd.recep_Id,
        rd.itm_Id,
        i.itm_Descripcion,
        rd.recdet_Cantidad,
        rd.recdet_PrecioUnitario,
        rd.recdet_FechaVencimiento,
        rd.recdet_NumeroLote
    FROM
        [Inventario].[tbRecepcionesDetalles] rd
        INNER JOIN [Inventario].[tbItems] i ON rd.itm_Id = i.itm_Id
    WHERE
        rd.recep_Id = @recep_Id
        AND rd.recdet_EsEliminado = 0
    ORDER BY
        rd.recdet_Id;
END
GO


# [Inventario].[PR_Inventario_RecepcionesMercancia_Delete]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_RecepcionesMercancia_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Eliminar (Lógico) Recepción
CREATE PROCEDURE [Inventario].[PR_Inventario_RecepcionesMercancia_Delete]
   @recep_Id INT,
   @recep_UsuarioModifica INT
AS
BEGIN
   BEGIN TRANSACTION
   BEGIN TRY
       -- Primero eliminamos los detalles
       UPDATE [Inventario].[tbRecepcionesDetalles]
       SET recdet_EsEliminado = 1,
           recdet_UsuarioModifica = @recep_UsuarioModifica,
           recdet_FechaModifica = GETDATE()
       WHERE recep_Id = @recep_Id
       
       -- Luego la recepción
       UPDATE [Inventario].[tbRecepcionesMercancia]
       SET recep_EsEliminado = 1,
           recep_UsuarioModifica = @recep_UsuarioModifica,
           recep_FechaModifica = GETDATE()
       WHERE recep_Id = @recep_Id
       
       COMMIT TRANSACTION
   END TRY
   BEGIN CATCH
       ROLLBACK TRANSACTION
       THROW
   END CATCH
END
GO


# [Inventario].[PR_Inventario_RecepcionesMercancia_Detail]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_RecepcionesMercancia_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: Detalle de Recepción de Mercancía
-- =============================================
CREATE   PROCEDURE [Inventario].[PR_Inventario_RecepcionesMercancia_Detail]
    @recep_Id INT
AS
BEGIN
    SET NOCOUNT ON;

    SELECT
        rm.recep_Id,
        rm.recep_Descripcion,
        rm.recep_Fecha,
        rm.refg_Id,
        r.refg_Nombre,
        rm.recep_TipoRecepcion,
        rm.recep_OrigenId,
        p.proc_Descripcion,
        rm.recep_NumeroDocumento
    FROM
        [Inventario].[tbRecepcionesMercancia] rm
        INNER JOIN [Refugio].[tbRefugios] r ON rm.refg_Id = r.refg_Id
        LEFT JOIN [Refugio].[tbProcedencias] p ON rm.recep_OrigenId = p.proc_Id
    WHERE
        rm.recep_Id = @recep_Id
        AND rm.recep_EsEliminado = 0;
END
GO


# [Inventario].[PR_Inventario_RecepcionesMercancia_Find]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_RecepcionesMercancia_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Inventario].[PR_Inventario_RecepcionesMercancia_Find]
    @recep_Id INT
AS
BEGIN
    SET NOCOUNT ON;

    SELECT 
        [recep_Id],
        [recep_Descripcion],
        [recep_Fecha],
        refugio.[refg_Id],
		refugio.refg_Nombre,
        [recep_EsEliminado],
        [recep_UsuarioCrea],
        [recep_FechaCrea],
        [recep_UsuarioModifica],
        [recep_FechaModifica],
        [recep_TipoRecepcion],
        [recep_OrigenId],
        [recep_NumeroDocumento]
    FROM [Inventario].[tbRecepcionesMercancia] rm
	INNER JOIN Refugio.tbRefugios refugio
	ON rm.refg_Id = refugio.refg_Id
    WHERE recep_Id = @recep_Id;
END
GO


# [Inventario].[PR_Inventario_RecepcionesMercancia_Insert]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_RecepcionesMercancia_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: Insertar Recepción de Mercancía
-- =============================================
CREATE   PROCEDURE [Inventario].[PR_Inventario_RecepcionesMercancia_Insert]
    @recep_Descripcion VARCHAR(500),
    @recep_Fecha DATETIME,
    @refg_Id INT,
    @recep_TipoRecepcion VARCHAR(50),
    @recep_OrigenId INT = NULL,
    @recep_NumeroDocumento VARCHAR(50) = NULL,
    @recep_UsuarioCrea INT
AS
BEGIN
    SET NOCOUNT ON;

    BEGIN TRY
        BEGIN TRANSACTION;

        INSERT INTO [Inventario].[tbRecepcionesMercancia]
        (
            recep_Descripcion,
            recep_Fecha,
            refg_Id,
            recep_EsEliminado,
            recep_UsuarioCrea,
            recep_FechaCrea,
            recep_TipoRecepcion,
            recep_OrigenId,
            recep_NumeroDocumento
        )
        VALUES
        (
            @recep_Descripcion,
            @recep_Fecha,
            @refg_Id,
            0, -- recep_EsEliminado
            @recep_UsuarioCrea,
            GETDATE(),
            @recep_TipoRecepcion,
            @recep_OrigenId,
            @recep_NumeroDocumento
        );

        COMMIT TRANSACTION;
        SELECT 1 AS Result; -- Éxito
    END TRY
    BEGIN CATCH
        IF @@TRANCOUNT > 0
            ROLLBACK TRANSACTION;

        SELECT 0 AS Result; -- Error
    END CATCH
END
GO


# [Inventario].[PR_Inventario_RecepcionesMercancia_List]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_RecepcionesMercancia_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: Listado de Recepciones de Mercancía
-- =============================================
CREATE   PROCEDURE [Inventario].[PR_Inventario_RecepcionesMercancia_List]
AS
BEGIN
    SET NOCOUNT ON;

    SELECT
        rm.recep_Id,
        rm.recep_Descripcion,
        rm.recep_Fecha,
        rm.refg_Id,
        r.refg_Nombre,
        rm.recep_TipoRecepcion,
        rm.recep_NumeroDocumento
    FROM
        [Inventario].[tbRecepcionesMercancia] rm
        INNER JOIN [Refugio].[tbRefugios] r ON rm.refg_Id = r.refg_Id
    WHERE
        rm.recep_EsEliminado = 0
    ORDER BY
        rm.recep_Fecha DESC, rm.recep_Id DESC;
END
GO


# [Inventario].[PR_Inventario_RecepcionesMercancia_Update]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_RecepcionesMercancia_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Actualizar Recepción
CREATE PROCEDURE [Inventario].[PR_Inventario_RecepcionesMercancia_Update]
   @recep_Id INT,
   @recep_Descripcion NVARCHAR(250),
   @recep_Fecha DATETIME,
   @recep_TipoRecepcion CHAR(1),
   @recep_OrigenId INT = NULL,
   @recep_NumeroDocumento VARCHAR(50) = NULL,
   @refg_Id INT,
   @recep_UsuarioModifica INT
AS
BEGIN
   UPDATE [Inventario].[tbRecepcionesMercancia]
   SET recep_Descripcion = @recep_Descripcion,
       recep_Fecha = @recep_Fecha,
       recep_TipoRecepcion = @recep_TipoRecepcion,
       recep_OrigenId = @recep_OrigenId,
       recep_NumeroDocumento = @recep_NumeroDocumento,
	   refg_Id = @refg_Id,
       recep_UsuarioModifica = @recep_UsuarioModifica,
       recep_FechaModifica = GETDATE()
   WHERE recep_Id = @recep_Id
END
GO


# [Inventario].[PR_Inventario_Salidas_Detail]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Salidas_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Inventario].[PR_Inventario_Salidas_Detail]
    @sal_Id INT
AS
BEGIN
    SELECT 
        s.[sal_Id],
        s.[sal_Descripcion],
        s.[sal_TipoSalida],
        r.[refg_Id],
        r.[refg_Nombre],
        s.[sal_Fecha],
        s.[sal_EsEliminado],
        s.[sal_UsuarioCrea],
        usuarioCrea.usu_Nombre AS UsuarioCreacion,
        s.[sal_FechaCrea],
        s.[sal_UsuarioModifica],
        usuarioModifica.usu_Nombre AS UsuarioModificacion,
        s.[sal_FechaModifica]
    FROM [Inventario].[tbSalidas] s
    INNER JOIN [Refugio].[tbRefugios] r ON s.refg_Id = r.refg_Id
    LEFT JOIN [Seguridad].[tbUsuarios] usuarioCrea ON s.sal_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] usuarioModifica ON s.sal_UsuarioModifica = usuarioModifica.usu_Id
    WHERE s.sal_Id = @sal_Id;
END
GO


# [Inventario].[PR_Inventario_Salidas_Find]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Salidas_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Inventario].[PR_Inventario_Salidas_Find]
    @sal_Id INT
AS
BEGIN
    SET NOCOUNT ON;

    SELECT 
        s.[sal_Id],
        s.[sal_Descripcion],
        s.[sal_TipoSalida],
        r.[refg_Id],
        r.[refg_Nombre],
        s.[sal_Fecha],
        s.[sal_EsEliminado],
        s.[sal_UsuarioCrea],
        s.[sal_FechaCrea],
        s.[sal_UsuarioModifica],
        s.[sal_FechaModifica]
    FROM [Inventario].[tbSalidas] s
    INNER JOIN [Refugio].[tbRefugios] r ON s.refg_Id = r.refg_Id
    WHERE s.sal_Id = @sal_Id;
END
GO


# [Inventario].[PR_Inventario_Salidas_Insert]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Salidas_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =====================================================
-- PASO 15: PROCEDIMIENTOS PARA SALIDAS
-- =====================================================

-- SP: Insertar Salida
CREATE PROCEDURE [Inventario].[PR_Inventario_Salidas_Insert]
   @sal_Descripcion NVARCHAR(250),
   @sal_TipoSalida CHAR(1),
   @refg_Id INT,
   @sal_Fecha DATETIME,
   @sal_UsuarioCrea INT
AS
BEGIN
   INSERT INTO [Inventario].[tbSalidas]
   (
       sal_Descripcion,
       sal_TipoSalida,
       refg_Id,
       sal_Fecha,
       sal_EsEliminado,
       sal_UsuarioCrea,
       sal_FechaCrea
   )
   VALUES
   (
       @sal_Descripcion,
       @sal_TipoSalida,
       @refg_Id,
       @sal_Fecha,
       0,
       @sal_UsuarioCrea,
       GETDATE()
   )
   
   SELECT SCOPE_IDENTITY() AS sal_Id
END
GO


# [Inventario].[PR_Inventario_Salidas_List]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_Salidas_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Inventario].[PR_Inventario_Salidas_List]
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        s.[sal_Id],
        s.[sal_Descripcion],
        s.[sal_TipoSalida],
        s.[refg_Id],
        r.[refg_Nombre],
        s.[sal_Fecha]
    FROM [Inventario].[tbSalidas] s
    INNER JOIN [Refugio].[tbRefugios] r ON s.refg_Id = r.refg_Id
    WHERE s.sal_EsEliminado = 0
    ORDER BY s.sal_Fecha DESC;
END
GO


# [Inventario].[PR_Inventario_SalidasDetalles_BySalida]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_SalidasDetalles_BySalida]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Listar Detalles por Salida
CREATE   PROCEDURE [Inventario].[PR_Inventario_SalidasDetalles_BySalida]
   @sal_Id INT
AS
BEGIN
   SET NOCOUNT ON;
   
   SELECT 
       sd.saldet_Id,
       sd.sal_Id,
       s.sal_Descripcion,
       sd.itm_Id,
       i.itm_Codigo,
       i.itm_Descripcion,
       c.cat_Descripcion,
       sd.saldet_Cantidad,
       sd.saldet_Observaciones,
       i.itm_Precio,
       i.itm_Precio * sd.saldet_Cantidad AS ValorTotal
   FROM [Inventario].[tbSalidasDetalles] sd
   INNER JOIN [Inventario].[tbSalidas] s ON sd.sal_Id = s.sal_Id
   INNER JOIN [Inventario].[tbItems] i ON sd.itm_Id = i.itm_Id
   INNER JOIN [Inventario].[tbCategorias] c ON i.cat_Id = c.cat_Id
   WHERE sd.sal_Id = @sal_Id
       AND sd.saldet_EsEliminado = 0
   ORDER BY sd.saldet_Id;
END
GO


# [Inventario].[PR_Inventario_SalidasDetalles_Delete]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_SalidasDetalles_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Eliminar (Lógico) Detalle de Salida
CREATE   PROCEDURE [Inventario].[PR_Inventario_SalidasDetalles_Delete]
   @saldet_Id INT,
   @saldet_UsuarioModifica INT
AS
BEGIN
   SET NOCOUNT ON;
   
   BEGIN TRANSACTION
   BEGIN TRY
       DECLARE @sal_Id INT, @itm_Id INT, @cantidad INT, @refg_Id INT;
       
       -- Obtener datos del detalle a eliminar
       SELECT @sal_Id = sal_Id, @itm_Id = itm_Id, @cantidad = saldet_Cantidad
       FROM [Inventario].[tbSalidasDetalles]
       WHERE saldet_Id = @saldet_Id AND saldet_EsEliminado = 0;
       
       SELECT @refg_Id = refg_Id FROM [Inventario].[tbSalidas] WHERE sal_Id = @sal_Id;
       
       -- Marcar como eliminado
       UPDATE [Inventario].[tbSalidasDetalles]
       SET saldet_EsEliminado = 1,
           saldet_UsuarioModifica = @saldet_UsuarioModifica,
           saldet_FechaModifica = GETDATE()
       WHERE saldet_Id = @saldet_Id;
       
       -- Restaurar existencias
       UPDATE [Inventario].[tbExistencias]
       SET exi_Cantidad = exi_Cantidad + @cantidad,
           exi_UltimaActualizacion = GETDATE()
       WHERE itm_Id = @itm_Id AND refg_Id = @refg_Id;
       
       -- Registrar movimiento de restauración
       INSERT INTO [Inventario].[tbMovimientos]
       (mov_TipoMovimiento, mov_DocumentoId, itm_Id, refg_Id, 
        mov_Cantidad, mov_SaldoAnterior, mov_SaldoActual, mov_UsuarioCrea)
       SELECT 
           'E', @sal_Id, @itm_Id, @refg_Id, @cantidad,
           exi_Cantidad - @cantidad, exi_Cantidad, @saldet_UsuarioModifica
       FROM [Inventario].[tbExistencias]
       WHERE itm_Id = @itm_Id AND refg_Id = @refg_Id;
       
       COMMIT TRANSACTION;
       
   END TRY
   BEGIN CATCH
       ROLLBACK TRANSACTION;
       THROW;
   END CATCH
END
GO


# [Inventario].[PR_Inventario_SalidasDetalles_Find]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_SalidasDetalles_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Buscar Detalle Específico
CREATE   PROCEDURE [Inventario].[PR_Inventario_SalidasDetalles_Find]
   @saldet_Id INT
AS
BEGIN
   SET NOCOUNT ON;
   
   SELECT 
       sd.saldet_Id,
       sd.sal_Id,
       s.sal_Descripcion,
       sd.itm_Id,
       i.itm_Descripcion AS itm_Descripcion,
       sd.saldet_Cantidad,
       sd.saldet_Observaciones,
       sd.saldet_EsEliminado,
       sd.saldet_UsuarioCrea,
       usuarioCrea.usu_Nombre AS usuarioCrea,
       sd.saldet_FechaCrea,
       sd.saldet_UsuarioModifica,
       usuarioModifica.usu_Nombre AS usuarioModifica,
       sd.saldet_FechaModifica
   FROM [Inventario].[tbSalidasDetalles] sd
   INNER JOIN [Inventario].[tbSalidas] s ON sd.sal_Id = s.sal_Id
   INNER JOIN [Inventario].[tbItems] i ON sd.itm_Id = i.itm_Id
   LEFT JOIN [Seguridad].[tbUsuarios] usuarioCrea ON sd.saldet_UsuarioCrea = usuarioCrea.usu_Id
   LEFT JOIN [Seguridad].[tbUsuarios] usuarioModifica ON sd.saldet_UsuarioModifica = usuarioModifica.usu_Id
   WHERE sd.saldet_Id = @saldet_Id;
END
GO


# [Inventario].[PR_Inventario_SalidasDetalles_Insert]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_SalidasDetalles_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Insertar Detalle de Salida
CREATE   PROCEDURE [Inventario].[PR_Inventario_SalidasDetalles_Insert]
   @sal_Id INT,
   @itm_Id INT,
   @saldet_Cantidad INT,
   @saldet_Observaciones NVARCHAR(500) = NULL,
   @saldet_UsuarioCrea INT
AS
BEGIN
   SET NOCOUNT ON;
   
   BEGIN TRANSACTION
   BEGIN TRY
       -- Verificar existencia disponible
       DECLARE @existenciaActual INT, @refg_Id INT;
       
       SELECT @refg_Id = refg_Id FROM [Inventario].[tbSalidas] WHERE sal_Id = @sal_Id;
       
       SELECT @existenciaActual = ISNULL(exi_Cantidad, 0)
       FROM [Inventario].[tbExistencias]
       WHERE itm_Id = @itm_Id AND refg_Id = @refg_Id;
       
       IF @existenciaActual < @saldet_Cantidad
       BEGIN
           RAISERROR('No hay suficiente existencia. Disponible: %d, Solicitado: %d', 16, 1, @existenciaActual, @saldet_Cantidad);
           RETURN;
       END
       
       -- Insertar detalle
       INSERT INTO [Inventario].[tbSalidasDetalles]
       (
           sal_Id,
           itm_Id,
           saldet_Cantidad,
           saldet_Observaciones,
           saldet_EsEliminado,
           saldet_UsuarioCrea,
           saldet_FechaCrea
       )
       VALUES
       (
           @sal_Id,
           @itm_Id,
           @saldet_Cantidad,
           @saldet_Observaciones,
           0,
           @saldet_UsuarioCrea,
           GETDATE()
       );
       
       -- Actualizar existencias
       UPDATE [Inventario].[tbExistencias]
       SET exi_Cantidad = exi_Cantidad - @saldet_Cantidad,
           exi_UltimaActualizacion = GETDATE()
       WHERE itm_Id = @itm_Id AND refg_Id = @refg_Id;
       
       -- Registrar movimiento
       INSERT INTO [Inventario].[tbMovimientos]
       (mov_TipoMovimiento, mov_DocumentoId, itm_Id, refg_Id, 
        mov_Cantidad, mov_SaldoAnterior, mov_SaldoActual, mov_UsuarioCrea)
       SELECT 
           'S', @sal_Id, @itm_Id, @refg_Id, @saldet_Cantidad,
           exi_Cantidad + @saldet_Cantidad, exi_Cantidad, @saldet_UsuarioCrea
       FROM [Inventario].[tbExistencias]
       WHERE itm_Id = @itm_Id AND refg_Id = @refg_Id;
       
       COMMIT TRANSACTION;
       SELECT SCOPE_IDENTITY() AS saldet_Id;
       
   END TRY
   BEGIN CATCH
       ROLLBACK TRANSACTION;
       THROW;
   END CATCH
END
GO


# [Inventario].[PR_Inventario_SalidasDetalles_Update]

In [0]:
/****** Object:  StoredProcedure [Inventario].[PR_Inventario_SalidasDetalles_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Actualizar Detalle de Salida
CREATE   PROCEDURE [Inventario].[PR_Inventario_SalidasDetalles_Update]
   @saldet_Id INT,
   @sal_Id INT,
   @itm_Id INT,
   @saldet_Cantidad INT,
   @saldet_Observaciones NVARCHAR(500) = NULL,
   @saldet_UsuarioModifica INT
AS
BEGIN
   SET NOCOUNT ON;
   
   BEGIN TRANSACTION
   BEGIN TRY
       DECLARE @cantidadAnterior INT, @refg_Id INT, @existenciaActual INT;
       
       -- Obtener cantidad anterior y refugio
       SELECT @cantidadAnterior = saldet_Cantidad 
       FROM [Inventario].[tbSalidasDetalles] 
       WHERE saldet_Id = @saldet_Id;
       
       SELECT @refg_Id = refg_Id FROM [Inventario].[tbSalidas] WHERE sal_Id = @sal_Id;
       
       -- Calcular diferencia y verificar existencia
       DECLARE @diferencia INT = @saldet_Cantidad - @cantidadAnterior;
       
       SELECT @existenciaActual = ISNULL(exi_Cantidad, 0)
       FROM [Inventario].[tbExistencias]
       WHERE itm_Id = @itm_Id AND refg_Id = @refg_Id;
       
       IF (@existenciaActual - @diferencia) < 0
       BEGIN
           RAISERROR('No hay suficiente existencia para el ajuste. Disponible: %d, Se necesita: %d adicionales', 
                    16, 1, @existenciaActual, @diferencia);
           RETURN;
       END
       
       -- Actualizar detalle
       UPDATE [Inventario].[tbSalidasDetalles]
       SET sal_Id = @sal_Id,
           itm_Id = @itm_Id,
           saldet_Cantidad = @saldet_Cantidad,
           saldet_Observaciones = @saldet_Observaciones,
           saldet_UsuarioModifica = @saldet_UsuarioModifica,
           saldet_FechaModifica = GETDATE()
       WHERE saldet_Id = @saldet_Id;
       
       -- Ajustar existencias si hay diferencia
       IF @diferencia != 0
       BEGIN
           UPDATE [Inventario].[tbExistencias]
           SET exi_Cantidad = exi_Cantidad - @diferencia,
               exi_UltimaActualizacion = GETDATE()
           WHERE itm_Id = @itm_Id AND refg_Id = @refg_Id;
           
           -- Registrar movimiento de ajuste
           INSERT INTO [Inventario].[tbMovimientos]
           (mov_TipoMovimiento, mov_DocumentoId, itm_Id, refg_Id, 
            mov_Cantidad, mov_SaldoAnterior, mov_SaldoActual, mov_UsuarioCrea)
           SELECT 
               'A', @sal_Id, @itm_Id, @refg_Id, ABS(@diferencia),
               exi_Cantidad + @diferencia, exi_Cantidad, @saldet_UsuarioModifica
           FROM [Inventario].[tbExistencias]
           WHERE itm_Id = @itm_Id AND refg_Id = @refg_Id;
       END
       
       COMMIT TRANSACTION;
       
   END TRY
   BEGIN CATCH
       ROLLBACK TRANSACTION;
       THROW;
   END CATCH
END
GO


# [Medico].[PR_Medico_CitaMedica_Delete]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_CitaMedica_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_CitaMedica_Delete]
    @cita_Id INT,
    @cita_UsuarioModifica INT
AS
BEGIN
    SET NOCOUNT ON

    UPDATE [Medico].[tbCitaMedica]
    SET
        cita_EsEliminado = 1,
        cita_UsuarioModifica = @cita_UsuarioModifica,
        cita_FechaModifica = GETDATE()
    WHERE cita_Id = @cita_Id
END
GO


# [Medico].[PR_Medico_CitaMedica_Detail]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_CitaMedica_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_CitaMedica_Detail]
    @cita_Id INT
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        cita.cita_Id,
        cita.masc_Id,
        masc.masc_Nombre,
        cita.cita_FechaConsulta,
        cita.tipoCon_Id,
        tipoCon.tipoCon_Descripcion AS TipoConsulta,
        cita.grav_Id,
        grav.grav_Descripcion AS Gravedad,
        cita.cita_MotivoConsulta,
        cita.cita_Diagnostico,
        cita.cita_Peso,
        cita.cita_Temperatura,
        cita.cita_FrecuenciaCardiaca,
        cita.cita_FrecuenciaRespiratoria,
        cita.com_Id,
        com.com_Descripcion AS Comportamiento,
        cita.cita_ProcedimientosRealizados,
        cita.cita_ResultadosExamenes,
        cita.cita_ProximaCita,
        cita.cita_MotivoProximaCita,
        usuarioCrea.usu_Nombre AS UsuarioCreacion,
        cita.cita_FechaCrea,
        usuarioModifica.usu_Nombre AS UsuarioModificacion,
        cita.cita_FechaModifica
    FROM [Medico].[tbCitaMedica] AS cita
    INNER JOIN [Refugio].[tbMascotas] AS masc
        ON cita.masc_Id = masc.masc_Id
    LEFT JOIN [Medico].[tbTiposConsulta] AS tipoCon
        ON cita.tipoCon_Id = tipoCon.tipoCon_Id
    LEFT JOIN [Medico].[tbGravedades] AS grav
        ON cita.grav_Id = grav.grav_Id
    LEFT JOIN [Refugio].[tbComportamientos] AS com
        ON cita.com_Id = com.com_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON cita.cita_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON cita.cita_UsuarioModifica = usuarioModifica.usu_Id
    WHERE cita.cita_Id = @cita_Id
    AND cita.cita_EsEliminado = 0
END
GO


# [Medico].[PR_Medico_CitaMedica_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_CitaMedica_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_CitaMedica_Dropdown]
    @masc_Id INT = NULL
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        cita.cita_Id,
        CONCAT('#', cita.cita_Id, ' - ', FORMAT(cita.cita_FechaConsulta, 'dd/MM/yyyy'),
               ' - ', ISNULL(tipoCon.tipoCon_Descripcion, 'Sin tipo')) AS Descripcion
    FROM [Medico].[tbCitaMedica] AS cita
    LEFT JOIN [Medico].[tbTiposConsulta] AS tipoCon
        ON cita.tipoCon_Id = tipoCon.tipoCon_Id
    WHERE cita.cita_EsEliminado = 0
    AND (@masc_Id IS NULL OR cita.masc_Id = @masc_Id)
    ORDER BY cita.cita_FechaConsulta DESC
END
GO


# [Medico].[PR_Medico_CitaMedica_Find]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_CitaMedica_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_CitaMedica_Find]
    @cita_Id INT
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        cita.cita_Id,
        cita.masc_Id,
        cita.cita_FechaConsulta,
        cita.tipoCon_Id,
        cita.grav_Id,
        cita.cita_MotivoConsulta,
        cita.cita_Diagnostico,
        cita.cita_Peso,
        cita.cita_Temperatura,
        cita.cita_FrecuenciaCardiaca,
        cita.cita_FrecuenciaRespiratoria,
        cita.com_Id,
        cita.vac_Id,
        cita.cita_ProcedimientosRealizados,
        cita.cita_ResultadosExamenes,
        cita.cita_ProximaCita,
        cita.cita_MotivoProximaCita,
        cita.cita_UsuarioCrea,
        usuarioCrea.usu_Nombre AS usuarioCrea,
        cita.cita_FechaCrea,
        cita.cita_UsuarioModifica,
        usuarioModifica.usu_Nombre AS usuarioModifica,
        cita.cita_FechaModifica
    FROM [Medico].[tbCitaMedica] AS cita
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON cita.cita_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON cita.cita_UsuarioModifica = usuarioModifica.usu_Id
    WHERE cita.cita_Id = @cita_Id
    AND cita.cita_EsEliminado = 0
END
GO


# [Medico].[PR_Medico_CitaMedica_Insert]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_CitaMedica_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_CitaMedica_Insert]
    @masc_Id INT,
    @cita_FechaConsulta DATETIME,
    @tipoCon_Id INT = NULL,
    @grav_Id INT = NULL,
    @cita_MotivoConsulta NVARCHAR(500) = NULL,
    @cita_Diagnostico NVARCHAR(500) = NULL,
    @cita_Peso DECIMAL(5,2) = NULL,
    @cita_Temperatura DECIMAL(4,2) = NULL,
    @cita_FrecuenciaCardiaca INT = NULL,
    @cita_FrecuenciaRespiratoria INT = NULL,
    @com_Id INT = NULL,
    @vac_Id INT = NULL,
    @cita_ProcedimientosRealizados NVARCHAR(500) = NULL,
    @cita_ResultadosExamenes NVARCHAR(500) = NULL,
    @cita_ProximaCita DATETIME = NULL,
    @cita_MotivoProximaCita NVARCHAR(200) = NULL,
    @cita_UsuarioCrea INT
AS
BEGIN
    SET NOCOUNT ON

    INSERT INTO [Medico].[tbCitaMedica]
    (
        masc_Id,
        cita_FechaConsulta,
        tipoCon_Id,
        grav_Id,
        cita_MotivoConsulta,
        cita_Diagnostico,
        cita_Peso,
        cita_Temperatura,
        cita_FrecuenciaCardiaca,
        cita_FrecuenciaRespiratoria,
        com_Id,
        vac_Id,
        cita_ProcedimientosRealizados,
        cita_ResultadosExamenes,
        cita_ProximaCita,
        cita_MotivoProximaCita,
        cita_UsuarioCrea,
        cita_FechaCrea
    )
    VALUES
    (
        @masc_Id,
        @cita_FechaConsulta,
        @tipoCon_Id,
        @grav_Id,
        @cita_MotivoConsulta,
        @cita_Diagnostico,
        @cita_Peso,
        @cita_Temperatura,
        @cita_FrecuenciaCardiaca,
        @cita_FrecuenciaRespiratoria,
        @com_Id,
        @vac_Id,
        @cita_ProcedimientosRealizados,
        @cita_ResultadosExamenes,
        @cita_ProximaCita,
        @cita_MotivoProximaCita,
        @cita_UsuarioCrea,
        GETDATE()
    )

    SELECT SCOPE_IDENTITY() AS cita_Id
END
GO


# [Medico].[PR_Medico_CitaMedica_List]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_CitaMedica_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_CitaMedica_List]
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        ROW_NUMBER() OVER(ORDER BY cita.cita_FechaConsulta DESC) AS Fila,
        cita.cita_Id,
        cita.masc_Id,
        masc.masc_Nombre,
        cita.cita_FechaConsulta,
        cita.tipoCon_Id,
        tipoCon.tipoCon_Descripcion AS TipoConsulta,
        cita.grav_Id,
        grav.grav_Descripcion AS Gravedad,
        cita.cita_MotivoConsulta,
        cita.cita_Diagnostico,
        cita.cita_Peso,
        cita.cita_Temperatura,
        cita.cita_FrecuenciaCardiaca,
        cita.cita_FrecuenciaRespiratoria,
        cita.cita_ProximaCita
    FROM [Medico].[tbCitaMedica] AS cita
    INNER JOIN [Refugio].[tbMascotas] AS masc
        ON cita.masc_Id = masc.masc_Id
    LEFT JOIN [Medico].[tbTiposConsulta] AS tipoCon
        ON cita.tipoCon_Id = tipoCon.tipoCon_Id
    LEFT JOIN [Medico].[tbGravedades] AS grav
        ON cita.grav_Id = grav.grav_Id
    WHERE cita.cita_EsEliminado = 0
    ORDER BY cita.cita_FechaConsulta DESC
END
GO


# [Medico].[PR_Medico_CitaMedica_Update]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_CitaMedica_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_CitaMedica_Update]
    @cita_Id INT,
    @masc_Id INT,
    @cita_FechaConsulta DATETIME,
    @tipoCon_Id INT = NULL,
    @grav_Id INT = NULL,
    @cita_MotivoConsulta NVARCHAR(500) = NULL,
    @cita_Diagnostico NVARCHAR(500) = NULL,
    @cita_Peso DECIMAL(5,2) = NULL,
    @cita_Temperatura DECIMAL(4,2) = NULL,
    @cita_FrecuenciaCardiaca INT = NULL,
    @cita_FrecuenciaRespiratoria INT = NULL,
    @com_Id INT = NULL,
    @vac_Id INT = NULL,
    @cita_ProcedimientosRealizados NVARCHAR(500) = NULL,
    @cita_ResultadosExamenes NVARCHAR(500) = NULL,
    @cita_ProximaCita DATETIME = NULL,
    @cita_MotivoProximaCita NVARCHAR(200) = NULL,
    @cita_UsuarioModifica INT
AS
BEGIN
    SET NOCOUNT ON

    UPDATE [Medico].[tbCitaMedica]
    SET
        masc_Id = @masc_Id,
        cita_FechaConsulta = @cita_FechaConsulta,
        tipoCon_Id = @tipoCon_Id,
        grav_Id = @grav_Id,
        cita_MotivoConsulta = @cita_MotivoConsulta,
        cita_Diagnostico = @cita_Diagnostico,
        cita_Peso = @cita_Peso,
        cita_Temperatura = @cita_Temperatura,
        cita_FrecuenciaCardiaca = @cita_FrecuenciaCardiaca,
        cita_FrecuenciaRespiratoria = @cita_FrecuenciaRespiratoria,
        com_Id = @com_Id,
        vac_Id = @vac_Id,
        cita_ProcedimientosRealizados = @cita_ProcedimientosRealizados,
        cita_ResultadosExamenes = @cita_ResultadosExamenes,
        cita_ProximaCita = @cita_ProximaCita,
        cita_MotivoProximaCita = @cita_MotivoProximaCita,
        cita_UsuarioModifica = @cita_UsuarioModifica,
        cita_FechaModifica = GETDATE()
    WHERE cita_Id = @cita_Id
END
GO


# [Medico].[PR_Medico_Gravedades_Delete]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Gravedades_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Gravedades_Delete]
    @grav_Id INT
AS
BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        UPDATE [Medico].[tbGravedades]
        SET grav_EsEliminado = 1
        WHERE grav_Id = @grav_Id
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_Gravedades_Detail]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Gravedades_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Gravedades_Detail]
    @grav_Id INT
AS
BEGIN
    SET NOCOUNT ON
    SELECT  g.grav_Id, g.grav_Descripcion,
            usuarioCrea.usu_Nombre AS UsuarioCreacion,
            g.grav_FechaCrea,
            usuarioModifica.usu_Nombre AS UsuarioModificacion,
            g.grav_FechaModifica
    FROM [Medico].[tbGravedades] AS g
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON g.grav_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON g.grav_UsuarioModifica = usuarioModifica.usu_Id
    WHERE g.grav_EsEliminado = 0 AND g.grav_Id = @grav_Id
END
GO


# [Medico].[PR_Medico_Gravedades_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Gravedades_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Gravedades_Dropdown]
AS
BEGIN
    SET NOCOUNT ON
    SELECT grav_Id, grav_Descripcion
    FROM [Medico].[tbGravedades]
    WHERE grav_EsEliminado = 0
    ORDER BY grav_Id ASC
END
GO


# [Medico].[PR_Medico_Gravedades_Find]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Gravedades_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_Gravedades_Find]
    @grav_Id INT
AS
BEGIN
    SET NOCOUNT ON
    SELECT g.grav_Id, g.grav_Descripcion, g.grav_EsActivo,
           g.grav_UsuarioCrea,
           usuarioCrea.usu_Nombre AS usuarioCrea,
           g.grav_FechaCrea,
           g.grav_UsuarioModifica,
           usuarioModifica.usu_Nombre AS usuarioModifica,
           g.grav_FechaModifica
    FROM [Medico].[tbGravedades] AS g
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON g.grav_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON g.grav_UsuarioModifica = usuarioModifica.usu_Id
    WHERE g.grav_EsEliminado = 0 AND g.grav_Id = @grav_Id
END
GO


# [Medico].[PR_Medico_Gravedades_Insert]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Gravedades_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_Gravedades_Insert]
    @grav_Descripcion NVARCHAR(50),
    @grav_EsActivo BIT = 1,
    @grav_UsuarioCrea INT
AS
BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        INSERT INTO [Medico].[tbGravedades] (grav_Descripcion, grav_EsActivo, grav_UsuarioCrea, grav_FechaCrea)
        VALUES (@grav_Descripcion, @grav_EsActivo, @grav_UsuarioCrea, GETDATE())
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_Gravedades_List]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Gravedades_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO
CREATE PROCEDURE [Medico].[PR_Medico_Gravedades_List]
AS
BEGIN
    SET NOCOUNT ON
    SELECT grav_Id, grav_Descripcion,
           CASE WHEN grav_EsActivo = 1 THEN 'Activo' ELSE 'Inactivo' END AS grav_EsActivo
    FROM [Medico].[tbGravedades]
    WHERE grav_EsEliminado != 1
END

GO


# [Medico].[PR_Medico_Gravedades_Update]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Gravedades_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_Gravedades_Update]
    @grav_Id INT,
    @grav_Descripcion NVARCHAR(50),
    @grav_EsActivo BIT = 1,
    @grav_UsuarioModifica INT
AS
BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        UPDATE [Medico].[tbGravedades]
        SET grav_Descripcion = @grav_Descripcion,
            grav_EsActivo = @grav_EsActivo,
            grav_UsuarioModifica = @grav_UsuarioModifica,
            grav_FechaModifica = GETDATE()
        WHERE grav_Id = @grav_Id
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_Recetas_Delete]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Recetas_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Recetas_Delete]
    @receta_Id INT,
    @receta_UsuarioModifica INT
AS
BEGIN
    SET NOCOUNT ON

    UPDATE [Medico].[tbRecetas]
    SET
        receta_EsEliminado = 1,
        receta_UsuarioModifica = @receta_UsuarioModifica,
        receta_FechaModifica = GETDATE()
    WHERE receta_Id = @receta_Id
END
GO


# [Medico].[PR_Medico_Recetas_Detail]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Recetas_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Recetas_Detail]
    @receta_Id INT
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        rec.receta_Id,
        rec.cita_Id,
        rec.masc_Id,
        masc.masc_Nombre AS Mascota,
        rec.receta_Medicamento,
        rec.tipoMed_Id,
        tipoMed.tipoMed_Descripcion AS TipoMedicamento,
        rec.viaAdmin_Id,
        viaAdmin.viaAdmin_Descripcion AS ViaAdministracion,
        rec.receta_Dosis,
        rec.receta_Frecuencia,
        rec.receta_Duracion,
        rec.receta_Instrucciones,
        rec.receta_FechaInicio,
        rec.receta_FechaFin,
        rec.receta_Estado,
        usuarioCrea.usu_Nombre AS UsuarioCreacion,
        rec.receta_FechaCrea,
        usuarioModifica.usu_Nombre AS UsuarioModificacion,
        rec.receta_FechaModifica
    FROM [Medico].[tbRecetas] AS rec
    INNER JOIN [Refugio].[tbMascotas] AS masc
        ON rec.masc_Id = masc.masc_Id
    LEFT JOIN [Medico].[tbTiposMedicamento] AS tipoMed
        ON rec.tipoMed_Id = tipoMed.tipoMed_Id
    LEFT JOIN [Medico].[tbViasAdministracion] AS viaAdmin
        ON rec.viaAdmin_Id = viaAdmin.viaAdmin_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON rec.receta_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON rec.receta_UsuarioModifica = usuarioModifica.usu_Id
    WHERE rec.receta_Id = @receta_Id
    AND rec.receta_EsEliminado = 0
END
GO


# [Medico].[PR_Medico_Recetas_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Recetas_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Recetas_Dropdown]
    @masc_Id INT = NULL
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        receta_Id,
        CONCAT(receta_Medicamento, ' - ', receta_Estado) AS Descripcion
    FROM [Medico].[tbRecetas]
    WHERE receta_EsEliminado = 0
    AND (@masc_Id IS NULL OR masc_Id = @masc_Id)
    ORDER BY receta_FechaInicio DESC
END
GO


# [Medico].[PR_Medico_Recetas_Find]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Recetas_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Recetas_Find]
    @receta_Id INT
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        rec.receta_Id,
        rec.cita_Id,
        rec.masc_Id,
        rec.receta_Medicamento,
        rec.tipoMed_Id,
        rec.viaAdmin_Id,
        rec.receta_Dosis,
        rec.receta_Frecuencia,
        rec.receta_Duracion,
        rec.receta_Instrucciones,
        rec.receta_FechaInicio,
        rec.receta_FechaFin,
        rec.receta_Estado,
        rec.receta_UsuarioCrea,
        usuarioCrea.usu_Nombre AS usuarioCrea,
        rec.receta_FechaCrea,
        rec.receta_UsuarioModifica,
        usuarioModifica.usu_Nombre AS usuarioModifica,
        rec.receta_FechaModifica
    FROM [Medico].[tbRecetas] AS rec
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON rec.receta_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON rec.receta_UsuarioModifica = usuarioModifica.usu_Id
    WHERE rec.receta_Id = @receta_Id
    AND rec.receta_EsEliminado = 0
END
GO


# [Medico].[PR_Medico_Recetas_Insert]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Recetas_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Recetas_Insert]
    @cita_Id INT,
    @masc_Id INT,
    @receta_Medicamento NVARCHAR(200),
    @tipoMed_Id INT = NULL,
    @viaAdmin_Id INT = NULL,
    @receta_Dosis NVARCHAR(100) = NULL,
    @receta_Frecuencia NVARCHAR(100) = NULL,
    @receta_Duracion NVARCHAR(50) = NULL,
    @receta_Instrucciones NVARCHAR(500) = NULL,
    @receta_FechaInicio DATE = NULL,
    @receta_FechaFin DATE = NULL,
    @receta_Estado NVARCHAR(20) = 'Activo',
    @receta_UsuarioCrea INT
AS
BEGIN
    SET NOCOUNT ON

    INSERT INTO [Medico].[tbRecetas]
    (
        cita_Id,
        masc_Id,
        receta_Medicamento,
        tipoMed_Id,
        viaAdmin_Id,
        receta_Dosis,
        receta_Frecuencia,
        receta_Duracion,
        receta_Instrucciones,
        receta_FechaInicio,
        receta_FechaFin,
        receta_Estado,
        receta_UsuarioCrea,
        receta_FechaCrea
    )
    VALUES
    (
        @cita_Id,
        @masc_Id,
        @receta_Medicamento,
        @tipoMed_Id,
        @viaAdmin_Id,
        @receta_Dosis,
        @receta_Frecuencia,
        @receta_Duracion,
        @receta_Instrucciones,
        @receta_FechaInicio,
        @receta_FechaFin,
        @receta_Estado,
        @receta_UsuarioCrea,
        GETDATE()
    )

    SELECT SCOPE_IDENTITY() AS receta_Id
END
GO


# [Medico].[PR_Medico_Recetas_List]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Recetas_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Recetas_List]
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        ROW_NUMBER() OVER(ORDER BY rec.receta_FechaInicio DESC) AS Fila,
        rec.receta_Id,
        rec.cita_Id,
        rec.masc_Id,
        masc.masc_Nombre,
        rec.receta_Medicamento,
        rec.tipoMed_Id,
        tipoMed.tipoMed_Descripcion AS TipoMedicamento,
        rec.viaAdmin_Id,
        viaAdmin.viaAdmin_Descripcion AS ViaAdministracion,
        rec.receta_Dosis,
        rec.receta_Frecuencia,
        rec.receta_Duracion,
        rec.receta_FechaInicio,
        rec.receta_FechaFin,
        rec.receta_Estado
    FROM [Medico].[tbRecetas] AS rec
    INNER JOIN [Refugio].[tbMascotas] AS masc
        ON rec.masc_Id = masc.masc_Id
    LEFT JOIN [Medico].[tbTiposMedicamento] AS tipoMed
        ON rec.tipoMed_Id = tipoMed.tipoMed_Id
    LEFT JOIN [Medico].[tbViasAdministracion] AS viaAdmin
        ON rec.viaAdmin_Id = viaAdmin.viaAdmin_Id
    WHERE rec.receta_EsEliminado = 0
    ORDER BY rec.receta_FechaInicio DESC
END
GO


# [Medico].[PR_Medico_Recetas_Update]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Recetas_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Recetas_Update]
    @receta_Id INT,
    @cita_Id INT,
    @masc_Id INT,
    @receta_Medicamento NVARCHAR(200),
    @tipoMed_Id INT = NULL,
    @viaAdmin_Id INT = NULL,
    @receta_Dosis NVARCHAR(100) = NULL,
    @receta_Frecuencia NVARCHAR(100) = NULL,
    @receta_Duracion NVARCHAR(50) = NULL,
    @receta_Instrucciones NVARCHAR(500) = NULL,
    @receta_FechaInicio DATE = NULL,
    @receta_FechaFin DATE = NULL,
    @receta_Estado NVARCHAR(20) = 'Activo',
    @receta_UsuarioModifica INT
AS
BEGIN
    SET NOCOUNT ON

    UPDATE [Medico].[tbRecetas]
    SET
        cita_Id = @cita_Id,
        masc_Id = @masc_Id,
        receta_Medicamento = @receta_Medicamento,
        tipoMed_Id = @tipoMed_Id,
        viaAdmin_Id = @viaAdmin_Id,
        receta_Dosis = @receta_Dosis,
        receta_Frecuencia = @receta_Frecuencia,
        receta_Duracion = @receta_Duracion,
        receta_Instrucciones = @receta_Instrucciones,
        receta_FechaInicio = @receta_FechaInicio,
        receta_FechaFin = @receta_FechaFin,
        receta_Estado = @receta_Estado,
        receta_UsuarioModifica = @receta_UsuarioModifica,
        receta_FechaModifica = GETDATE()
    WHERE receta_Id = @receta_Id
END
GO


# [Medico].[PR_Medico_TiposConsulta_Delete]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposConsulta_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposConsulta_Delete]
    @tipoCon_Id INT
AS
BEGIN
    SET NOCOUNT ON

    BEGIN TRY
        UPDATE [Medico].[tbTiposConsulta]
        SET tipoCon_EsEliminado = 1
        WHERE tipoCon_Id = @tipoCon_Id

        RETURN 1 -- Éxito
    END TRY
    BEGIN CATCH
        RETURN 0 -- Error
    END CATCH
END
GO


# [Medico].[PR_Medico_TiposConsulta_Detail]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposConsulta_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposConsulta_Detail]
    @tipoCon_Id INT
AS
BEGIN
    SET NOCOUNT ON

    SELECT  tc.tipoCon_Id,
            tc.tipoCon_Descripcion,
            usuarioCrea.usu_Nombre AS UsuarioCreacion,
            tc.tipoCon_FechaCrea,
            usuarioModifica.usu_Nombre AS UsuarioModificacion,
            tc.tipoCon_FechaModifica
    FROM [Medico].[tbTiposConsulta] AS tc
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON tc.tipoCon_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON tc.tipoCon_UsuarioModifica = usuarioModifica.usu_Id
    WHERE tc.tipoCon_EsEliminado = 0
      AND tc.tipoCon_Id = @tipoCon_Id
END
GO


# [Medico].[PR_Medico_TiposConsulta_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposConsulta_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposConsulta_Dropdown]
AS
BEGIN
    SET NOCOUNT ON

    SELECT  tipoCon_Id,
            tipoCon_Descripcion
    FROM [Medico].[tbTiposConsulta]
    WHERE tipoCon_EsEliminado = 0
    ORDER BY tipoCon_Descripcion ASC
END
GO


# [Medico].[PR_Medico_TiposConsulta_Find]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposConsulta_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposConsulta_Find]
    @tipoCon_Id INT
AS BEGIN
    SET NOCOUNT ON
    SELECT tc.tipoCon_Id, tc.tipoCon_Descripcion, tc.tipoCon_EsActivo,
           tc.tipoCon_UsuarioCrea,
           usuarioCrea.usu_Nombre AS usuarioCrea,
           tc.tipoCon_FechaCrea,
           tc.tipoCon_UsuarioModifica,
           usuarioModifica.usu_Nombre AS usuarioModifica,
           tc.tipoCon_FechaModifica
    FROM [Medico].[tbTiposConsulta] AS tc
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON tc.tipoCon_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON tc.tipoCon_UsuarioModifica = usuarioModifica.usu_Id
    WHERE tc.tipoCon_EsEliminado = 0 AND tc.tipoCon_Id = @tipoCon_Id
END
GO


# [Medico].[PR_Medico_TiposConsulta_Insert]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposConsulta_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposConsulta_Insert]
    @tipoCon_Descripcion NVARCHAR(100),
    @tipoCon_EsActivo BIT = 1,
    @tipoCon_UsuarioCrea INT
AS BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        INSERT INTO [Medico].[tbTiposConsulta] (tipoCon_Descripcion, tipoCon_EsActivo, tipoCon_UsuarioCrea, tipoCon_FechaCrea)
        VALUES (@tipoCon_Descripcion, @tipoCon_EsActivo, @tipoCon_UsuarioCrea, GETDATE())
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_TiposConsulta_List]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposConsulta_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposConsulta_List]
AS BEGIN
    SET NOCOUNT ON
    SELECT tipoCon_Id, tipoCon_Descripcion,
           CASE WHEN tipoCon_EsActivo = 1 THEN 'Activo' ELSE 'Inactivo' END AS tipoCon_EsActivo
    FROM [Medico].[tbTiposConsulta]
    WHERE tipoCon_EsEliminado != 1
END

GO


# [Medico].[PR_Medico_TiposConsulta_Update]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposConsulta_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposConsulta_Update]
    @tipoCon_Id INT,
    @tipoCon_Descripcion NVARCHAR(100),
    @tipoCon_EsActivo BIT = 1,
    @tipoCon_UsuarioModifica INT
AS BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        UPDATE [Medico].[tbTiposConsulta]
        SET tipoCon_Descripcion = @tipoCon_Descripcion,
            tipoCon_EsActivo = @tipoCon_EsActivo,
            tipoCon_UsuarioModifica = @tipoCon_UsuarioModifica,
            tipoCon_FechaModifica = GETDATE()
        WHERE tipoCon_Id = @tipoCon_Id
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_TiposEsterilizacion_Delete]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposEsterilizacion_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposEsterilizacion_Delete]
    @tipoEst_Id INT
AS
BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        UPDATE [Medico].[tbTiposEsterilizacion]
        SET tipoEst_EsEliminado = 1
        WHERE tipoEst_Id = @tipoEst_Id
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_TiposEsterilizacion_Detail]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposEsterilizacion_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposEsterilizacion_Detail]
    @tipoEst_Id INT
AS
BEGIN
    SET NOCOUNT ON
    SELECT  te.tipoEst_Id, te.tipoEst_Descripcion, te.tipoEst_Sexo,
            usuarioCrea.usu_Nombre AS UsuarioCreacion,
            te.tipoEst_FechaCrea,
            usuarioModifica.usu_Nombre AS UsuarioModificacion,
            te.tipoEst_FechaModifica
    FROM [Medico].[tbTiposEsterilizacion] AS te
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON te.tipoEst_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON te.tipoEst_UsuarioModifica = usuarioModifica.usu_Id
    WHERE te.tipoEst_EsEliminado = 0 AND te.tipoEst_Id = @tipoEst_Id
END
GO


# [Medico].[PR_Medico_TiposEsterilizacion_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposEsterilizacion_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposEsterilizacion_Dropdown]
AS
BEGIN
    SET NOCOUNT ON
    SELECT tipoEst_Id, tipoEst_Descripcion, tipoEst_Sexo
    FROM [Medico].[tbTiposEsterilizacion]
    WHERE tipoEst_EsEliminado = 0
    ORDER BY tipoEst_Sexo ASC, tipoEst_Descripcion ASC
END
GO


# [Medico].[PR_Medico_TiposEsterilizacion_Find]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposEsterilizacion_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposEsterilizacion_Find]
    @tipoEst_Id INT
AS BEGIN
    SET NOCOUNT ON
    SELECT te.tipoEst_Id, te.tipoEst_Descripcion, te.tipoEst_Sexo, te.tipoEst_EsActivo,
           te.tipoEst_UsuarioCrea,
           usuarioCrea.usu_Nombre AS usuarioCrea,
           te.tipoEst_FechaCrea,
           te.tipoEst_UsuarioModifica,
           usuarioModifica.usu_Nombre AS usuarioModifica,
           te.tipoEst_FechaModifica
    FROM [Medico].[tbTiposEsterilizacion] AS te
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON te.tipoEst_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON te.tipoEst_UsuarioModifica = usuarioModifica.usu_Id
    WHERE te.tipoEst_EsEliminado = 0 AND te.tipoEst_Id = @tipoEst_Id
END
GO


# [Medico].[PR_Medico_TiposEsterilizacion_Insert]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposEsterilizacion_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposEsterilizacion_Insert]
    @tipoEst_Descripcion NVARCHAR(100),
    @tipoEst_Sexo NVARCHAR(10) = NULL,
    @tipoEst_EsActivo BIT = 1,
    @tipoEst_UsuarioCrea INT
AS BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        INSERT INTO [Medico].[tbTiposEsterilizacion] (tipoEst_Descripcion, tipoEst_Sexo, tipoEst_EsActivo, tipoEst_UsuarioCrea, tipoEst_FechaCrea)
        VALUES (@tipoEst_Descripcion, @tipoEst_Sexo, @tipoEst_EsActivo, @tipoEst_UsuarioCrea, GETDATE())
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_TiposEsterilizacion_List]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposEsterilizacion_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposEsterilizacion_List]
AS BEGIN
    SET NOCOUNT ON
    SELECT tipoEst_Id, tipoEst_Descripcion, tipoEst_Sexo,
           CASE WHEN tipoEst_EsActivo = 1 THEN 'Activo' ELSE 'Inactivo' END AS tipoEst_EsActivo
    FROM [Medico].[tbTiposEsterilizacion]
    WHERE tipoEst_EsEliminado != 1
END

GO


# [Medico].[PR_Medico_TiposEsterilizacion_Update]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposEsterilizacion_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposEsterilizacion_Update]
    @tipoEst_Id INT,
    @tipoEst_Descripcion NVARCHAR(100),
    @tipoEst_Sexo NVARCHAR(10) = NULL,
    @tipoEst_EsActivo BIT = 1,
    @tipoEst_UsuarioModifica INT
AS BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        UPDATE [Medico].[tbTiposEsterilizacion]
        SET tipoEst_Descripcion = @tipoEst_Descripcion,
            tipoEst_Sexo = @tipoEst_Sexo,
            tipoEst_EsActivo = @tipoEst_EsActivo,
            tipoEst_UsuarioModifica = @tipoEst_UsuarioModifica,
            tipoEst_FechaModifica = GETDATE()
        WHERE tipoEst_Id = @tipoEst_Id
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_TiposMedicamento_Delete]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposMedicamento_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposMedicamento_Delete]
    @tipoMed_Id INT
AS
BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        UPDATE [Medico].[tbTiposMedicamento]
        SET tipoMed_EsEliminado = 1
        WHERE tipoMed_Id = @tipoMed_Id
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_TiposMedicamento_Detail]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposMedicamento_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposMedicamento_Detail]
    @tipoMed_Id INT
AS
BEGIN
    SET NOCOUNT ON
    SELECT  tm.tipoMed_Id, tm.tipoMed_Descripcion,
            usuarioCrea.usu_Nombre AS UsuarioCreacion,
            tm.tipoMed_FechaCrea,
            usuarioModifica.usu_Nombre AS UsuarioModificacion,
            tm.tipoMed_FechaModifica
    FROM [Medico].[tbTiposMedicamento] AS tm
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON tm.tipoMed_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON tm.tipoMed_UsuarioModifica = usuarioModifica.usu_Id
    WHERE tm.tipoMed_EsEliminado = 0 AND tm.tipoMed_Id = @tipoMed_Id
END
GO


# [Medico].[PR_Medico_TiposMedicamento_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposMedicamento_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposMedicamento_Dropdown]
AS
BEGIN
    SET NOCOUNT ON
    SELECT tipoMed_Id, tipoMed_Descripcion
    FROM [Medico].[tbTiposMedicamento]
    WHERE tipoMed_EsEliminado = 0
    ORDER BY tipoMed_Descripcion ASC
END
GO


# [Medico].[PR_Medico_TiposMedicamento_Find]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposMedicamento_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposMedicamento_Find]
    @tipoMed_Id INT
AS BEGIN
    SET NOCOUNT ON
    SELECT tm.tipoMed_Id, tm.tipoMed_Descripcion, tm.tipoMed_EsActivo,
           tm.tipoMed_UsuarioCrea,
           usuarioCrea.usu_Nombre AS usuarioCrea,
           tm.tipoMed_FechaCrea,
           tm.tipoMed_UsuarioModifica,
           usuarioModifica.usu_Nombre AS usuarioModifica,
           tm.tipoMed_FechaModifica
    FROM [Medico].[tbTiposMedicamento] AS tm
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON tm.tipoMed_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON tm.tipoMed_UsuarioModifica = usuarioModifica.usu_Id
    WHERE tm.tipoMed_EsEliminado = 0 AND tm.tipoMed_Id = @tipoMed_Id
END
GO


# [Medico].[PR_Medico_TiposMedicamento_Insert]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposMedicamento_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposMedicamento_Insert]
    @tipoMed_Descripcion NVARCHAR(100),
    @tipoMed_EsActivo BIT = 1,
    @tipoMed_UsuarioCrea INT
AS BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        INSERT INTO [Medico].[tbTiposMedicamento] (tipoMed_Descripcion, tipoMed_EsActivo, tipoMed_UsuarioCrea, tipoMed_FechaCrea)
        VALUES (@tipoMed_Descripcion, @tipoMed_EsActivo, @tipoMed_UsuarioCrea, GETDATE())
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_TiposMedicamento_List]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposMedicamento_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposMedicamento_List]
AS BEGIN
    SET NOCOUNT ON
    SELECT tipoMed_Id, tipoMed_Descripcion,
           CASE WHEN tipoMed_EsActivo = 1 THEN 'Activo' ELSE 'Inactivo' END AS tipoMed_EsActivo
    FROM [Medico].[tbTiposMedicamento]
    WHERE tipoMed_EsEliminado != 1
END

GO


# [Medico].[PR_Medico_TiposMedicamento_Update]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposMedicamento_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposMedicamento_Update]
    @tipoMed_Id INT,
    @tipoMed_Descripcion NVARCHAR(100),
    @tipoMed_EsActivo BIT = 1,
    @tipoMed_UsuarioModifica INT
AS BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        UPDATE [Medico].[tbTiposMedicamento]
        SET tipoMed_Descripcion = @tipoMed_Descripcion,
            tipoMed_EsActivo = @tipoMed_EsActivo,
            tipoMed_UsuarioModifica = @tipoMed_UsuarioModifica,
            tipoMed_FechaModifica = GETDATE()
        WHERE tipoMed_Id = @tipoMed_Id
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_TiposParasito_Delete]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposParasito_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposParasito_Delete]
    @tipoPar_Id INT
AS
BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        UPDATE [Medico].[tbTiposParasito]
        SET tipoPar_EsEliminado = 1
        WHERE tipoPar_Id = @tipoPar_Id
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_TiposParasito_Detail]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposParasito_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposParasito_Detail]
    @tipoPar_Id INT
AS
BEGIN
    SET NOCOUNT ON
    SELECT  tp.tipoPar_Id, tp.tipoPar_Descripcion, tp.tipoPar_Categoria,
            usuarioCrea.usu_Nombre AS UsuarioCreacion,
            tp.tipoPar_FechaCrea,
            usuarioModifica.usu_Nombre AS UsuarioModificacion,
            tp.tipoPar_FechaModifica
    FROM [Medico].[tbTiposParasito] AS tp
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON tp.tipoPar_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON tp.tipoPar_UsuarioModifica = usuarioModifica.usu_Id
    WHERE tp.tipoPar_EsEliminado = 0 AND tp.tipoPar_Id = @tipoPar_Id
END
GO


# [Medico].[PR_Medico_TiposParasito_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposParasito_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposParasito_Dropdown]
AS
BEGIN
    SET NOCOUNT ON
    SELECT tipoPar_Id, tipoPar_Descripcion, tipoPar_Categoria
    FROM [Medico].[tbTiposParasito]
    WHERE tipoPar_EsEliminado = 0
    ORDER BY tipoPar_Categoria ASC, tipoPar_Descripcion ASC
END
GO


# [Medico].[PR_Medico_TiposParasito_Find]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposParasito_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposParasito_Find]
    @tipoPar_Id INT
AS BEGIN
    SET NOCOUNT ON
    SELECT tp.tipoPar_Id, tp.tipoPar_Descripcion, tp.tipoPar_Categoria, tp.tipoPar_EsActivo,
           tp.tipoPar_UsuarioCrea,
           usuarioCrea.usu_Nombre AS usuarioCrea,
           tp.tipoPar_FechaCrea,
           tp.tipoPar_UsuarioModifica,
           usuarioModifica.usu_Nombre AS usuarioModifica,
           tp.tipoPar_FechaModifica
    FROM [Medico].[tbTiposParasito] AS tp
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON tp.tipoPar_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON tp.tipoPar_UsuarioModifica = usuarioModifica.usu_Id
    WHERE tp.tipoPar_EsEliminado = 0 AND tp.tipoPar_Id = @tipoPar_Id
END
GO


# [Medico].[PR_Medico_TiposParasito_Insert]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposParasito_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposParasito_Insert]
    @tipoPar_Descripcion NVARCHAR(100),
    @tipoPar_Categoria NVARCHAR(50) = NULL,
    @tipoPar_EsActivo BIT = 1,
    @tipoPar_UsuarioCrea INT
AS BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        INSERT INTO [Medico].[tbTiposParasito] (tipoPar_Descripcion, tipoPar_Categoria, tipoPar_EsActivo, tipoPar_UsuarioCrea, tipoPar_FechaCrea)
        VALUES (@tipoPar_Descripcion, @tipoPar_Categoria, @tipoPar_EsActivo, @tipoPar_UsuarioCrea, GETDATE())
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_TiposParasito_List]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposParasito_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO
CREATE PROCEDURE [Medico].[PR_Medico_TiposParasito_List]
AS BEGIN
    SET NOCOUNT ON
    SELECT tipoPar_Id, tipoPar_Descripcion, tipoPar_Categoria,
           CASE WHEN tipoPar_EsActivo = 1 THEN 'Activo' ELSE 'Inactivo' END AS tipoPar_EsActivo
    FROM [Medico].[tbTiposParasito]
    WHERE tipoPar_EsEliminado != 1
END

GO


# [Medico].[PR_Medico_TiposParasito_Update]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_TiposParasito_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_TiposParasito_Update]
    @tipoPar_Id INT,
    @tipoPar_Descripcion NVARCHAR(100),
    @tipoPar_Categoria NVARCHAR(50) = NULL,
    @tipoPar_EsActivo BIT = 1,
    @tipoPar_UsuarioModifica INT
AS BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        UPDATE [Medico].[tbTiposParasito]
        SET tipoPar_Descripcion = @tipoPar_Descripcion,
            tipoPar_Categoria = @tipoPar_Categoria,
            tipoPar_EsActivo = @tipoPar_EsActivo,
            tipoPar_UsuarioModifica = @tipoPar_UsuarioModifica,
            tipoPar_FechaModifica = GETDATE()
        WHERE tipoPar_Id = @tipoPar_Id
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_Tratamientos_Delete]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Tratamientos_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Tratamientos_Delete]
    @trat_Id INT,
    @trat_UsuarioModifica INT
AS
BEGIN
    SET NOCOUNT ON

    UPDATE [Medico].[tbTratamientos]
    SET
        trat_EsEliminado = 1,
        trat_UsuarioModifica = @trat_UsuarioModifica,
        trat_FechaModifica = GETDATE()
    WHERE trat_Id = @trat_Id
END
GO


# [Medico].[PR_Medico_Tratamientos_Detail]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Tratamientos_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Tratamientos_Detail]
    @trat_Id INT
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        trat.trat_Id,
        trat.masc_Id,
        masc.masc_Nombre AS Mascota,
        trat.cita_Id,
        trat.receta_Id,
        trat.tipoPar_Id,
        tipoPar.tipoPar_Descripcion AS TipoParasito,
        tipoPar.tipoPar_Categoria AS CategoriaParasito,
        trat.trat_ParasitoDetectado,
        trat.trat_Medicamento,
        trat.tipoMed_Id,
        tipoMed.tipoMed_Descripcion AS TipoMedicamento,
        trat.viaAdmin_Id,
        viaAdmin.viaAdmin_Descripcion AS ViaAdministracion,
        trat.trat_FechaAplicacion,
        trat.trat_AplicadoPor,
        trat.trat_ProximaDosis,
        trat.trat_Estado,
        trat.trat_Observaciones,
        usuarioCrea.usu_Nombre AS UsuarioCreacion,
        trat.trat_FechaCrea,
        usuarioModifica.usu_Nombre AS UsuarioModificacion,
        trat.trat_FechaModifica
    FROM [Medico].[tbTratamientos] AS trat
    INNER JOIN [Refugio].[tbMascotas] AS masc
        ON trat.masc_Id = masc.masc_Id
    LEFT JOIN [Medico].[tbTiposParasito] AS tipoPar
        ON trat.tipoPar_Id = tipoPar.tipoPar_Id
    LEFT JOIN [Medico].[tbTiposMedicamento] AS tipoMed
        ON trat.tipoMed_Id = tipoMed.tipoMed_Id
    LEFT JOIN [Medico].[tbViasAdministracion] AS viaAdmin
        ON trat.viaAdmin_Id = viaAdmin.viaAdmin_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON trat.trat_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON trat.trat_UsuarioModifica = usuarioModifica.usu_Id
    WHERE trat.trat_Id = @trat_Id
    AND trat.trat_EsEliminado = 0
END
GO


# [Medico].[PR_Medico_Tratamientos_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Tratamientos_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Tratamientos_Dropdown]
    @masc_Id INT = NULL
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        trat_Id,
        CONCAT(trat_Medicamento, ' - ', FORMAT(trat_FechaAplicacion, 'dd/MM/yyyy')) AS Descripcion
    FROM [Medico].[tbTratamientos]
    WHERE trat_EsEliminado = 0
    AND (@masc_Id IS NULL OR masc_Id = @masc_Id)
    ORDER BY trat_FechaAplicacion DESC
END
GO


# [Medico].[PR_Medico_Tratamientos_Find]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Tratamientos_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Tratamientos_Find]
    @trat_Id INT
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        trat.trat_Id,
        trat.masc_Id,
        trat.cita_Id,
        trat.receta_Id,
        trat.tipoPar_Id,
        trat.trat_ParasitoDetectado,
        trat.trat_Medicamento,
        trat.tipoMed_Id,
        trat.viaAdmin_Id,
        trat.trat_FechaAplicacion,
        trat.trat_AplicadoPor,
        trat.trat_ProximaDosis,
        trat.trat_Estado,
        trat.trat_Observaciones,
        trat.trat_UsuarioCrea,
        usuarioCrea.usu_Nombre AS usuarioCrea,
        trat.trat_FechaCrea,
        trat.trat_UsuarioModifica,
        usuarioModifica.usu_Nombre AS usuarioModifica,
        trat.trat_FechaModifica
    FROM [Medico].[tbTratamientos] AS trat
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON trat.trat_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON trat.trat_UsuarioModifica = usuarioModifica.usu_Id
    WHERE trat.trat_Id = @trat_Id
    AND trat.trat_EsEliminado = 0
END
GO


# [Medico].[PR_Medico_Tratamientos_Insert]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Tratamientos_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Tratamientos_Insert]
    @masc_Id INT,
    @cita_Id INT = NULL,
    @receta_Id INT = NULL,
    @tipoPar_Id INT = NULL,
    @trat_ParasitoDetectado NVARCHAR(200) = NULL,
    @trat_Medicamento NVARCHAR(200) = NULL,
    @tipoMed_Id INT = NULL,
    @viaAdmin_Id INT = NULL,
    @trat_FechaAplicacion DATETIME,
    @trat_AplicadoPor NVARCHAR(100) = NULL,
    @trat_ProximaDosis DATE = NULL,
    @trat_Estado NVARCHAR(20) = 'Iniciado',
    @trat_Observaciones NVARCHAR(500) = NULL,
    @trat_UsuarioCrea INT
AS
BEGIN
    SET NOCOUNT ON

    INSERT INTO [Medico].[tbTratamientos]
    (
        masc_Id,
        cita_Id,
        receta_Id,
        tipoPar_Id,
        trat_ParasitoDetectado,
        trat_Medicamento,
        tipoMed_Id,
        viaAdmin_Id,
        trat_FechaAplicacion,
        trat_AplicadoPor,
        trat_ProximaDosis,
        trat_Estado,
        trat_Observaciones,
        trat_UsuarioCrea,
        trat_FechaCrea
    )
    VALUES
    (
        @masc_Id,
        @cita_Id,
        @receta_Id,
        @tipoPar_Id,
        @trat_ParasitoDetectado,
        @trat_Medicamento,
        @tipoMed_Id,
        @viaAdmin_Id,
        @trat_FechaAplicacion,
        @trat_AplicadoPor,
        @trat_ProximaDosis,
        @trat_Estado,
        @trat_Observaciones,
        @trat_UsuarioCrea,
        GETDATE()
    )

    SELECT SCOPE_IDENTITY() AS trat_Id
END
GO


# [Medico].[PR_Medico_Tratamientos_List]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Tratamientos_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Tratamientos_List]
AS
BEGIN
    SET NOCOUNT ON

    SELECT
        ROW_NUMBER() OVER(ORDER BY trat.trat_FechaAplicacion DESC) AS Fila,
        trat.trat_Id,
        trat.masc_Id,
        masc.masc_Nombre AS Mascota,
        trat.tipoPar_Id,
        tipoPar.tipoPar_Descripcion AS TipoParasito,
        tipoPar.tipoPar_Categoria AS CategoriaParasito,
        trat.trat_ParasitoDetectado,
        trat.trat_Medicamento,
        trat.tipoMed_Id,
        tipoMed.tipoMed_Descripcion AS TipoMedicamento,
        trat.viaAdmin_Id,
        viaAdmin.viaAdmin_Descripcion AS ViaAdministracion,
        trat.trat_FechaAplicacion,
        trat.trat_ProximaDosis,
        trat.trat_Estado
    FROM [Medico].[tbTratamientos] AS trat
    INNER JOIN [Refugio].[tbMascotas] AS masc
        ON trat.masc_Id = masc.masc_Id
    LEFT JOIN [Medico].[tbTiposParasito] AS tipoPar
        ON trat.tipoPar_Id = tipoPar.tipoPar_Id
    LEFT JOIN [Medico].[tbTiposMedicamento] AS tipoMed
        ON trat.tipoMed_Id = tipoMed.tipoMed_Id
    LEFT JOIN [Medico].[tbViasAdministracion] AS viaAdmin
        ON trat.viaAdmin_Id = viaAdmin.viaAdmin_Id
    WHERE trat.trat_EsEliminado = 0
    ORDER BY trat.trat_FechaAplicacion DESC
END
GO


# [Medico].[PR_Medico_Tratamientos_Update]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_Tratamientos_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Medico].[PR_Medico_Tratamientos_Update]
    @trat_Id INT,
    @masc_Id INT,
    @cita_Id INT = NULL,
    @receta_Id INT = NULL,
    @tipoPar_Id INT = NULL,
    @trat_ParasitoDetectado NVARCHAR(200) = NULL,
    @trat_Medicamento NVARCHAR(200) = NULL,
    @tipoMed_Id INT = NULL,
    @viaAdmin_Id INT = NULL,
    @trat_FechaAplicacion DATETIME,
    @trat_AplicadoPor NVARCHAR(100) = NULL,
    @trat_ProximaDosis DATE = NULL,
    @trat_Estado NVARCHAR(20) = 'Iniciado',
    @trat_Observaciones NVARCHAR(500) = NULL,
    @trat_UsuarioModifica INT
AS
BEGIN
    SET NOCOUNT ON

    UPDATE [Medico].[tbTratamientos]
    SET
        masc_Id = @masc_Id,
        cita_Id = @cita_Id,
        receta_Id = @receta_Id,
        tipoPar_Id = @tipoPar_Id,
        trat_ParasitoDetectado = @trat_ParasitoDetectado,
        trat_Medicamento = @trat_Medicamento,
        tipoMed_Id = @tipoMed_Id,
        viaAdmin_Id = @viaAdmin_Id,
        trat_FechaAplicacion = @trat_FechaAplicacion,
        trat_AplicadoPor = @trat_AplicadoPor,
        trat_ProximaDosis = @trat_ProximaDosis,
        trat_Estado = @trat_Estado,
        trat_Observaciones = @trat_Observaciones,
        trat_UsuarioModifica = @trat_UsuarioModifica,
        trat_FechaModifica = GETDATE()
    WHERE trat_Id = @trat_Id
END
GO


# [Medico].[PR_Medico_ViasAdministracion_Delete]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_ViasAdministracion_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_ViasAdministracion_Delete]
    @viaAdmin_Id INT
AS
BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        UPDATE [Medico].[tbViasAdministracion]
        SET viaAdmin_EsEliminado = 1
        WHERE viaAdmin_Id = @viaAdmin_Id
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_ViasAdministracion_Detail]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_ViasAdministracion_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_ViasAdministracion_Detail]
    @viaAdmin_Id INT
AS
BEGIN
    SET NOCOUNT ON
    SELECT  va.viaAdmin_Id, va.viaAdmin_Descripcion,
            usuarioCrea.usu_Nombre AS UsuarioCreacion,
            va.viaAdmin_FechaCrea,
            usuarioModifica.usu_Nombre AS UsuarioModificacion,
            va.viaAdmin_FechaModifica
    FROM [Medico].[tbViasAdministracion] AS va
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON va.viaAdmin_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON va.viaAdmin_UsuarioModifica = usuarioModifica.usu_Id
    WHERE va.viaAdmin_EsEliminado = 0 AND va.viaAdmin_Id = @viaAdmin_Id
END
GO


# [Medico].[PR_Medico_ViasAdministracion_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_ViasAdministracion_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Medico].[PR_Medico_ViasAdministracion_Dropdown]
AS
BEGIN
    SET NOCOUNT ON
    SELECT viaAdmin_Id, viaAdmin_Descripcion
    FROM [Medico].[tbViasAdministracion]
    WHERE viaAdmin_EsEliminado = 0
    ORDER BY viaAdmin_Descripcion ASC
END
GO


# [Medico].[PR_Medico_ViasAdministracion_Find]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_ViasAdministracion_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_ViasAdministracion_Find]
    @viaAdmin_Id INT
AS BEGIN
    SET NOCOUNT ON
    SELECT va.viaAdmin_Id, va.viaAdmin_Descripcion, va.viaAdmin_EsActivo,
           va.viaAdmin_UsuarioCrea,
           usuarioCrea.usu_Nombre AS usuarioCrea,
           va.viaAdmin_FechaCrea,
           va.viaAdmin_UsuarioModifica,
           usuarioModifica.usu_Nombre AS usuarioModifica,
           va.viaAdmin_FechaModifica
    FROM [Medico].[tbViasAdministracion] AS va
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON va.viaAdmin_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON va.viaAdmin_UsuarioModifica = usuarioModifica.usu_Id
    WHERE va.viaAdmin_EsEliminado = 0 AND va.viaAdmin_Id = @viaAdmin_Id
END
GO


# [Medico].[PR_Medico_ViasAdministracion_Insert]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_ViasAdministracion_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_ViasAdministracion_Insert]
    @viaAdmin_Descripcion NVARCHAR(100),
    @viaAdmin_EsActivo BIT = 1,
    @viaAdmin_UsuarioCrea INT
AS BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        INSERT INTO [Medico].[tbViasAdministracion] (viaAdmin_Descripcion, viaAdmin_EsActivo, viaAdmin_UsuarioCrea, viaAdmin_FechaCrea)
        VALUES (@viaAdmin_Descripcion, @viaAdmin_EsActivo, @viaAdmin_UsuarioCrea, GETDATE())
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Medico].[PR_Medico_ViasAdministracion_List]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_ViasAdministracion_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO
CREATE PROCEDURE [Medico].[PR_Medico_ViasAdministracion_List]
AS BEGIN
    SET NOCOUNT ON
    SELECT viaAdmin_Id, viaAdmin_Descripcion,
           CASE WHEN viaAdmin_EsActivo = 1 THEN 'Activo' ELSE 'Inactivo' END AS viaAdmin_EsActivo
    FROM [Medico].[tbViasAdministracion]
    WHERE viaAdmin_EsEliminado != 1
END

GO


# [Medico].[PR_Medico_ViasAdministracion_Update]

In [0]:
/****** Object:  StoredProcedure [Medico].[PR_Medico_ViasAdministracion_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Medico].[PR_Medico_ViasAdministracion_Update]
    @viaAdmin_Id INT,
    @viaAdmin_Descripcion NVARCHAR(100),
    @viaAdmin_EsActivo BIT = 1,
    @viaAdmin_UsuarioModifica INT
AS BEGIN
    SET NOCOUNT ON
    BEGIN TRY
        UPDATE [Medico].[tbViasAdministracion]
        SET viaAdmin_Descripcion = @viaAdmin_Descripcion,
            viaAdmin_EsActivo = @viaAdmin_EsActivo,
            viaAdmin_UsuarioModifica = @viaAdmin_UsuarioModifica,
            viaAdmin_FechaModifica = GETDATE()
        WHERE viaAdmin_Id = @viaAdmin_Id
        RETURN 1
    END TRY
    BEGIN CATCH
        RETURN 0
    END CATCH
END
GO


# [Refugio].[PR_Refugio_Adopcion_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Adopcion_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Adopcion_List]
AS
BEGIN
    SELECT  
        m.masc_Id,
        m.masc_Nombre,
        r.raza_Descripcion,
        r.raza_TipoAnimal,
        m.masc_Edad,
        m.masc_Sexo,
        m.masc_EsAdoptado,
        m.masc_EsReservado,
        COUNT(a.adop_Id) AS CantidadSolicitantes
    FROM 
        [Refugio].[tbMascotas] AS m
    INNER JOIN 
        [Refugio].[tbRazas] AS r ON m.raza_Id = r.raza_Id
    LEFT JOIN 
        [Refugio].[tbSolicitudes] AS s ON m.masc_Id = s.masc_Id
    LEFT JOIN 
        [Refugio].[tbAdopciones] AS a ON s.sol_Id = a.sol_Id AND a.adop_EsEliminado != 1
    WHERE 
        m.masc_EsEliminado != 1
        AND m.masc_EsAdoptado != 1
    GROUP BY 
        m.masc_Id, 
        m.masc_Nombre, 
        r.raza_Descripcion, 
        r.raza_TipoAnimal,
        m.masc_Edad, 
        m.masc_Sexo, 
        m.masc_EsAdoptado, 
        m.masc_EsReservado
    ORDER BY 
        CantidadSolicitantes DESC
END
GO


# [Refugio].[PR_Refugio_Adopciones_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Adopciones_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-----------------> DELETE

CREATE PROCEDURE [Refugio].[PR_Refugio_Adopciones_Delete] 
	@adop_Id INT
AS
  BEGIN
          UPDATE [Refugio].[tbAdopciones]
          SET    adop_EsEliminado	= 1
          WHERE  adop_Id		= @adop_Id
  END
GO


# [Refugio].[PR_Refugio_Adopciones_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Adopciones_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Adopciones_Detail]  
@masc_Id INT
AS BEGIN
SELECT	solicitudes.sol_Id,
		solicitudes.sol_Identidad,
		solicitudes.sol_Nombres,
		solicitudes.sol_Apellidos,
		solicitudes.sol_Telefono,
		solicitudes.sol_Correo,
		mascota.masc_Id,
		mascota.masc_Imagen,
		mascota.masc_Nombre,
		mascota.masc_EsAdoptado,
		raza.raza_Descripcion,
		albergue.refg_Nombre,
		adopcion.adop_Estado,
		usuarioCrea.usu_Nombre AS sol_NombreUsuarioCrea,
		solicitudes.sol_FechaCrea,
		usuarioModifica.usu_Nombre AS sol_NombreUsuarioModifica,
		solicitudes.sol_FechaModifica
		FROM [Refugio].[tbSolicitudes] AS solicitudes
			INNER JOIN [Refugio].[tbMascotas] AS mascota 
			ON solicitudes.masc_Id = mascota.masc_Id
			LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
			ON		solicitudes.sol_UsuarioCrea = usuarioCrea.usu_Id 
			LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
			ON		solicitudes.sol_UsuarioModifica = usuarioModifica.usu_Id
			INNER JOIN  [Refugio].[tbRazas] AS raza
			ON			mascota.raza_Id = raza.raza_Id
			INNER JOIN	[Refugio].[tbRefugios] AS albergue
			ON			mascota.refg_Id = albergue.refg_Id
			LEFT JOIN (SELECT sol_Id, adop_Estado, adop_FechaCrea,
							   ROW_NUMBER() OVER (PARTITION BY sol_Id ORDER BY adop_FechaCrea DESC) AS rn
						FROM [Refugio].[tbAdopciones]
					) adopcion ON adopcion.sol_Id = solicitudes.sol_Id AND adopcion.rn = 1
WHERE solicitudes.sol_EsEliminado != 1
AND 	mascota.masc_Id = @masc_Id
END
GO


# [Refugio].[PR_Refugio_Adopciones_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Adopciones_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbAdopciones
CREATE PROCEDURE [Refugio].[PR_Refugio_Adopciones_Find]
@adop_Id INT
AS
BEGIN
    SELECT	adop_Id,
			sol_Id,
			adop_EsAprobado,
			adop_UsuarioCrea,
			usuarioCrea.Usu_Nombre AS usuarioCrea,
			adop_FechaCrea,
			adop_UsuarioModifica,
			usuarioModifica.Usu_Nombre AS usuarioModifica,
			adop_FechaModifica
    FROM [Refugio].[tbAdopciones] AS fichaadopcion
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		fichaadopcion.adop_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		fichaadopcion.adop_UsuarioModifica = usuarioModifica.usu_Id
    WHERE fichaadopcion.adop_EsEliminado != 1
	AND 	adop_Id = @adop_Id
END
GO


# [Refugio].[PR_Refugio_Adopciones_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Adopciones_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Adopciones_Insert] 
    @sol_Id INT,
    @adop_Estado VARCHAR(15),
    @adop_UsuarioCrea INT
AS BEGIN
INSERT INTO [Refugio].[tbAdopciones] (
        sol_Id,
        adop_Estado,
        adop_UsuarioCrea,
        adop_FechaCrea
    )
VALUES
    (
        @sol_Id,
        @adop_Estado,
		@adop_UsuarioCrea,
        Getdate()
    )
END
GO


# [Refugio].[PR_Refugio_Adopciones_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Adopciones_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Adopciones_List]
AS
BEGIN
       SELECT  
        m.masc_Id,
        m.masc_Nombre,
        r.raza_Descripcion,
        r.raza_TipoAnimal,
        m.masc_Edad,
        m.masc_Sexo,
        m.masc_EsAdoptado,
        m.masc_EsReservado,
        COUNT(s.sol_Id) AS CantidadSolicitantes
    FROM 
        [Refugio].[tbMascotas] AS m
    INNER JOIN 
        [Refugio].[tbRazas] AS r ON m.raza_Id = r.raza_Id
    INNER JOIN 
        [Refugio].[tbSolicitudes] AS s ON m.masc_Id = s.masc_Id 
    WHERE 
        m.masc_EsEliminado != 1  
		--and m.masc_EsAdoptado = 0
    GROUP BY 
        m.masc_Id, 
        m.masc_Nombre, 
        r.raza_Descripcion, 
        r.raza_TipoAnimal,
        m.masc_Edad, 
        m.masc_Sexo, 
        m.masc_EsAdoptado, 
        m.masc_EsReservado
    ORDER BY 
        CantidadSolicitantes DESC
END
GO


# [Refugio].[PR_Refugio_Adopciones_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Adopciones_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-----------------> UPDATE
CREATE PROCEDURE [Refugio].[PR_Refugio_Adopciones_Update] 
@adop_Id INT,
@sol_Id INT,
@adop_EsAprobado BIT,
@adop_UsuarioModifica INT 
AS BEGIN
UPDATE [Refugio].[tbAdopciones]
SET sol_Id = @sol_Id,
  adop_EsAprobado = @adop_EsAprobado,
  adop_UsuarioModifica = @adop_UsuarioModifica,
  adop_FechaModifica = Getdate()
WHERE adop_Id = @adop_Id
END 

GO


# [Refugio].[PR_Refugio_CitaMedica_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_CitaMedica_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


-----------------> DELETE

CREATE PROCEDURE [Refugio].[PR_Refugio_CitaMedica_Delete] 
	@medic_Id INT
AS
  BEGIN
          UPDATE [Refugio].[tbHistorialMedico]
          SET    medic_EsEliminado	= 1
          WHERE  medic_Id		= @medic_Id
  END
GO


# [Refugio].[PR_Refugio_CitaMedica_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_CitaMedica_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_CitaMedica_Detail] 
@medic_Id INT
AS
BEGIN
    SELECT  medic_Id, 
            mascotas.masc_Id,
            mascotas.masc_Nombre, 
            razas.raza_Descripcion,
            historialMedico.com_Id,
            comportamientos.com_Descripcion, 
            medic_FechaConsulta,
            medic_TipoConsulta,
            medic_MotivoConsulta,
            medic_Diagnostico,
            medic_Peso,
            medic_Temperatura,
            medic_FrecuenciaCardiaca,
            medic_FrecuenciaRespiratoria,
            vac_Id,
            medic_MedicamentosRecetados,
            medic_Dosificacion,
            medic_ProcedimientosRealizados,
            medic_ResultadosExamenes,
            medic_ProximaCita,
            medic_MotivoProximaCita,
            historialMedico.medic_UsuarioCrea,
            usuarioCrea.usu_Nombre AS UsuarioCreacion, 
            medic_FechaCrea, 
            historialMedico.medic_UsuarioModifica,
            usuarioModifica.usu_Nombre AS UsuarioModificacion, 
            medic_FechaModifica
    FROM    [Refugio].tbCitaMedica AS historialMedico
    INNER JOIN  [Refugio].[tbMascotas] AS mascotas
    ON  historialMedico.masc_Id = mascotas.masc_Id
    INNER JOIN  [Refugio].[tbRazas] AS razas
    ON      mascotas.raza_Id = razas.raza_Id
    INNER JOIN [Refugio].[tbComportamientos] AS comportamientos
    ON          historialMedico.com_Id = comportamientos.com_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
    ON      historialMedico.medic_UsuarioCrea = usuarioCrea.usu_Id 
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
    ON      historialMedico.medic_UsuarioModifica = usuarioModifica.usu_Id
    WHERE   medic_EsEliminado != 1
    AND     historialMedico.medic_Id = @medic_Id
END
GO


# [Refugio].[PR_Refugio_CitaMedica_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_CitaMedica_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_CitaMedica_Find]
@medic_Id INT
AS
BEGIN
    SELECT  citaMedica.medic_Id,
            mascota.masc_Nombre,
            citaMedica.masc_Id,
            citaMedica.com_Id,
			comportamientos.com_Descripcion,
            citaMedica.medic_FechaConsulta,
            citaMedica.medic_TipoConsulta,
            citaMedica.medic_MotivoConsulta,
            citaMedica.medic_Diagnostico,
            citaMedica.medic_Peso,
            citaMedica.medic_Temperatura,
            citaMedica.medic_FrecuenciaCardiaca,
            citaMedica.medic_FrecuenciaRespiratoria,
            citaMedica.vac_Id,
            vacunas.vac_Descripcion,
            citaMedica.medic_MedicamentosRecetados,
            citaMedica.medic_Dosificacion,
            citaMedica.medic_ProcedimientosRealizados,
            citaMedica.medic_ResultadosExamenes,
            citaMedica.medic_ProximaCita,
            citaMedica.medic_MotivoProximaCita,
            citaMedica.medic_UsuarioCrea,
			usuarioCrea.Usu_Nombre AS usuarioCrea,
            citaMedica.medic_FechaCrea,
            citaMedica.medic_UsuarioModifica,
			usuarioModifica.Usu_Nombre AS usuarioModifica,
            citaMedica.medic_FechaModifica
    FROM [Refugio].[tbCitaMedica] AS citaMedica
    INNER JOIN [Refugio].[tbMascotas] AS mascota
    ON			citaMedica.masc_Id = mascota.masc_Id
	INNER JOIN [Refugio].[tbComportamientos] AS comportamientos
	ON			citaMedica.com_Id = comportamientos.com_Id
    LEFT JOIN   [Refugio].[tbVacunas] AS vacunas
    ON          citaMedica.vac_Id = vacunas.vac_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		citaMedica.medic_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		citaMedica.medic_UsuarioModifica = usuarioModifica.usu_Id
    WHERE citaMedica.medic_EsEliminado != 1
	AND 	citaMedica.medic_Id = @medic_Id
END
GO


# [Refugio].[PR_Refugio_CitaMedica_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_CitaMedica_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_CitaMedica_Insert] 
    @masc_Id INT,
    @com_Id INT,
    @medic_FechaConsulta DATETIME,
    @medic_TipoConsulta NVARCHAR(255) = NULL,
    @medic_MotivoConsulta NVARCHAR(255) = NULL,
    @medic_Diagnostico NVARCHAR(255) = NULL,
    @medic_Peso INT = NULL,
    @medic_Temperatura INT = NULL,
    @medic_FrecuenciaCardiaca INT = NULL,
    @medic_FrecuenciaRespiratoria INT = NULL,
    @vac_Id INT = NULL,
    @medic_MedicamentosRecetados NVARCHAR(255) = NULL,
    @medic_Dosificacion NVARCHAR(255) = NULL,
    @medic_ProcedimientosRealizados NVARCHAR(255) = NULL,
    @medic_ResultadosExamenes NVARCHAR(255) = NULL,
    @medic_ProximaCita DATETIME = NULL,
    @medic_MotivoProximaCita NVARCHAR(255) = NULL,
    @medic_UsuarioCrea INT
AS 
BEGIN
    INSERT INTO [Refugio].[tbCitaMedica] (
        masc_Id,
        com_Id,
        medic_FechaConsulta,
        medic_TipoConsulta,
        medic_MotivoConsulta,
        medic_Diagnostico,
        medic_Peso,
        medic_Temperatura,
        medic_FrecuenciaCardiaca,
        medic_FrecuenciaRespiratoria,
        vac_Id,
        medic_MedicamentosRecetados,
        medic_Dosificacion,
        medic_ProcedimientosRealizados,
        medic_ResultadosExamenes,
        medic_ProximaCita,
        medic_MotivoProximaCita,
        medic_EsEliminado,
        medic_UsuarioCrea,
        medic_FechaCrea
    )
    VALUES (
        @masc_Id,
        @com_Id,
        @medic_FechaConsulta,
        @medic_TipoConsulta,
        @medic_MotivoConsulta,
        @medic_Diagnostico,
        @medic_Peso,
        @medic_Temperatura,
        @medic_FrecuenciaCardiaca,
        @medic_FrecuenciaRespiratoria,
        @vac_Id,
        @medic_MedicamentosRecetados,
        @medic_Dosificacion,
        @medic_ProcedimientosRealizados,
        @medic_ResultadosExamenes,
        @medic_ProximaCita,
        @medic_MotivoProximaCita,
        0, -- medic_EsEliminado
        @medic_UsuarioCrea,
        GETDATE() -- medic_FechaCrea
    )
END
GO


# [Refugio].[PR_Refugio_CitaMedica_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_CitaMedica_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_CitaMedica_List]
AS
BEGIN
    SELECT  
			--citaMedica.medic_Id,
   --         mascota.masc_Nombre
			citaMedica.medic_Id,
			mascota.masc_Nombre,
			comportamientos.com_Descripcion,
			citaMedica.medic_FechaConsulta,
			citaMedica.medic_TipoConsulta,
			citaMedica.medic_MotivoConsulta,
			citaMedica.medic_Diagnostico,
			citaMedica.medic_Peso,
			citaMedica.medic_Temperatura,
			citaMedica.medic_FrecuenciaCardiaca,
			citaMedica.medic_FrecuenciaRespiratoria,
			vacunas.vac_Descripcion,
			citaMedica.medic_MedicamentosRecetados,
			citaMedica.medic_Dosificacion,
			citaMedica.medic_ProcedimientosRealizados,
			citaMedica.medic_ResultadosExamenes,
			citaMedica.medic_ProximaCita,
			citaMedica.medic_MotivoProximaCita,
			citaMedica.medic_FechaCrea,
			citaMedica.medic_FechaModifica
    FROM	[Refugio].[tbCitaMedica] AS citaMedica
    INNER JOIN	Refugio.tbMascotas AS mascota
    ON			citaMedica.masc_Id = mascota.masc_Id
	INNER JOIN	Refugio.tbComportamientos AS comportamientos
	ON			citaMedica.com_Id = comportamientos.com_Id
    LEFT JOIN   Refugio.tbVacunas AS vacunas
    ON          citaMedica.vac_Id = vacunas.vac_Id
    WHERE		citaMedica.medic_EsEliminado != 1
    ORDER BY    citaMedica.medic_FechaConsulta DESC
END
GO


# [Refugio].[PR_Refugio_CitaMedica_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_CitaMedica_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
-----------------> UPDATE
CREATE PROCEDURE [Refugio].[PR_Refugio_CitaMedica_Update] 
@medic_Id INT,
@masc_Id INT,
@medic_Esterilizacion BIT,
@medic_Comportamiento NVARCHAR(255),
@medic_SaludCuidado NVARCHAR(255),
@medic_InformacionAdicional NVARCHAR(255),
@medic_UsuarioModifica INT 
AS BEGIN
UPDATE [Refugio].[tbHistorialMedico]
SET masc_Id = @masc_Id,
  medic_Esterilizacion = @medic_Esterilizacion,
  medic_SaludCuidado = @medic_SaludCuidado,
  medic_InformacionAdicional = @medic_InformacionAdicional,
  medic_UsuarioModifica = @medic_UsuarioModifica,
  medic_FechaModifica = Getdate()
WHERE medic_Id = @medic_Id
END 
GO


# [Refugio].[PR_Refugio_Comportamiento_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Comportamiento_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
------------------> tbProcedencias
create PROCEDURE [Refugio].[PR_Refugio_Comportamiento_List]
AS
BEGIN
    SELECT	com_Id,
			com_Descripcion
    FROM [Refugio].[tbComportamientos] AS comportamientos
    WHERE comportamientos.com_EsEliminado != 1
END
GO


# [Refugio].[PR_Refugio_Donaciones_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Donaciones_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: PR_Refugio_Donaciones_Delete
-- Descripción: Elimina lógicamente una donación
-- =============================================
CREATE   PROCEDURE [Refugio].[PR_Refugio_Donaciones_Delete]
    @dona_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        BEGIN TRANSACTION;
        
        -- Validar que la donación existe
        IF NOT EXISTS (SELECT 1 FROM [Refugio].[tbDonaciones] WHERE dona_Id = @dona_Id AND dona_EsEliminado = 0)
        BEGIN
            RAISERROR('La donación especificada no existe o ya está eliminada.', 16, 1);
            RETURN;
        END
        
        -- Eliminar lógicamente la donación
        UPDATE [Refugio].[tbDonaciones]
        SET 
            dona_EsEliminado = 1,
            dona_FechaModifica = GETDATE()
        WHERE dona_Id = @dona_Id;
        
        COMMIT TRANSACTION;
        
        SELECT 0 AS Resultado, 'Donación eliminada correctamente.' AS Mensaje;
        
    END TRY
    BEGIN CATCH
        ROLLBACK TRANSACTION;
        
        SELECT 1 AS Resultado, ERROR_MESSAGE() AS Mensaje;
    END CATCH
END
GO


# [Refugio].[PR_Refugio_Donaciones_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Donaciones_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: PR_Refugio_Donaciones_Detail
-- Descripción: Obtiene el detalle completo de una donación
-- =============================================
CREATE   PROCEDURE [Refugio].[PR_Refugio_Donaciones_Detail]
    @dona_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        d.dona_Id,
        d.dona_TipoDonacion,
        d.dona_NombreDonante,
        d.dona_TelefonoDonante,
        d.dona_EmailDonante,
        d.dona_MontoMonetario,
        d.dona_DescripcionArticulos,
        d.dona_ValorEstimado,
        d.dona_FechaDonacion,
        d.dona_Estado,
        d.dona_Observaciones,
        r.refg_Nombre,
        r.refg_Ubicacion,
        r.refg_Telefono,
        d.dona_FechaCrea,
        uc.Usu_Nombre AS dona_NombreUsuarioCrea,
        d.dona_FechaModifica,
        um.Usu_Nombre AS dona_NombreUsuarioModifica
    FROM [Refugio].[tbDonaciones] d
    INNER JOIN [Refugio].[tbRefugios] r ON d.refg_Id = r.refg_Id
    INNER JOIN [Seguridad].[tbUsuarios] uc ON d.dona_UsuarioCrea = uc.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] um ON d.dona_UsuarioModifica = um.usu_Id
    WHERE d.dona_Id = @dona_Id 
      AND d.dona_EsEliminado = 0;
END

GO


# [Refugio].[PR_Refugio_Donaciones_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Donaciones_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: PR_Refugio_Donaciones_Find
-- Descripción: Busca una donación por ID para edición
-- =============================================
CREATE   PROCEDURE [Refugio].[PR_Refugio_Donaciones_Find]
    @dona_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        d.dona_Id,
        d.dona_TipoDonacion,
        d.dona_NombreDonante,
        d.dona_TelefonoDonante,
        d.dona_EmailDonante,
        d.dona_MontoMonetario,
        d.dona_DescripcionArticulos,
        d.dona_ValorEstimado,
        d.dona_FechaDonacion,
        d.dona_Estado,
        d.dona_Observaciones,
        d.refg_Id,
        d.dona_UsuarioCrea,
        d.dona_FechaCrea,
        d.dona_UsuarioModifica,
        d.dona_FechaModifica
    FROM [Refugio].[tbDonaciones] d
    WHERE d.dona_Id = @dona_Id 
      AND d.dona_EsEliminado = 0;
END

GO


# [Refugio].[PR_Refugio_Donaciones_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Donaciones_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: PR_Refugio_Donaciones_Insert
-- Descripción: Inserta una nueva donación
-- =============================================
CREATE   PROCEDURE [Refugio].[PR_Refugio_Donaciones_Insert]
    @dona_TipoDonacion NVARCHAR(50),
    @dona_NombreDonante NVARCHAR(100),
    @dona_TelefonoDonante NVARCHAR(15) = NULL,
    @dona_EmailDonante NVARCHAR(100) = NULL,
    @dona_MontoMonetario DECIMAL(18,2) = NULL,
    @dona_DescripcionArticulos NVARCHAR(500) = NULL,
    @dona_ValorEstimado DECIMAL(18,2) = NULL,
    @dona_FechaDonacion DATE,
    @dona_Estado NVARCHAR(30),
    @dona_Observaciones NVARCHAR(1000) = NULL,
    @refg_Id INT,
    @dona_UsuarioCrea INT
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        BEGIN TRANSACTION;
        
        -- Validar que el refugio existe
        IF NOT EXISTS (SELECT 1 FROM [Refugio].[tbRefugios] WHERE refg_Id = @refg_Id AND refg_EsEliminado = 0)
        BEGIN
            RAISERROR('El refugio especificado no existe o está eliminado.', 16, 1);
            RETURN;
        END
        
        -- Validar que el usuario existe
        IF NOT EXISTS (SELECT 1 FROM [Seguridad].[tbUsuarios] WHERE usu_Id = @dona_UsuarioCrea)
        BEGIN
            RAISERROR('El usuario especificado no existe.', 16, 1);
            RETURN;
        END
        
        -- Insertar la donación
        INSERT INTO [Refugio].[tbDonaciones] (
            dona_TipoDonacion,
            dona_NombreDonante,
            dona_TelefonoDonante,
            dona_EmailDonante,
            dona_MontoMonetario,
            dona_DescripcionArticulos,
            dona_ValorEstimado,
            dona_FechaDonacion,
            dona_Estado,
            dona_Observaciones,
            refg_Id,
            dona_UsuarioCrea,
            dona_FechaCrea,
            dona_EsEliminado
        )
        VALUES (
            @dona_TipoDonacion,
            @dona_NombreDonante,
            @dona_TelefonoDonante,
            @dona_EmailDonante,
            @dona_MontoMonetario,
            @dona_DescripcionArticulos,
            @dona_ValorEstimado,
            @dona_FechaDonacion,
            @dona_Estado,
            @dona_Observaciones,
            @refg_Id,
            @dona_UsuarioCrea,
            GETDATE(),
            0
        );
        
        COMMIT TRANSACTION;
        
        SELECT 0 AS Resultado, 'Donación insertada correctamente.' AS Mensaje;
        
    END TRY
    BEGIN CATCH
        ROLLBACK TRANSACTION;
        
        SELECT 1 AS Resultado, ERROR_MESSAGE() AS Mensaje;
    END CATCH
END

GO


# [Refugio].[PR_Refugio_Donaciones_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Donaciones_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: PR_Refugio_Donaciones_List
-- Descripción: Obtiene la lista de todas las donaciones
-- =============================================
CREATE   PROCEDURE [Refugio].[PR_Refugio_Donaciones_List]
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        d.dona_Id,
        d.dona_TipoDonacion,
        d.dona_NombreDonante,
        d.dona_TelefonoDonante,
        d.dona_EmailDonante,
        d.dona_MontoMonetario,
        d.dona_DescripcionArticulos,
        d.dona_ValorEstimado,
        d.dona_FechaDonacion,
        d.dona_Estado,
        d.dona_Observaciones,
        r.refg_Nombre,
        d.dona_FechaCrea,
        uc.Usu_Nombre AS dona_NombreUsuarioCrea
    FROM [Refugio].[tbDonaciones] d
    INNER JOIN [Refugio].[tbRefugios] r ON d.refg_Id = r.refg_Id
    INNER JOIN [Seguridad].[tbUsuarios] uc ON d.dona_UsuarioCrea = uc.usu_Id
    WHERE d.dona_EsEliminado = 0
    ORDER BY d.dona_FechaDonacion DESC, d.dona_FechaCrea DESC;
END

GO


# [Refugio].[PR_Refugio_Donaciones_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Donaciones_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- Procedimiento: PR_Refugio_Donaciones_Update
-- Descripción: Actualiza una donación existente
-- =============================================
CREATE   PROCEDURE [Refugio].[PR_Refugio_Donaciones_Update]
    @dona_Id INT,
    @dona_TipoDonacion NVARCHAR(50),
    @dona_NombreDonante NVARCHAR(100),
    @dona_TelefonoDonante NVARCHAR(15) = NULL,
    @dona_EmailDonante NVARCHAR(100) = NULL,
    @dona_MontoMonetario DECIMAL(18,2) = NULL,
    @dona_DescripcionArticulos NVARCHAR(500) = NULL,
    @dona_ValorEstimado DECIMAL(18,2) = NULL,
    @dona_FechaDonacion DATE,
    @dona_Estado NVARCHAR(30),
    @dona_Observaciones NVARCHAR(1000) = NULL,
    @refg_Id INT,
    @dona_UsuarioModifica INT
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        BEGIN TRANSACTION;
        
        -- Validar que la donación existe
        IF NOT EXISTS (SELECT 1 FROM [Refugio].[tbDonaciones] WHERE dona_Id = @dona_Id AND dona_EsEliminado = 0)
        BEGIN
            RAISERROR('La donación especificada no existe o está eliminada.', 16, 1);
            RETURN;
        END
        
        -- Validar que el refugio existe
        IF NOT EXISTS (SELECT 1 FROM [Refugio].[tbRefugios] WHERE refg_Id = @refg_Id AND refg_EsEliminado = 0)
        BEGIN
            RAISERROR('El refugio especificado no existe o está eliminado.', 16, 1);
            RETURN;
        END
        
        -- Validar que el usuario existe
        IF NOT EXISTS (SELECT 1 FROM [Seguridad].[tbUsuarios] WHERE usu_Id = @dona_UsuarioModifica)
        BEGIN
            RAISERROR('El usuario especificado no existe.', 16, 1);
            RETURN;
        END
        
        -- Actualizar la donación
        UPDATE [Refugio].[tbDonaciones]
        SET 
            dona_TipoDonacion = @dona_TipoDonacion,
            dona_NombreDonante = @dona_NombreDonante,
            dona_TelefonoDonante = @dona_TelefonoDonante,
            dona_EmailDonante = @dona_EmailDonante,
            dona_MontoMonetario = @dona_MontoMonetario,
            dona_DescripcionArticulos = @dona_DescripcionArticulos,
            dona_ValorEstimado = @dona_ValorEstimado,
            dona_FechaDonacion = @dona_FechaDonacion,
            dona_Estado = @dona_Estado,
            dona_Observaciones = @dona_Observaciones,
            refg_Id = @refg_Id,
            dona_UsuarioModifica = @dona_UsuarioModifica,
            dona_FechaModifica = GETDATE()
        WHERE dona_Id = @dona_Id;
        
        COMMIT TRANSACTION;
        
        SELECT 0 AS Resultado, 'Donación actualizada correctamente.' AS Mensaje;
        
    END TRY
    BEGIN CATCH
        ROLLBACK TRANSACTION;
        
        SELECT 1 AS Resultado, ERROR_MESSAGE() AS Mensaje;
    END CATCH
END

GO


# [Refugio].[PR_Refugio_Empleados_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Empleados_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Empleados_Detail] 
@emp_Id INT
AS BEGIN
SELECT empleado.emp_Id,
	emp_Codigo,
	persona.per_PrimerNombre,
	persona.per_SegundoNombre,
	persona.per_ApellidoPaterno,
	persona.per_ApellidoMaterno,
	persona.per_Identidad,
	persona.per_FechaNacimiento,
	persona.per_Domicilio,
	persona.per_Telefono,
	persona.per_Correo,
	empleadocargo.cag_Descripcion,
	refugio.refg_Nombre,
	Help.IsActive(emp_EsActivo) AS EsActivo,
	usuarioCrea.usu_Nombre AS UsuarioCreacion,
	persona.per_FechaCrea,
	usuarioModifica.usu_Nombre AS UsuarioModificacion,
	persona.per_FechaModifica
FROM [Refugio].[tbEmpleados] AS empleado
	INNER JOIN [General].[tbPersonas] AS persona ON empleado.per_Id = persona.per_Id
	INNER JOIN [Refugio].[tbRefugios] AS refugio ON empleado.refg_Id = refugio.refg_Id
	INNER JOIN [Refugio].[tbEmpleadosCargos] AS empleadocargo ON empleado.cag_Id = empleadocargo.cag_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		persona.per_UsuarioCrea = usuarioCrea.usu_Id 
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		persona.per_UsuarioModifica = usuarioModifica.usu_Id
WHERE persona.per_EsEliminado != 1 and empleado.emp_Id = @emp_Id
END
GO


# [Refugio].[PR_Refugio_Empleados_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Empleados_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbEmpleados
CREATE PROCEDURE [Refugio].[PR_Refugio_Empleados_Find]
@emp_Id INT
AS
BEGIN
    SELECT	empleado.emp_Id,
			emp_Codigo,
			empleado.per_Id,
			persona.per_PrimerNombre,
			persona.per_SegundoNombre,
			persona.per_ApellidoPaterno,
			persona.per_ApellidoMaterno,
			persona.per_Identidad,
			persona.per_FechaNacimiento,
			persona.per_Domicilio,
			persona.per_Telefono,
			persona.per_Correo,
			empleado.cag_Id,
			empleadocargo.cag_Descripcion,
			empleado.refg_Id,
			refugio.refg_Nombre,
			emp_EsActivo,
			Help.IsActive(emp_EsActivo) AS [esActivo],
			persona.per_UsuarioCrea,
			usuarioCrea.Usu_Nombre AS usuarioCrea,
			persona.per_FechaCrea,
			persona.per_UsuarioModifica,
			usuarioModifica.Usu_Nombre AS usuarioModifica,
			persona.per_FechaModifica	
    FROM		[Refugio].[tbEmpleados] AS empleado
	INNER JOIN	[General].[tbPersonas] AS persona
	ON			empleado.per_Id = persona.per_Id
	INNER JOIN	[Refugio].[tbRefugios] AS refugio
	ON			empleado.refg_Id = refugio.refg_Id
	INNER JOIN	[Refugio].[tbEmpleadosCargos] AS empleadocargo
	ON			empleado.cag_Id = empleadocargo.cag_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		persona.per_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		persona.per_UsuarioModifica = usuarioModifica.usu_Id
    WHERE		persona.per_EsEliminado != 1
	AND 	empleado.emp_Id = @emp_Id
END
GO


# [Refugio].[PR_Refugio_Empleados_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Empleados_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbEmpleados
CREATE PROCEDURE [Refugio].[PR_Refugio_Empleados_Insert]
@emp_Codigo	varchar(7),
@refg_Id	int,
@cag_Id	int,
@emp_EsActivo	bit,
@per_Identidad	varchar(13),
@per_PrimerNombre	nvarchar(50),
@per_SegundoNombre	nvarchar(50),
@per_ApellidoPaterno	nvarchar(50),
@per_ApellidoMaterno	nvarchar(50),
@per_FechaNacimiento	date,
@per_Domicilio	nvarchar(100),
@per_Telefono	varchar(8),
@per_Correo	varchar(150),
@per_UsuarioCrea INT
AS
BEGIN
	--BEGIN TRANSACTION
	--	BEGIN TRY
		--Guardamos datos en personas.
		EXEC	[General].[PR_General_Personas_Insert]
				@per_Identidad,
                @per_PrimerNombre,
                @per_SegundoNombre,
                @per_ApellidoPaterno,
                @per_ApellidoMaterno,
                @per_FechaNacimiento,
                @per_Domicilio,
                @per_Telefono,
                @per_Correo,
				@per_UsuarioCrea
		DECLARE @per_Id INT = IDENT_CURRENT('[General].[tbPersonas]');

		--Guardamos datos en empleados.
		INSERT INTO [Refugio].[tbEmpleados]
		(
			emp_Codigo,
			per_Id,
			refg_Id,
			cag_Id,
			emp_EsActivo
		)
		VALUES
		(
			@emp_Codigo,
			@per_Id,
			@refg_Id,
			@cag_Id,
			@emp_EsActivo
		)
	--	COMMIT TRANSACTION
	--END TRY
	--BEGIN CATCH
	--	SELECT ERROR_MESSAGE();
	--	ROLLBACK TRANSACTION
	--END CATCH
END
GO


# [Refugio].[PR_Refugio_Empleados_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Empleados_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

	CREATE PROCEDURE [Refugio].[PR_Refugio_Empleados_List]
	AS
	BEGIN
		SELECT	emp_Id,
				emp_Codigo,
				Help.ConcatEspace(persona.per_PrimerNombre, persona.per_ApellidoPaterno) AS [emp_Nombres],
				empleadocargo.cag_Descripcion,
				refugio.refg_Nombre,
				Help.IsActive(emp_EsActivo) AS EsActivo
		FROM		[Refugio].[tbEmpleados] AS empleado
		INNER JOIN	[General].[tbPersonas] AS persona
		ON			empleado.per_Id = persona.per_Id
		INNER JOIN	[Refugio].[tbRefugios] AS refugio
		ON			empleado.refg_Id = refugio.refg_Id
		INNER JOIN	[Refugio].[tbEmpleadosCargos] AS empleadocargo
		ON			empleado.cag_Id = empleadocargo.cag_Id
		WHERE		persona.per_EsEliminado != 1
	END
GO


# [Refugio].[PR_Refugio_Empleados_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Empleados_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Empleados_Update]
@emp_Id	int,
@emp_Codigo	varchar(7),
@per_Id	int,
@refg_Id	int,
@cag_Id	int,
@emp_EsActivo	bit,
@per_Identidad	varchar(13),
@per_PrimerNombre	nvarchar(50),
@per_SegundoNombre	nvarchar(50),
@per_ApellidoPaterno	nvarchar(50),
@per_ApellidoMaterno	nvarchar(50),
@per_FechaNacimiento	date,
@per_Domicilio	nvarchar(100),
@per_Telefono	varchar(8),
@per_Correo	varchar(150),
@per_UsuarioModifica INT
AS
BEGIN
	BEGIN TRANSACTION
		BEGIN TRY
		--Guardamos datos en personas.
		EXEC	[General].[PR_General_Personas_Update]
				@per_Id,
				@per_Identidad,
                @per_PrimerNombre,
                @per_SegundoNombre,
                @per_ApellidoPaterno,
                @per_ApellidoMaterno,
                @per_FechaNacimiento,
                @per_Domicilio,
                @per_Telefono,
                @per_Correo,
				@per_UsuarioModifica

		--Guardamos datos en empleados.
		UPDATE [Refugio].[tbEmpleados]
		SET		emp_Codigo = @emp_Codigo,
				refg_Id = @refg_Id,
				cag_Id = @cag_Id,
				emp_EsActivo = @emp_EsActivo
		WHERE	emp_Id = @emp_Id
		COMMIT TRANSACTION
	END TRY
	BEGIN CATCH
		SELECT ERROR_MESSAGE();
		ROLLBACK TRANSACTION
	END CATCH
END
GO


# [Refugio].[PR_Refugio_EmpleadosCargos_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_EmpleadosCargos_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


-----------------> DELETE

CREATE PROCEDURE [Refugio].[PR_Refugio_EmpleadosCargos_Delete] 
@cag_Id INT
AS
  BEGIN
          UPDATE [Refugio].[tbEmpleadosCargos]
          SET    cag_EsEliminado	= 1
          WHERE  cag_Id			= @cag_Id
  END
GO


# [Refugio].[PR_Refugio_EmpleadosCargos_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_EmpleadosCargos_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_EmpleadosCargos_Detail] 
@cag_Id INT
AS BEGIN
SELECT cag_Id,
	cag_Descripcion,
	cag_Salario,
	Help.IsActive(cag_EsActivo) AS EsActivo,
	usuarioCrea.usu_Nombre AS UsuarioCreacion,
	cag_FechaCrea,
	usuarioModifica.usu_Nombre AS UsuarioModificacion,
	cag_FechaModifica
	FROM [Refugio].[tbEmpleadosCargos] AS empleadocargo
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		empleadocargo.cag_UsuarioCrea = usuarioCrea.usu_Id 
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		empleadocargo.cag_UsuarioModifica = usuarioModifica.usu_Id
WHERE empleadocargo.cag_EsEliminado != 1
AND		cag_Id = @cag_Id
END
GO


# [Refugio].[PR_Refugio_EmpleadosCargos_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_EmpleadosCargos_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_EmpleadosCargos_Dropdown]
AS
BEGIN
	SELECT	cag_Id,
			cag_Descripcion
    FROM	[Refugio].[tbEmpleadosCargos] AS empleadoscargos
	WHERE empleadoscargos.cag_EsEliminado != 1
END
GO


# [Refugio].[PR_Refugio_EmpleadosCargos_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_EmpleadosCargos_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


------------------> tbEmpleadosCargos
CREATE PROCEDURE [Refugio].[PR_Refugio_EmpleadosCargos_Find]
@cag_Id INT
AS
BEGIN
    SELECT	cag_Id,
			cag_Descripcion,
			cag_Salario,
			cag_EsActivo,
			cag_UsuarioCrea,
			usuarioCrea.Usu_Nombre AS usuarioCrea,
			cag_FechaCrea,
			cag_UsuarioModifica,
			usuarioModifica.Usu_Nombre AS usuarioModifica,
			cag_FechaModifica
    FROM [Refugio].[tbEmpleadosCargos] AS empleadocargo
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		empleadocargo.cag_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		empleadocargo.cag_UsuarioModifica = usuarioModifica.usu_Id
    WHERE empleadocargo.cag_EsEliminado != 1
	AND 	cag_Id = @cag_Id
END
GO


# [Refugio].[PR_Refugio_EmpleadosCargos_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_EmpleadosCargos_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_EmpleadosCargos_Insert]
    @cag_Descripcion  VARCHAR(150),
    @cag_Salario      DECIMAL(10,2),   
    @cag_EsActivo     BIT,            
    @cag_UsuarioCrea  INT
AS
BEGIN
    INSERT INTO [Refugio].[tbEmpleadosCargos]
    (cag_Descripcion, cag_Salario, cag_EsActivo, cag_EsEliminado, cag_UsuarioCrea, cag_FechaCrea)
    VALUES
    (@cag_Descripcion, @cag_Salario, @cag_EsActivo, 0, @cag_UsuarioCrea, GETDATE())
END
GO


# [Refugio].[PR_Refugio_EmpleadosCargos_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_EmpleadosCargos_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_EmpleadosCargos_List]
AS
BEGIN
    SELECT cag_Id,
           cag_Descripcion,
           cag_Salario,
           Help.IsActive(cag_EsActivo) AS EsActivo
    FROM [Refugio].[tbEmpleadosCargos] AS empleadocargo
    WHERE empleadocargo.cag_EsEliminado != 1
END

GO


# [Refugio].[PR_Refugio_EmpleadosCargos_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_EmpleadosCargos_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_EmpleadosCargos_Update]
    @cag_Id              INT,
    @cag_Descripcion     VARCHAR(150),
    @cag_Salario         DECIMAL(10, 2),
    @cag_EsActivo        BIT,
    @cag_UsuarioModifica INT
AS BEGIN
    UPDATE [Refugio].[tbEmpleadosCargos]
    SET cag_Descripcion     = @cag_Descripcion,
        cag_Salario         = @cag_Salario,
        cag_EsActivo        = @cag_EsActivo,
        cag_UsuarioModifica = @cag_UsuarioModifica,
        cag_FechaModifica   = GETDATE()
    WHERE cag_Id = @cag_Id
END

GO


# [Refugio].[PR_Refugio_Eventos_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Eventos_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Eventos_Detail]
@eve_Id int
AS
BEGIN
	SELECT	eve_Id,
			eve_Descripcion,
			eventos.refg_Id,
			refugios.refg_Nombre,
			eve_HoraInicio,
			eve_HoraFinal,
			eve_Fecha,
			eve_UsuarioCrea,
			eve_FechaCrea,
			eve_UsuarioModifica,
			eve_FechaModifica
	FROM	[Refugio].[tbEventos] as eventos
	INNER JOIN [Refugio].[tbRefugios] AS refugios
	ON		eventos.refg_Id = refugios.refg_Id
	WHERE	eve_EsEliminado != 1
	AND		eve_Id = @eve_Id
END

GO


# [Refugio].[PR_Refugio_Eventos_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Eventos_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Eventos_Find]
@eve_Id int
AS
BEGIN
	SELECT	eve_Id,
			eve_Descripcion,
			eventos.refg_Id,
			refugios.refg_Nombre,
			eve_HoraInicio,
			eve_HoraFinal,
			eve_Fecha
	FROM	[Refugio].[tbEventos] as eventos
	INNER JOIN [Refugio].[tbRefugios] AS refugios
	ON		eventos.refg_Id = refugios.refg_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		eventos.eve_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		eventos.eve_UsuarioModifica = usuarioModifica.usu_Id
	WHERE	eve_EsEliminado != 1
	AND		eve_Id = @eve_Id
END

GO


# [Refugio].[PR_Refugio_Eventos_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Eventos_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Eventos_Insert]
@eve_Descripcion	nvarchar(50),
@refg_Id	int,
@eve_HoraInicio	time(2),
@eve_HoraFinal	time(2),
@eve_Fecha	date,
@eve_UsuarioCrea	int
AS
BEGIN
		INSERT INTO [Refugio].[tbEventos]
        (
            eve_Descripcion,
            refg_Id,
            eve_HoraInicio,
            eve_HoraFinal,
            eve_Fecha,
            eve_UsuarioCrea,
            eve_FechaCrea
        )
		VALUES
        (
            @eve_Descripcion,
            @refg_Id,
            @eve_HoraInicio,
            @eve_HoraFinal,
            @eve_Fecha,
            @eve_UsuarioCrea,
            GETDATE()
        )
END
GO


# [Refugio].[PR_Refugio_Eventos_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Eventos_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Eventos_List]
AS
BEGIN
	SELECT	eve_Id,
			eve_Descripcion,
			eventos.refg_Id,
			refugios.refg_Nombre,
			eve_Fecha
	FROM	[Refugio].[tbEventos] as eventos
	INNER JOIN [Refugio].[tbRefugios] AS refugios
	ON		eventos.refg_Id = refugios.refg_Id
	WHERE	eve_EsEliminado != 1
END

GO


# [Refugio].[PR_Refugio_Eventos_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Eventos_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Eventos_Update]
@eve_Id	int,
@eve_Descripcion	nvarchar(50),
@refg_Id	int,
@eve_HoraInicio	time(2),
@eve_HoraFinal	time(2),
@eve_Fecha	date,
@eve_UsuarioModifica	int
AS
BEGIN
		UPDATE	[Refugio].[tbEventos]
        SET eve_Descripcion = @eve_Descripcion,
            refg_Id = @refg_Id,
            eve_HoraInicio = @eve_HoraInicio,
            eve_HoraFinal = @eve_HoraFinal,
            eve_Fecha = @eve_Fecha,
            eve_UsuarioModifica = @eve_UsuarioModifica,
            eve_FechaModifica = GETDATE()
        WHERE eve_Id = @eve_Id
END

GO


# [Refugio].[PR_Refugio_FichaAdopcion_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_FichaAdopcion_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-----------------> DELETE

CREATE PROCEDURE [Refugio].[PR_Refugio_FichaAdopcion_Delete] 
	@ficha_Id INT
AS
  BEGIN
          UPDATE [Refugio].[tbFichaAdopcion]
          SET    ficha_EsEliminado	= 1
          WHERE  ficha_Id		= @ficha_Id
  END
GO


# [Refugio].[PR_Refugio_FichasMedicas_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_FichasMedicas_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


-----------------> DELETE

CREATE PROCEDURE [Refugio].[PR_Refugio_FichasMedicas_Delete] 
	@medic_Id INT
AS
  BEGIN
          UPDATE [Refugio].[tbFichasMedicas]
          SET    medic_EsEliminado	= 1
          WHERE  medic_Id		= @medic_Id
  END
GO


# [Refugio].[PR_Refugio_HistorialMedico_VacunasByMascotas]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_HistorialMedico_VacunasByMascotas]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
create PROCEDURE [Refugio].[PR_Refugio_HistorialMedico_VacunasByMascotas]
@masc_Id INT
AS
BEGIN
    SELECT  vacuna.vac_Descripcion AS [vacunas]
	FROM	[Refugio].[tbVacunas] AS vacuna
	INNER JOIN [Refugio].[tbHistorialMedico_tbVacunas] AS hisvac
	ON		vacuna.vac_Id = hisvac.vac_Id
	INNER JOIN [Refugio].[tbHistorialMedico] AS historialMedico
	ON		hisvac.medic_Id = historialMedico.medic_Id
	WHERE	historialMedico.masc_Id = @masc_Id
	AND		vacuna.vac_EsEliminado != @masc_Id
END
GO


# [Refugio].[PR_Refugio_Mascotas_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Mascotas_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


-----------------> DELETE

CREATE PROCEDURE [Refugio].[PR_Refugio_Mascotas_Delete] 
@masc_Id INT
AS
  BEGIN
          UPDATE [Refugio].[tbMascotas]
          SET    masc_EsEliminado	= 1
          WHERE  masc_Id		= @masc_Id
  END
GO


# [Refugio].[PR_Refugio_Mascotas_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Mascotas_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbMascotas *
CREATE PROCEDURE [Refugio].[PR_Refugio_Mascotas_Detail]
@masc_Id INT
AS
BEGIN
	SELECT mascotas.masc_Id,
		   mascotas.masc_Imagen,
		   mascotas.masc_Nombre,
		   raza.raza_Descripcion,
		   mascotas.masc_Edad,
		   mascotas.masc_Sexo,
		   mascotas.masc_Peso,
		   mascotas.masc_Color,
		   mascotas.masc_Historia,
		   albergue.refg_Nombre,
		   procedencia.proc_Descripcion,
		   mascotas.masc_EsAdoptado,
		   mascotas.masc_EsReservado,
		   usuarioCrea.usu_Nombre AS masc_NombreUsuarioCrea,
		   mascotas.masc_FechaCrea,
		   usuarioModifica.usu_Nombre AS masc_NombreUsuarioModifica,
		   mascotas.masc_FechaModifica
    FROM		[Refugio].[tbMascotas] AS mascotas
    INNER JOIN  [Refugio].[tbRazas] AS raza
    ON			mascotas.raza_Id = raza.raza_Id
    INNER JOIN	[Refugio].[tbRefugios] AS albergue
    ON			mascotas.refg_Id = albergue.refg_Id
    INNER JOIN	[Refugio].[tbProcedencias] AS procedencia
    ON			mascotas.proc_Id = procedencia.proc_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		mascotas.masc_UsuarioCrea = usuarioCrea.usu_Id 
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		mascotas.masc_UsuarioModifica = usuarioModifica.usu_Id
    WHERE		mascotas.masc_EsEliminado != 1 and mascotas.masc_Id = @masc_Id
END
GO


# [Refugio].[PR_Refugio_Mascotas_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Mascotas_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Mascotas_Dropdown]
AS 
BEGIN
	SELECT mascota.masc_Id, mascota.masc_Nombre, mascota.masc_Imagen
	FROM Refugio.tbMascotas AS mascota
END
GO


# [Refugio].[PR_Refugio_Mascotas_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Mascotas_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Mascotas_Find]
@masc_Id INT
AS
BEGIN
	SELECT mascotas.masc_Id,
		   mascotas.masc_Imagen,
		   mascotas.masc_Nombre,
		   mascotas.raza_Id,
		   raza.raza_Descripcion,
		   mascotas.masc_Edad,
		   mascotas.masc_Sexo,
		   mascotas.masc_Peso,
		   mascotas.masc_Color,
		   mascotas.masc_Historia,
		   mascotas.refg_Id,
		   albergue.refg_Nombre,
		   mascotas.proc_Id,
		   procedencia.proc_Descripcion,
		   mascotas.masc_EsAdoptado,
		   mascotas.masc_EsReservado,
		   mascotas.masc_UsuarioCrea,
		   usuarioCrea.Usu_Nombre AS usuarioCrea,
		   mascotas.masc_FechaCrea,
		   mascotas.masc_UsuarioModifica,
		   usuarioModifica.Usu_Nombre AS usuarioModifica,
		   mascotas.masc_FechaModifica
    FROM		[Refugio].[tbMascotas] AS mascotas
    INNER JOIN  [Refugio].[tbRazas] AS raza
    ON			mascotas.raza_Id = raza.raza_Id
    INNER JOIN	[Refugio].[tbRefugios] AS albergue
    ON			mascotas.refg_Id = albergue.refg_Id
    INNER JOIN	[Refugio].[tbProcedencias] AS procedencia
    ON			mascotas.proc_Id = procedencia.proc_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		mascotas.masc_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		mascotas.masc_UsuarioModifica = usuarioModifica.usu_Id
    WHERE		mascotas.masc_EsEliminado != 1
	AND 		masc_Id = @masc_Id
END
GO


# [Refugio].[PR_Refugio_Mascotas_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Mascotas_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Mascotas_Insert] 
    @masc_Imagen image,
    @masc_Nombre NVARCHAR(50),
    @raza_Id INT,
    @masc_Edad INT,
    @masc_Sexo CHAR(1),
    @masc_Peso DECIMAL(18, 0),
    @masc_Color NVARCHAR(50),
    @masc_Historia NVARCHAR(500),
    @refg_Id INT,
    @proc_Id INT,
    @masc_UsuarioCrea INT
AS BEGIN
INSERT INTO [Refugio].[tbMascotas] (
        masc_Imagen,
        masc_Nombre,
        raza_Id,
        masc_Edad,
        masc_Sexo,
        masc_Peso,
        masc_Color,
        masc_Historia,
        refg_Id,
        proc_Id,
        masc_UsuarioCrea,
        masc_FechaCrea
    )
VALUES
    (
        @masc_Imagen,
        @masc_Nombre,
        @raza_Id,
        @masc_Edad,
        @masc_Sexo,
        @masc_Peso,
        @masc_Color,
        @masc_Historia,
        @refg_Id,
        @proc_Id,
        @masc_UsuarioCrea,
        Getdate()
    )
END
GO


# [Refugio].[PR_Refugio_Mascotas_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Mascotas_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Mascotas_List]
AS
BEGIN
	SELECT      mascotas.masc_Id,
				ROW_NUMBER() OVER(ORDER BY mascotas.masc_FechaModifica DESC, mascotas.masc_FechaCrea DESC) AS masc_Fila,
				mascotas.masc_Imagen,
		        mascotas.masc_Nombre,
		        raza.raza_Descripcion,
		        mascotas.masc_Edad,
		        mascotas.masc_Sexo,
		        mascotas.masc_EsAdoptado
    FROM		[Refugio].[tbMascotas] AS mascotas
    INNER JOIN  [Refugio].[tbRazas] AS raza
    ON			mascotas.raza_Id = raza.raza_Id
    INNER JOIN	[Refugio].[tbRefugios] AS albergue
    ON			mascotas.refg_Id = albergue.refg_Id
    INNER JOIN	[Refugio].[tbProcedencias] AS procedencia
    ON			mascotas.proc_Id = procedencia.proc_Id
    WHERE		mascotas.masc_EsEliminado != 1 and mascotas.masc_EsAdoptado != 1
END
GO


# [Refugio].[PR_Refugio_Mascotas_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Mascotas_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-----------------> UPDATE
CREATE PROCEDURE [Refugio].[PR_Refugio_Mascotas_Update] 
@masc_Id INT,
@masc_Imagen image,
@masc_Nombre NVARCHAR(50),
@raza_Id INT,
@masc_Edad INT,
@masc_Sexo CHAR(1),
@masc_Peso DECIMAL(18, 0),
@masc_Color NVARCHAR(50),
@masc_Historia NVARCHAR(500),
@refg_Id INT,
@proc_Id INT,
@masc_EsAdoptado BIT,
@masc_EsReservado BIT,
@masc_UsuarioModifica INT 
AS BEGIN
UPDATE [Refugio].[tbMascotas]
SET masc_Imagen = @masc_Imagen,
  masc_Nombre = @masc_Nombre,
  raza_Id = @raza_Id,
  masc_Edad = @masc_Edad,
  masc_Sexo = @masc_Sexo,
  masc_Peso = @masc_Peso,
  masc_Color = @masc_Color,
  masc_Historia = @masc_Historia,
  refg_Id = @refg_Id,
  proc_Id = @proc_Id,
  masc_EsAdoptado = @masc_EsAdoptado,
  masc_EsReservado = @masc_EsReservado,
  masc_UsuarioModifica = @masc_UsuarioModifica,
  masc_FechaModifica = Getdate()
WHERE masc_Id = @masc_Id
END 
GO


# [Refugio].[PR_Refugio_Procedencias_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Procedencias_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


-----------------> DELETE

CREATE PROCEDURE [Refugio].[PR_Refugio_Procedencias_Delete] 
@proc_Id INT
AS
  BEGIN
          UPDATE [Refugio].[tbProcedencias]
          SET    proc_EsEliminado	= 1
          WHERE  proc_Id = @proc_Id
  END
GO


# [Refugio].[PR_Refugio_Procedencias_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Procedencias_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbProcedencias
CREATE PROCEDURE [Refugio].[PR_Refugio_Procedencias_Detail]
    @proc_Id INT 
AS
BEGIN
    SELECT  proc_Id,
            proc_Descripcion,
            usuarioCrea.usu_Nombre AS proc_NombreUsuarioCrea,
            proc_FechaCrea,
            usuarioModifica.usu_Nombre AS proc_NombreUsuarioModifica,
            proc_FechaModifica
    FROM [Refugio].[tbProcedencias] AS procedencias
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON procedencias.proc_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON procedencias.proc_UsuarioModifica = usuarioModifica.usu_Id
    WHERE proc_EsEliminado != 1
      AND proc_Id = @proc_Id 
END
GO


# [Refugio].[PR_Refugio_Procedencias_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Procedencias_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Procedencias_Dropdown]
AS
BEGIN
	SELECT	procedencia.proc_Id,
			procedencia.proc_Descripcion
    FROM	[Refugio].[tbProcedencias] AS procedencia
	WHERE procedencia.proc_EsEliminado != 1
END
GO


# [Refugio].[PR_Refugio_Procedencias_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Procedencias_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Procedencias_Find]
@proc_Id INT
AS
BEGIN
    SELECT proc_Id, proc_Descripcion, proc_EsActivo,
           proc_UsuarioCrea,
           usuarioCrea.Usu_Nombre AS usuarioCrea,
           proc_FechaCrea,
           proc_UsuarioModifica,
           usuarioModifica.Usu_Nombre AS usuarioModifica,
           proc_FechaModifica
    FROM [Refugio].[tbProcedencias] AS p
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON p.proc_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON p.proc_UsuarioModifica = usuarioModifica.usu_Id
    WHERE p.proc_EsEliminado != 1 AND p.proc_Id = @proc_Id
END
GO


# [Refugio].[PR_Refugio_Procedencias_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Procedencias_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Procedencias_Insert]
    @proc_Descripcion NVARCHAR(50),
    @proc_EsActivo BIT = 1,
    @proc_UsuarioCrea INT
AS BEGIN
    INSERT INTO [Refugio].[tbProcedencias] (proc_Descripcion, proc_EsActivo, proc_UsuarioCrea, proc_FechaCrea)
    VALUES (@proc_Descripcion, @proc_EsActivo, @proc_UsuarioCrea, GETDATE())
END
GO


# [Refugio].[PR_Refugio_Procedencias_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Procedencias_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Procedencias_List]
AS
BEGIN
    SELECT proc_Id, proc_Descripcion,
           CASE WHEN proc_EsActivo = 1 THEN 'Activo' ELSE 'Inactivo' END AS proc_EsActivo
    FROM [Refugio].[tbProcedencias] AS procedencias
    WHERE proc_EsEliminado != 1
END

GO


# [Refugio].[PR_Refugio_Procedencias_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Procedencias_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Procedencias_Update]
    @proc_Id INT,
    @proc_Descripcion NVARCHAR(50),
    @proc_EsActivo BIT = 1,
    @proc_UsuarioModifica INT
AS BEGIN
    UPDATE [Refugio].[tbProcedencias]
    SET proc_Descripcion = @proc_Descripcion,
        proc_EsActivo = @proc_EsActivo,
        proc_UsuarioModifica = @proc_UsuarioModifica,
        proc_FechaModifica = GETDATE()
    WHERE proc_Id = @proc_Id
END
GO


# [Refugio].[PR_Refugio_Raza_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Raza_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Raza_Dropdown]
AS
BEGIN
	SELECT	raza.raza_Id,
			raza.raza_Descripcion
    FROM	[Refugio].[tbRazas] AS raza
	WHERE raza.raza_EsEliminado != 1
END
GO


# [Refugio].[PR_Refugio_Razas_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Razas_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO



-----------------> DELETE

CREATE PROCEDURE [Refugio].[PR_Refugio_Razas_Delete] 
@raza_Id INT
AS
  BEGIN
          UPDATE [Refugio].[tbRazas]
          SET    raza_EsEliminado	= 1
          WHERE  raza_Id		= @raza_Id
  END
GO


# [Refugio].[PR_Refugio_Razas_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Razas_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Razas_Detail]
    @raza_Id INT  
AS
BEGIN
    SELECT  raza_Id,
            raza_Descripcion,
            raza_Tamano,
            raza_TipoAnimal,
            raza_TipoPelaje,
            raza_ImagenUrl,
            usuarioCrea.usu_Nombre AS UsuarioCreacion,
            raza_FechaCrea,
            usuarioModifica.usu_Nombre AS UsuarioModificacion,
            raza_FechaModifica
    FROM [Refugio].[tbRazas] AS razas
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON razas.raza_UsuarioCrea = usuarioCrea.usu_Id 
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON razas.raza_UsuarioModifica = usuarioModifica.usu_Id
    WHERE raza_EsEliminado != 1
      AND raza_Id = @raza_Id  
END
GO


# [Refugio].[PR_Refugio_Razas_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Razas_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Razas_Find]
@raza_Id INT
AS
BEGIN
    SELECT raza_Id,
           raza_Descripcion,
           raza_Tamano,
           raza_TipoAnimal,
           raza_TipoPelaje,
           raza_ImagenUrl,
           raza_EsActivo,
           raza_UsuarioCrea,
           usuarioCrea.Usu_Nombre AS usuarioCrea,
           raza_FechaCrea,
           raza_UsuarioModifica,
           usuarioModifica.Usu_Nombre AS usuarioModifica,
           raza_FechaModifica
    FROM [Refugio].[tbRazas] AS razas
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea ON razas.raza_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica ON razas.raza_UsuarioModifica = usuarioModifica.usu_Id
    WHERE raza_EsEliminado != 1 AND raza_Id = @raza_Id
END;
SELECT 'Razas_Find OK' AS r
GO


# [Refugio].[PR_Refugio_Razas_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Razas_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Razas_Insert]
    @raza_Descripcion   VARCHAR(50),
    @raza_Tamano        VARCHAR(20)  = NULL,
    @raza_TipoAnimal    VARCHAR(50)  = NULL,
    @raza_TipoPelaje    VARCHAR(30)  = NULL,
    @raza_ImagenUrl     VARCHAR(500) = NULL,
    @raza_EsActivo      BIT = 1,
    @raza_UsuarioCrea   INT
AS
BEGIN
    INSERT INTO [Refugio].[tbRazas] (raza_Descripcion,raza_Tamano,raza_TipoAnimal,raza_TipoPelaje,raza_ImagenUrl,raza_EsActivo,raza_UsuarioCrea,raza_FechaCrea)
    VALUES (@raza_Descripcion,@raza_Tamano,@raza_TipoAnimal,@raza_TipoPelaje,@raza_ImagenUrl,@raza_EsActivo,@raza_UsuarioCrea,GETDATE())
END;
SELECT 'Razas_Insert OK' AS r
GO


# [Refugio].[PR_Refugio_Razas_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Razas_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Razas_List]
AS
BEGIN
    SELECT ROW_NUMBER() OVER(ORDER BY raza_Id ASC) AS Fila,
           raza_Id,
           raza_Descripcion,
           raza_Tamano,
           raza_TipoAnimal,
           raza_TipoPelaje,
           raza_ImagenUrl,
           CASE WHEN raza_EsActivo = 1 THEN 'Activo' ELSE 'Inactivo' END AS raza_EsActivo
    FROM [Refugio].[tbRazas]
    WHERE raza_EsEliminado != 1
END

GO


# [Refugio].[PR_Refugio_Razas_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Razas_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Razas_Update]
    @raza_Id              INT,
    @raza_Descripcion     VARCHAR(50),
    @raza_Tamano          VARCHAR(20)  = NULL,
    @raza_TipoAnimal      VARCHAR(50)  = NULL,
    @raza_TipoPelaje      VARCHAR(30)  = NULL,
    @raza_ImagenUrl       VARCHAR(500) = NULL,
    @raza_EsActivo        BIT = 1,
    @raza_UsuarioModifica INT
AS
BEGIN
    UPDATE [Refugio].[tbRazas]
    SET raza_Descripcion     = @raza_Descripcion,
        raza_Tamano          = @raza_Tamano,
        raza_TipoAnimal      = @raza_TipoAnimal,
        raza_TipoPelaje      = @raza_TipoPelaje,
        raza_ImagenUrl       = @raza_ImagenUrl,
        raza_EsActivo        = @raza_EsActivo,
        raza_UsuarioModifica = @raza_UsuarioModifica,
        raza_FechaModifica   = GETDATE()
    WHERE raza_Id = @raza_Id
END;
SELECT 'Razas_Update OK' AS r
GO


# [Refugio].[PR_Refugio_Refugio_Dropdown]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Refugio_Dropdown]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Refugio_Dropdown]
AS
BEGIN
	SELECT refugio.refg_Id,
		   refugio.refg_Nombre
    FROM		[Refugio].[tbRefugios] AS refugio
	WHERE refugio.refg_EsEliminado != 1
END
GO


# [Refugio].[PR_Refugio_Refugios_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Refugios_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Refugios_Delete] 
	@refg_Id INT
AS
  BEGIN
          UPDATE [Refugio].[tbRefugios]
          SET    refg_EsEliminado	= 1
          WHERE  refg_Id		= @refg_Id
  END
GO


# [Refugio].[PR_Refugio_Refugios_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Refugios_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Refugios_Detail] 
@refg_Id int
AS BEGIN
SELECT 
    refg_Id,
    refg_Nombre,
    refg_Ubicacion,
    refg_RTN,
    refg_Telefono,
    refg_Correo,
    refugios.depto_Id,
    departamentos.depto_Descripcion,
    refugios.mpio_Id,
    municipios.mpio_Descripcion,
    refg_InformacionAdicional,
    Help.IsActive(refg_EsActivo) AS EsActivo,
    usuarioCrea.usu_Nombre AS refg_NombreUsuarioCrea,
    refg_FechaCrea,
    usuarioModifica.usu_Nombre AS refg_NombreUsuarioModifica,
    refg_FechaModifica
FROM [Refugio].[tbRefugios] AS refugios
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
    ON refugios.refg_UsuarioCrea = usuarioCrea.usu_Id 
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
    ON refugios.refg_UsuarioModifica = usuarioModifica.usu_Id
    INNER JOIN [General].[tbDepartamentos] AS departamentos
    ON refugios.depto_Id = departamentos.depto_Id
    INNER JOIN [General].[tbMunicipios] AS municipios
    ON refugios.mpio_Id = municipios.mpio_Id
WHERE refg_EsEliminado != 1 AND refugios.refg_Id = @refg_Id
END
GO


# [Refugio].[PR_Refugio_Refugios_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Refugios_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbRefugios
CREATE PROCEDURE [Refugio].[PR_Refugio_Refugios_Find]
@refg_Id INT
AS
BEGIN
    SELECT	refg_Id,
			refg_Nombre,
			refg_Ubicacion,
			refg_RTN,
			refg_Telefono,
			refg_Correo,
			depto_Id,
			mpio_Id,
			refg_InformacionAdicional,
			refg_EsActivo, 
			refg_UsuarioCrea,
			usuarioCrea.Usu_Nombre AS usuarioCrea,
			refg_FechaCrea,
			refg_UsuarioModifica,
			usuarioModifica.Usu_Nombre AS usuarioModifica,
			refg_FechaModifica
    FROM [Refugio].[tbRefugios] AS refugios
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		refugios.refg_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		refugios.refg_UsuarioModifica = usuarioModifica.usu_Id
    WHERE  refg_EsEliminado != 1
	AND 	refg_Id = @refg_Id
END
GO


# [Refugio].[PR_Refugio_Refugios_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Refugios_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Refugios_Insert]
    @refg_Nombre              VARCHAR(50),
    @refg_Ubicacion           VARCHAR(50),
    @refg_RTN                 VARCHAR(14),
    @refg_Telefono            VARCHAR(8),
    @refg_Correo              VARCHAR(150),
    @refg_InformacionAdicional VARCHAR(500),
    @depto_Id                 INT,
    @mpio_Id                  INT,
    @refg_EsActivo            BIT,          
    @refg_UsuarioCrea         INT
AS
BEGIN
    INSERT INTO [Refugio].[tbRefugios]
    (refg_Nombre, refg_Ubicacion, refg_RTN, refg_Telefono, refg_Correo,
     refg_InformacionAdicional, depto_Id, mpio_Id, refg_EsActivo, refg_EsEliminado,
     refg_UsuarioCrea, refg_FechaCrea)
    VALUES
    (@refg_Nombre, @refg_Ubicacion, @refg_RTN, @refg_Telefono, @refg_Correo,
     @refg_InformacionAdicional, @depto_Id, @mpio_Id, @refg_EsActivo, 0,
     @refg_UsuarioCrea, GETDATE())
END
GO


# [Refugio].[PR_Refugio_Refugios_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Refugios_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Refugios_List]
AS
BEGIN
    SELECT refg_Id,
           refg_Nombre,
           refg_RTN,
           refg_Ubicacion,
           CASE WHEN refg_EsActivo = 1 THEN 'Activo' ELSE 'Inactivo' END AS EsActivo
    FROM [Refugio].[tbRefugios]
    WHERE refg_EsEliminado != 1
END

GO


# [Refugio].[PR_Refugio_Refugios_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Refugios_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-----------------� UPDATE
CREATE PROCEDURE [Refugio].[PR_Refugio_Refugios_Update] 
@refg_Id INT,
@refg_Nombre NVARCHAR(50),
@refg_Ubicacion NVARCHAR(50),
@refg_RTN NVARCHAR(14),
@refg_Telefono NVARCHAR(8),
@refg_Correo VARCHAR(150),
@refg_InformacionAdicional NVARCHAR(500),
@refg_EsActivo bit,
@depto_Id INT,
@mpio_Id INT,
@refg_UsuarioModifica INT 

AS BEGIN 
BEGIN TRANSACTION 
    BEGIN TRY
UPDATE [Refugio].[tbRefugios]
SET refg_Nombre = @refg_Nombre,
  refg_Ubicacion = @refg_Ubicacion,
  refg_RTN = @refg_RTN,
  refg_Telefono = @refg_Telefono,
  refg_Correo = @refg_Correo,
  refg_InformacionAdicional = @refg_InformacionAdicional,
  refg_EsActivo = @refg_EsActivo,
  depto_Id = @depto_Id,
  mpio_Id = @mpio_Id,
  refg_UsuarioModifica = @refg_UsuarioModifica,
  refg_FechaModifica = Getdate()
WHERE refg_Id = @refg_Id 
  COMMIT TRANSACTION
END TRY 
BEGIN CATCH 
	ROLLBACK TRANSACTION
END CATCH
END 

GO


# [Refugio].[PR_Refugio_Solicitudes_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Solicitudes_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

 

-----------------> DELETE

CREATE PROCEDURE [Refugio].[PR_Refugio_Solicitudes_Delete] 
@sol_Id INT
AS
  BEGIN
          UPDATE [Refugio].[tbSolicitudes]
          SET    sol_EsEliminado	= 1
          WHERE  sol_Id			= @sol_Id
  END
GO


# [Refugio].[PR_Refugio_Solicitudes_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Solicitudes_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Solicitudes_Detail] 
@sol_Id INT
AS BEGIN
    SET NOCOUNT ON;
    
    SELECT	
        s.sol_Id,
        s.sol_Identidad,
        s.sol_Nombres,
        s.sol_Apellidos,
        s.sol_Telefono,
        s.sol_Correo,
        m.masc_Id,
        m.masc_Imagen,
        m.masc_Nombre,
        m.masc_EsAdoptado,
        r.raza_Descripcion,
        a.refg_Nombre,
        ISNULL(ad.adop_Estado, 'Pendiente') AS adop_Estado,
        uc.usu_Nombre AS sol_NombreUsuarioCrea,
        s.sol_FechaCrea,
        um.usu_Nombre AS sol_NombreUsuarioModifica,
        s.sol_FechaModifica
    FROM [Refugio].[tbSolicitudes] s
        INNER JOIN [Refugio].[tbMascotas] m ON s.masc_Id = m.masc_Id
        INNER JOIN [Refugio].[tbRazas] r ON m.raza_Id = r.raza_Id
        INNER JOIN [Refugio].[tbRefugios] a ON m.refg_Id = a.refg_Id
        LEFT JOIN [Seguridad].[tbUsuarios] uc ON s.sol_UsuarioCrea = uc.usu_Id 
        LEFT JOIN [Seguridad].[tbUsuarios] um ON s.sol_UsuarioModifica = um.usu_Id
        LEFT JOIN (
            SELECT sol_Id, adop_Estado,
                   ROW_NUMBER() OVER (PARTITION BY sol_Id ORDER BY adop_FechaCrea DESC) AS rn
            FROM [Refugio].[tbAdopciones]
        ) ad ON ad.sol_Id = s.sol_Id AND ad.rn = 1
    WHERE s.sol_EsEliminado = 0
      AND s.sol_Id = @sol_Id;
END
GO


# [Refugio].[PR_Refugio_Solicitudes_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Solicitudes_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbSolicitudes
CREATE PROCEDURE [Refugio].[PR_Refugio_Solicitudes_Find]
@sol_Id INT
AS
BEGIN
    SELECT	solicitudes.sol_Id,
			solicitudes.sol_Identidad,
			solicitudes.sol_Nombres,
			solicitudes.sol_Apellidos,
			solicitudes.sol_Telefono,
			solicitudes.sol_Correo,
			solicitudes.sol_Fecha,
			solicitudes.masc_Id,
			mascota.masc_Nombre,
			solicitudes.sol_UsuarioCrea,
			usuarioCrea.Usu_Nombre AS usuarioCrea,
			solicitudes.sol_FechaCrea,
			solicitudes.sol_UsuarioModifica,
			usuarioModifica.Usu_Nombre AS usuarioModifica,
			solicitudes.sol_FechaModifica
    FROM [Refugio].[tbSolicitudes] AS solicitudes
	INNER JOIN [Refugio].[tbMascotas] AS mascota
	ON		solicitudes.masc_Id = mascota.masc_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		solicitudes.sol_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		solicitudes.sol_UsuarioModifica = usuarioModifica.usu_Id
    WHERE	solicitudes.sol_EsEliminado != 1
	AND 	sol_Id = @sol_Id
END
GO


# [Refugio].[PR_Refugio_Solicitudes_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Solicitudes_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Solicitudes_Insert] 
    @sol_Identidad VARCHAR(13),
    @sol_Nombres NVARCHAR(50),
    @sol_Apellidos NVARCHAR(50),
    @sol_Telefono VARCHAR(8),
    @sol_Correo VARCHAR(150),
	@sol_Fecha DATETIME,
    @masc_Id INT,
    @sol_UsuarioCrea INT
AS BEGIN
INSERT INTO [Refugio].[tbSolicitudes] (
        sol_Identidad,
        sol_Nombres,
        sol_Apellidos,
        sol_Telefono,
        sol_Correo,
        sol_Fecha,
        masc_Id,
        sol_UsuarioCrea,
        sol_FechaCrea
    )
VALUES
    (
        @sol_Identidad,
        @sol_Nombres,
        @sol_Apellidos,
        @sol_Telefono,
        @sol_Correo,
		@sol_Fecha,
        @masc_Id,
        @sol_UsuarioCrea,
        Getdate()
    )
END
GO


# [Refugio].[PR_Refugio_Solicitudes_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Solicitudes_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbSolicitudes
CREATE PROCEDURE [Refugio].[PR_Refugio_Solicitudes_List]
AS
BEGIN
    SELECT	solicitudes.sol_Id,
			solicitudes.sol_Identidad,
			solicitudes.sol_Nombres,
			mascota.masc_Nombre,
			solicitudes.sol_Correo
    FROM [Refugio].[tbSolicitudes] AS solicitudes
	INNER JOIN [Refugio].[tbMascotas] AS mascota
	ON		solicitudes.masc_Id = mascota.masc_Id
    WHERE	solicitudes.sol_EsEliminado != 1
END
GO


# [Refugio].[PR_Refugio_Solicitudes_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Solicitudes_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Solicitudes_Update] 
@sol_Id INT,
@sol_Identidad VARCHAR(13),
@sol_Nombres NVARCHAR(50),
@sol_Apellidos NVARCHAR(50),
@sol_Telefono VARCHAR(8),
@sol_Correo VARCHAR(150),
@sol_Fecha DATETIME,
@masc_Id INT,
@sol_UsuarioModifica INT 
AS BEGIN
UPDATE [Refugio].[tbSolicitudes]
SET sol_Identidad = @sol_Identidad,
  sol_Nombres = @sol_Nombres,
  sol_Apellidos = @sol_Apellidos,
  sol_Telefono = @sol_Telefono,
  sol_Correo = @sol_Correo,
  sol_Fecha = @sol_Fecha,
  masc_Id = @masc_Id,
  sol_UsuarioModifica = @sol_UsuarioModifica,
  sol_FechaModifica = Getdate()
WHERE sol_Id = @sol_Id
END 
GO


# [Refugio].[PR_Refugio_Vacunas_Delete]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Vacunas_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
 
-----------------> DELETE

CREATE PROCEDURE [Refugio].[PR_Refugio_Vacunas_Delete] 
@vac_Id INT
AS
  BEGIN
          UPDATE [Refugio].[tbVacunas]
          SET    vac_EsEliminado	= 1
          WHERE  vac_Id			= @vac_Id
  END
GO


# [Refugio].[PR_Refugio_Vacunas_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Vacunas_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Vacunas_Detail] 
    @vac_Id INT 
AS
BEGIN
    SELECT  vac_Id,
            vac_Descripcion,
            vacu_Especie,
            vacu_DosisRecomendada,
            vacu_PeriodoRefuerzo,
            usuarioCrea.usu_Nombre AS vac_NombreUsuarioCrea,
            vac_FechaCrea,
            usuarioModifica.usu_Nombre AS vac_NombreUsuarioModifica,
            vac_FechaModifica
    FROM [Refugio].[tbVacunas] AS vacunas
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON vacunas.vac_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON vacunas.vac_UsuarioModifica = usuarioModifica.usu_Id
    WHERE vac_EsEliminado != 1
      AND vac_Id = @vac_Id 

END
GO


# [Refugio].[PR_Refugio_Vacunas_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Vacunas_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Vacunas_Find] 
@vac_Id INT
AS
BEGIN
    SELECT	vac_Id,
			vac_Descripcion,
			vacu_Especie,
			vacu_DosisRecomendada,
			vacu_PeriodoRefuerzo,
			vac_EsActivo,
			vac_UsuarioCrea,
			usuarioCrea.Usu_Nombre AS usuarioCrea,
			vac_FechaCrea,
			vac_UsuarioModifica,
			usuarioModifica.Usu_Nombre AS usuarioModifica,
			vac_FechaModifica
    FROM [Refugio].[tbVacunas] AS vacuna
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		vacuna.vac_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		vacuna.vac_UsuarioModifica = usuarioModifica.usu_Id
    WHERE vac_EsEliminado != 1
	AND 	vac_Id = @vac_Id
END

GO


# [Refugio].[PR_Refugio_Vacunas_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Vacunas_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Vacunas_Insert] 
    @vac_Descripcion nvarchar(100),
    @vacu_Especie varchar(50),
    @vacu_DosisRecomendada varchar(100),
    @vacu_PeriodoRefuerzo varchar(50),
    @vac_EsActivo bit,
    @vac_UsuarioCrea int
AS 
BEGIN
INSERT INTO [Refugio].[tbVacunas] (
        vac_Descripcion,
        vacu_Especie,
        vacu_DosisRecomendada,
        vacu_PeriodoRefuerzo,
        vac_EsActivo,
        vac_UsuarioCrea,
        vac_FechaCrea
    )
VALUES
    (
        @vac_Descripcion,
        @vacu_Especie,
        @vacu_DosisRecomendada,
        @vacu_PeriodoRefuerzo,
        @vac_EsActivo,
        @vac_UsuarioCrea,
        Getdate()
    )
END

GO


# [Refugio].[PR_Refugio_Vacunas_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Vacunas_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Vacunas_List]
AS
BEGIN
    SELECT vac_Id,
           vac_Descripcion,
           vacu_Especie,
           vacu_DosisRecomendada,
           vacu_PeriodoRefuerzo,
           Help.IsActive(vac_EsActivo) AS EsActivo
    FROM [Refugio].[tbVacunas]
    WHERE vac_EsEliminado != 1
END

GO


# [Refugio].[PR_Refugio_Vacunas_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Vacunas_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER OFF
GO

CREATE PROCEDURE [Refugio].[PR_Refugio_Vacunas_Update] 
@vac_Id int,
@vac_Descripcion nvarchar(100),
@vacu_Especie varchar(50),
@vacu_DosisRecomendada varchar(100),
@vacu_PeriodoRefuerzo varchar(50),
@vac_EsActivo bit,
@vac_UsuarioModifica int
AS BEGIN
UPDATE [Refugio].[tbVacunas]
SET	vac_Descripcion = @vac_Descripcion,
    vacu_Especie = @vacu_Especie,
    vacu_DosisRecomendada = @vacu_DosisRecomendada,
    vacu_PeriodoRefuerzo = @vacu_PeriodoRefuerzo,
    vac_EsActivo = @vac_EsActivo,
    vac_UsuarioModifica = @vac_UsuarioModifica,
    vac_FechaModifica = Getdate()
WHERE vac_Id = @vac_Id
END

GO


# [Refugio].[PR_Refugio_Voluntarios_Detail]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Voluntarios_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbVoluntarios
CREATE PROCEDURE [Refugio].[PR_Refugio_Voluntarios_Detail]
@vol_Id int
AS
BEGIN
    SELECT	vol_Id,
			vol_HorasTrabajadas,
			vol_Recurrente,
			persona.per_PrimerNombre,
			persona.per_SegundoNombre,
			persona.per_ApellidoPaterno,
			persona.per_ApellidoMaterno,
			persona.per_Identidad,
			persona.per_FechaNacimiento,
			persona.per_Domicilio,
			persona.per_Telefono,
			persona.per_Correo,
			persona.per_UsuarioCrea,
			usuarioCrea.usu_Nombre AS UsuarioCreacion,
			per_FechaCrea,
			persona.per_UsuarioModifica,
			usuarioModifica.usu_Nombre AS UsuarioModificacion,
			persona.per_FechaModifica
    FROM		[Refugio].[tbVoluntarios] AS voluntario
	INNER JOIN	[General].[tbPersonas] AS persona
	ON			voluntario.per_Id = persona.per_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		persona.per_UsuarioCrea = usuarioCrea.usu_Id 
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		persona.per_UsuarioModifica = usuarioModifica.usu_Id
    WHERE		persona.per_EsEliminado != 1 and voluntario.vol_Id = @vol_Id
END
GO


# [Refugio].[PR_Refugio_Voluntarios_Find]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Voluntarios_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbVoluntarios
CREATE PROCEDURE [Refugio].[PR_Refugio_Voluntarios_Find]
@vol_Id INT
AS
BEGIN
    SELECT	vol_Id,
			vol_HorasTrabajadas,
			voluntario.per_Id,
			persona.per_PrimerNombre,
			persona.per_SegundoNombre,
			persona.per_ApellidoPaterno,
			persona.per_ApellidoMaterno,
			persona.per_Identidad,
			persona.per_FechaNacimiento,
			persona.per_Domicilio,
			persona.per_Telefono,
			persona.per_Correo,
			vol_Recurrente,
			Help.IsActive(vol_Recurrente) AS estado,
			per_UsuarioCrea,
			usuarioCrea.Usu_Nombre AS usuarioCrea,
			per_FechaCrea,
			per_UsuarioModifica,
			usuarioModifica.Usu_Nombre AS usuarioModifica,
			per_FechaModifica
    FROM		[Refugio].[tbVoluntarios] AS voluntario
	INNER JOIN	[General].[tbPersonas] AS persona
	ON			voluntario.per_Id = persona.per_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
	ON		persona.per_UsuarioCrea = usuarioCrea.usu_Id
	LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
	ON		persona.per_UsuarioModifica = usuarioModifica.usu_Id
    WHERE		persona.per_EsEliminado != 1
	AND 	vol_Id = @vol_Id
END
GO


# [Refugio].[PR_Refugio_Voluntarios_Insert]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Voluntarios_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Refugio].[PR_Refugio_Voluntarios_Insert] 
    @vol_HorasTrabajadas INT,
	@vol_Recurrente BIT,
	@per_Identidad	varchar(13),
	@per_PrimerNombre	nvarchar(50),
	@per_SegundoNombre	nvarchar(50),
	@per_ApellidoPaterno	nvarchar(50),
	@per_ApellidoMaterno	nvarchar(50),
	@per_FechaNacimiento	date,
	@per_Domicilio	nvarchar(100),
	@per_Telefono	varchar(8),
	@per_Correo	varchar(150),
	@per_UsuarioCrea INT
AS BEGIN
--Guardamos datos en personas.
		EXEC	[General].[PR_General_Personas_Insert]
				@per_Identidad,
                @per_PrimerNombre,
                @per_SegundoNombre,
                @per_ApellidoPaterno,
                @per_ApellidoMaterno,
                @per_FechaNacimiento,
                @per_Domicilio,
                @per_Telefono,
                @per_Correo,
				@per_UsuarioCrea
		DECLARE @per_Id INT = IDENT_CURRENT('[General].[tbPersonas]');

		--Guardamos datos en voluntarios.
		INSERT INTO [Refugio].[tbVoluntarios] (
				vol_HorasTrabajadas,
				per_Id,
				vol_Recurrente
			)
		VALUES
			(
				@vol_HorasTrabajadas,
				@per_Id,
				@vol_Recurrente
			)
END
GO


# [Refugio].[PR_Refugio_Voluntarios_List]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Voluntarios_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

------------------> tbVoluntarios
CREATE PROCEDURE [Refugio].[PR_Refugio_Voluntarios_List]
AS
BEGIN
    SELECT	vol_Id,
			vol_HorasTrabajadas,
			Help.ConcatEspace(persona.per_PrimerNombre, persona.per_ApellidoPaterno) AS [vol_Nombres],
			persona.per_Identidad
    FROM		[Refugio].[tbVoluntarios] AS voluntario
	INNER JOIN	[General].[tbPersonas] AS persona
	ON			voluntario.per_Id = persona.per_Id
    WHERE		persona.per_EsEliminado != 1
END
GO


# [Refugio].[PR_Refugio_Voluntarios_Update]

In [0]:
/****** Object:  StoredProcedure [Refugio].[PR_Refugio_Voluntarios_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-----------------> UPDATE
CREATE PROCEDURE [Refugio].[PR_Refugio_Voluntarios_Update] 
@vol_Id INT,
@vol_HorasTrabajadas INT,
@per_Id INT,
@vol_Recurrente BIT,
@per_Identidad	varchar(13),
@per_PrimerNombre	nvarchar(50),
@per_SegundoNombre	nvarchar(50),
@per_ApellidoPaterno	nvarchar(50),
@per_ApellidoMaterno	nvarchar(50),
@per_FechaNacimiento	date,
@per_Domicilio	nvarchar(100),
@per_Telefono	varchar(8),
@per_Correo	varchar(150),
@per_UsuarioModifica INT

AS BEGIN
	--Guardamos datos en personas.
	EXEC	[General].[PR_General_Personas_Update]
			@per_Id,
			@per_Identidad,
            @per_PrimerNombre,
            @per_SegundoNombre,
            @per_ApellidoPaterno,
            @per_ApellidoMaterno,
            @per_FechaNacimiento,
            @per_Domicilio,
            @per_Telefono,
            @per_Correo,
			@per_UsuarioModifica

	--Guardamos datos en voluntario.
	UPDATE [Refugio].[tbVoluntarios]
	SET vol_HorasTrabajadas = @vol_HorasTrabajadas,
		per_Id = @per_Id,
		vol_Recurrente = @vol_Recurrente
	WHERE vol_Id = @vol_Id
END
GO


# [Reportes].[PR_Reportes_AdopcionesPorMes]

In [0]:
/****** Object:  StoredProcedure [Reportes].[PR_Reportes_AdopcionesPorMes]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Reportes].[PR_Reportes_AdopcionesPorMes]
    @mesesAtras INT = 6,
    @refg_Id INT = NULL
AS
BEGIN
    SET NOCOUNT ON;
    
    DECLARE @FechaInicio DATE = DATEADD(MONTH, -@mesesAtras, GETDATE())
    
    SELECT 
        YEAR(a.adop_FechaCrea) AS Año,
        MONTH(a.adop_FechaCrea) AS Mes,
        DATENAME(MONTH, a.adop_FechaCrea) AS NombreMes,
        COUNT(*) AS TotalAdopciones,
        CONCAT(DATENAME(MONTH, a.adop_FechaCrea), ' ', YEAR(a.adop_FechaCrea)) AS Periodo
    FROM [Refugio].[tbAdopciones] a
    INNER JOIN [Refugio].[tbSolicitudes] s ON a.sol_Id = s.sol_Id
    INNER JOIN [Refugio].[tbMascotas] m ON s.masc_Id = m.masc_Id
    WHERE a.adop_EsEliminado = 0 
      AND a.adop_EsAprobado = 1
      AND a.adop_FechaCrea >= @FechaInicio
      AND (@refg_Id IS NULL OR m.refg_Id = @refg_Id)
    GROUP BY YEAR(a.adop_FechaCrea), MONTH(a.adop_FechaCrea), DATENAME(MONTH, a.adop_FechaCrea)
    ORDER BY Año, Mes
END

GO


# [Reportes].[PR_Reportes_CitasMedicasPorTipo]

In [0]:
/****** Object:  StoredProcedure [Reportes].[PR_Reportes_CitasMedicasPorTipo]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Reportes].[PR_Reportes_CitasMedicasPorTipo]
    @fechaInicio DATETIME = NULL,
    @fechaFin DATETIME = NULL,
    @refg_Id INT = NULL
AS
BEGIN
    SET NOCOUNT ON;
    
    -- Si no se especifican fechas, usar los últimos 3 meses
    IF @fechaInicio IS NULL
        SET @fechaInicio = DATEADD(MONTH, -3, GETDATE())
    
    IF @fechaFin IS NULL
        SET @fechaFin = GETDATE()
    
    DECLARE @TotalCitas INT
    
    -- Obtener total de citas para calcular porcentajes
    SELECT @TotalCitas = COUNT(*)
    FROM [Refugio].[tbCitaMedica] cm
    INNER JOIN [Refugio].[tbMascotas] m ON cm.masc_Id = m.masc_Id
    WHERE cm.medic_FechaConsulta >= @fechaInicio 
      AND cm.medic_FechaConsulta <= @fechaFin
      AND m.masc_EsEliminado = 0
      AND (@refg_Id IS NULL OR m.refg_Id = @refg_Id)
    
    SELECT 
        ISNULL(cm.medic_TipoConsulta, 'Sin tipo') AS medic_TipoConsulta,
        COUNT(*) AS TotalCitas,
        CASE 
            WHEN @TotalCitas > 0 THEN 
                CAST(COUNT(*) AS DECIMAL(5,2)) / @TotalCitas * 100 
            ELSE 0 
        END AS PorcentajeCitas
    FROM [Refugio].[tbCitaMedica] cm
    INNER JOIN [Refugio].[tbMascotas] m ON cm.masc_Id = m.masc_Id
    WHERE cm.medic_FechaConsulta >= @fechaInicio 
      AND cm.medic_FechaConsulta <= @fechaFin
      AND m.masc_EsEliminado = 0
      AND (@refg_Id IS NULL OR m.refg_Id = @refg_Id)
    GROUP BY cm.medic_TipoConsulta
    ORDER BY TotalCitas DESC
END

GO


# [Reportes].[PR_Reportes_Dashboard]

In [0]:
/****** Object:  StoredProcedure [Reportes].[PR_Reportes_Dashboard]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Reportes].[PR_Reportes_Dashboard]
AS
BEGIN
    SET NOCOUNT ON;
    
    DECLARE @TotalMascotas INT = 0
    DECLARE @MascotasAdoptadas INT = 0
    DECLARE @MascotasDisponibles INT = 0
    DECLARE @CitasMedicasPendientes INT = 0
    DECLARE @VoluntariosActivos INT = 0
    DECLARE @EventosEsteMes INT = 0
    DECLARE @PorcentajeAdopciones DECIMAL(5,2) = 0
    
    -- Total de mascotas
    SELECT @TotalMascotas = COUNT(*) 
    FROM [Refugio].[tbMascotas] 
    WHERE masc_EsEliminado = 0
    
    -- Mascotas adoptadas
    SELECT @MascotasAdoptadas = COUNT(*) 
    FROM [Refugio].[tbMascotas] 
    WHERE masc_EsEliminado = 0 AND masc_EsAdoptado = 1
    
    -- Mascotas disponibles
    SET @MascotasDisponibles = @TotalMascotas - @MascotasAdoptadas
    
    -- Citas médicas pendientes (próximas 30 días)
    SELECT @CitasMedicasPendientes = COUNT(*) 
    FROM [Refugio].[tbCitaMedica] cm
    INNER JOIN [Refugio].[tbMascotas] m ON cm.masc_Id = m.masc_Id
    WHERE m.masc_EsEliminado = 0 
      AND cm.medic_ProximaCita >= GETDATE() 
      AND cm.medic_ProximaCita <= DATEADD(DAY, 30, GETDATE())
    
    -- Voluntarios activos (que han participado en eventos en los últimos 6 meses)
    SELECT @VoluntariosActivos = COUNT(DISTINCT v.vol_Id)
    FROM [Refugio].[tbVoluntarios] v
    INNER JOIN [Refugio].[tbEventos_tbVoluntarios] ev ON v.vol_Id = ev.vol_Id
    INNER JOIN [Refugio].[tbEventos] e ON ev.eve_Id = e.eve_Id
    WHERE e.eve_Fecha >= DATEADD(MONTH, -6, GETDATE())
      AND e.eve_EsEliminado = 0
    
    -- Eventos del mes actual
    SELECT @EventosEsteMes = COUNT(*) 
    FROM [Refugio].[tbEventos] 
    WHERE eve_EsEliminado = 0 
      AND YEAR(eve_Fecha) = YEAR(GETDATE()) 
      AND MONTH(eve_Fecha) = MONTH(GETDATE())
    
    -- Calcular porcentaje de adopciones
    IF @TotalMascotas > 0
        SET @PorcentajeAdopciones = CAST(@MascotasAdoptadas AS DECIMAL(5,2)) / @TotalMascotas * 100
    
    -- Retornar resultados
    SELECT 
        @TotalMascotas AS TotalMascotas,
        @MascotasAdoptadas AS MascotasAdoptadas,
        @MascotasDisponibles AS MascotasDisponibles,
        @CitasMedicasPendientes AS CitasMedicasPendientes,
        @VoluntariosActivos AS VoluntariosActivos,
        @EventosEsteMes AS EventosEsteMes,
        @PorcentajeAdopciones AS PorcentajeAdopciones
END

GO


# [Reportes].[PR_Reportes_Eventos]

In [0]:
/****** Object:  StoredProcedure [Reportes].[PR_Reportes_Eventos]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Reportes].[PR_Reportes_Eventos]
    @fechaInicio DATETIME = NULL,
    @fechaFin DATETIME = NULL,
    @refg_Id INT = NULL,
    @soloFuturos BIT = 0
AS
BEGIN
    SET NOCOUNT ON;
    
    -- Si no se especifican fechas, usar los últimos 6 meses hacia adelante
    IF @fechaInicio IS NULL
        SET @fechaInicio = DATEADD(MONTH, -6, GETDATE())
    
    IF @fechaFin IS NULL
        SET @fechaFin = DATEADD(MONTH, 6, GETDATE())
    
    SELECT 
        e.eve_Id,
        e.eve_Descripcion,
        e.eve_Fecha,
        e.eve_HoraInicio,
        e.eve_HoraFinal,
        r.refg_Nombre,
        COUNT(DISTINCT ev.vol_Id) AS VoluntariosParticipantes,
        CASE 
            WHEN e.eve_Fecha > GETDATE() THEN 'Próximo'
            WHEN e.eve_Fecha = CAST(GETDATE() AS DATE) THEN 'En curso'
            ELSE 'Realizado'
        END AS Estado,
        e.eve_FechaCrea
    FROM [Refugio].[tbEventos] e
    INNER JOIN [Refugio].[tbRefugios] r ON e.refg_Id = r.refg_Id
    LEFT JOIN [Refugio].tbEventos_tbVoluntarios ev ON e.eve_Id = ev.eve_Id
    WHERE e.eve_EsEliminado = 0
      AND e.eve_Fecha >= @fechaInicio 
      AND e.eve_Fecha <= @fechaFin
      AND (@refg_Id IS NULL OR e.refg_Id = @refg_Id)
      AND (@soloFuturos = 0 OR e.eve_Fecha >= GETDATE())
    GROUP BY 
        e.eve_Id, e.eve_Descripcion, e.eve_Fecha, e.eve_HoraInicio, 
        e.eve_HoraFinal, r.refg_Nombre, e.eve_FechaCrea
    ORDER BY e.eve_Fecha DESC
END

GO


# [Reportes].[PR_Reportes_MascotasPorRaza]

In [0]:
/****** Object:  StoredProcedure [Reportes].[PR_Reportes_MascotasPorRaza]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Reportes].[PR_Reportes_MascotasPorRaza]
    @refg_Id INT = NULL
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        r.raza_Id,
        ISNULL(r.raza_Descripcion, 'Sin raza') AS raza_Descripcion,
        COUNT(*) AS TotalMascotas,
        SUM(CASE WHEN m.masc_EsAdoptado = 1 THEN 1 ELSE 0 END) AS MascotasAdoptadas,
        SUM(CASE WHEN m.masc_EsAdoptado = 0 THEN 1 ELSE 0 END) AS MascotasDisponibles,
        CASE 
            WHEN COUNT(*) > 0 THEN 
                CAST(SUM(CASE WHEN m.masc_EsAdoptado = 1 THEN 1 ELSE 0 END) AS DECIMAL(5,2)) / COUNT(*) * 100 
            ELSE 0 
        END AS PorcentajeAdopcion
    FROM [Refugio].[tbMascotas] m
    LEFT JOIN [Refugio].[tbRazas] r ON m.raza_Id = r.raza_Id
    WHERE m.masc_EsEliminado = 0
      AND (@refg_Id IS NULL OR m.refg_Id = @refg_Id)
    GROUP BY r.raza_Id, r.raza_Descripcion
    ORDER BY TotalMascotas DESC
END

GO


# [Reportes].[PR_Reportes_SaludMascotas]

In [0]:
/****** Object:  StoredProcedure [Reportes].[PR_Reportes_SaludMascotas]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Reportes].[PR_Reportes_SaludMascotas]
    @refg_Id INT = NULL,
    @soloProblematicas BIT = 0
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        m.masc_Id,
        m.masc_Nombre,
        ISNULL(r.raza_Descripcion, 'Sin raza') AS raza_Descripcion,
        ref.refg_Nombre,
        (SELECT TOP 1 cm.medic_FechaConsulta 
         FROM [Refugio].[tbCitaMedica] cm 
         WHERE cm.masc_Id = m.masc_Id 
         ORDER BY cm.medic_FechaConsulta DESC) AS UltimaCitaMedica,
        ISNULL(
            (SELECT TOP 1 cm.medic_Diagnostico 
             FROM [Refugio].[tbCitaMedica] cm 
             WHERE cm.masc_Id = m.masc_Id 
             ORDER BY cm.medic_FechaConsulta DESC), 
            'Sin información'
        ) AS EstadoSalud,
        (SELECT COUNT(*) 
         FROM [Refugio].[tbCitaMedica] cm 
         WHERE cm.masc_Id = m.masc_Id) AS TotalCitas,
        CASE 
            WHEN (SELECT TOP 1 cm.medic_FechaConsulta 
                  FROM [Refugio].[tbCitaMedica] cm 
                  WHERE cm.masc_Id = m.masc_Id 
                  ORDER BY cm.medic_FechaConsulta DESC) IS NOT NULL
            THEN DATEDIFF(DAY, 
                 (SELECT TOP 1 cm.medic_FechaConsulta 
                  FROM [Refugio].[tbCitaMedica] cm 
                  WHERE cm.masc_Id = m.masc_Id 
                  ORDER BY cm.medic_FechaConsulta DESC), 
                 GETDATE())
            ELSE NULL
        END AS DiasSinCita,
        CASE 
            WHEN EXISTS (
                SELECT 1 FROM [Refugio].[tbCitaMedica] cm 
                WHERE cm.masc_Id = m.masc_Id 
                  AND cm.vac_Id IS NOT NULL 
                  AND cm.medic_FechaConsulta >= DATEADD(YEAR, -1, GETDATE())
            ) THEN 1 
            ELSE 0 
        END AS VacunasAlDia,
        CASE 
            WHEN (SELECT TOP 1 cm.medic_FechaConsulta 
                  FROM [Refugio].[tbCitaMedica] cm 
                  WHERE cm.masc_Id = m.masc_Id 
                  ORDER BY cm.medic_FechaConsulta DESC) IS NULL 
                 OR DATEDIFF(DAY, 
                    (SELECT TOP 1 cm.medic_FechaConsulta 
                     FROM [Refugio].[tbCitaMedica] cm 
                     WHERE cm.masc_Id = m.masc_Id 
                     ORDER BY cm.medic_FechaConsulta DESC), 
                    GETDATE()) > 180 
            THEN 'Alta'
            WHEN DATEDIFF(DAY, 
                 (SELECT TOP 1 cm.medic_FechaConsulta 
                  FROM [Refugio].[tbCitaMedica] cm 
                  WHERE cm.masc_Id = m.masc_Id 
                  ORDER BY cm.medic_FechaConsulta DESC), 
                 GETDATE()) > 90 
            THEN 'Media'
            ELSE 'Baja'
        END AS PrioridadAtencion,
        m.masc_Peso,
        m.masc_Edad
    FROM [Refugio].[tbMascotas] m
    LEFT JOIN [Refugio].[tbRazas] r ON m.raza_Id = r.raza_Id
    INNER JOIN [Refugio].[tbRefugios] ref ON m.refg_Id = ref.refg_Id
    WHERE m.masc_EsEliminado = 0
      AND (@refg_Id IS NULL OR m.refg_Id = @refg_Id)
      AND (@soloProblematicas = 0 OR 
           (SELECT TOP 1 cm.medic_FechaConsulta 
            FROM [Refugio].[tbCitaMedica] cm 
            WHERE cm.masc_Id = m.masc_Id 
            ORDER BY cm.medic_FechaConsulta DESC) IS NULL 
           OR DATEDIFF(DAY, 
              (SELECT TOP 1 cm.medic_FechaConsulta 
               FROM [Refugio].[tbCitaMedica] cm 
               WHERE cm.masc_Id = m.masc_Id 
               ORDER BY cm.medic_FechaConsulta DESC), 
              GETDATE()) > 90
          )
    ORDER BY 
        CASE 
            WHEN (SELECT TOP 1 cm.medic_FechaConsulta 
                  FROM [Refugio].[tbCitaMedica] cm 
                  WHERE cm.masc_Id = m.masc_Id 
                  ORDER BY cm.medic_FechaConsulta DESC) IS NULL THEN 1
            WHEN DATEDIFF(DAY, 
                 (SELECT TOP 1 cm.medic_FechaConsulta 
                  FROM [Refugio].[tbCitaMedica] cm 
                  WHERE cm.masc_Id = m.masc_Id 
                  ORDER BY cm.medic_FechaConsulta DESC), 
                 GETDATE()) > 180 THEN 2
            WHEN DATEDIFF(DAY, 
                 (SELECT TOP 1 cm.medic_FechaConsulta 
                  FROM [Refugio].[tbCitaMedica] cm 
                  WHERE cm.masc_Id = m.masc_Id 
                  ORDER BY cm.medic_FechaConsulta DESC), 
                 GETDATE()) > 90 THEN 3
            ELSE 4
        END,
        (SELECT TOP 1 cm.medic_FechaConsulta 
         FROM [Refugio].[tbCitaMedica] cm 
         WHERE cm.masc_Id = m.masc_Id 
         ORDER BY cm.medic_FechaConsulta DESC) DESC
END

GO


# [Reportes].[PR_Reportes_Voluntarios]

In [0]:
/****** Object:  StoredProcedure [Reportes].[PR_Reportes_Voluntarios]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


CREATE PROCEDURE [Reportes].[PR_Reportes_Voluntarios]
    @soloActivos BIT = 0,
    @refg_Id INT = NULL
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        v.vol_Id,
        v.per_Id,
        CONCAT(p.per_PrimerNombre, ' ', ISNULL(p.per_SegundoNombre + ' ', ''), 
               p.per_ApellidoPaterno, ' ', ISNULL(p.per_ApellidoMaterno, '')) AS NombreCompleto,
        p.per_PrimerNombre,
        p.per_ApellidoPaterno,
        p.per_Telefono,
        p.per_Correo,
        COUNT(DISTINCT ev.eve_Id) AS EventosParticipados,
        ISNULL(v.vol_HorasTrabajadas, 0) AS vol_HorasTrabajadas,
        MAX(e.eve_Fecha) AS UltimaParticipacion,
        CASE 
            WHEN COUNT(DISTINCT ev.eve_Id) > 0 AND MAX(e.eve_Fecha) >= DATEADD(MONTH, -6, GETDATE()) 
            THEN 'Activo' 
            ELSE 'Inactivo' 
        END AS Estado,
        v.vol_Recurrente
    FROM [Refugio].[tbVoluntarios] v
    INNER JOIN [General].[tbPersonas] p ON v.per_Id = p.per_Id
    LEFT JOIN [Refugio].tbEventos_tbVoluntarios ev ON v.vol_Id = ev.vol_Id
    LEFT JOIN [Refugio].[tbEventos] e ON ev.eve_Id = e.eve_Id AND e.eve_EsEliminado = 0
    WHERE p.per_EsEliminado = 0
      AND (@refg_Id IS NULL OR EXISTS (
          SELECT 1 FROM [Refugio].[tbEventos] et 
          WHERE et.eve_Id = e.eve_Id AND et.refg_Id = @refg_Id
      ))
    GROUP BY 
        v.vol_Id, v.per_Id, p.per_PrimerNombre, p.per_SegundoNombre, 
        p.per_ApellidoPaterno, p.per_ApellidoMaterno, p.per_Telefono, 
        p.per_Correo, v.vol_HorasTrabajadas, v.vol_Recurrente
    HAVING 
        (@soloActivos = 0 OR 
         (COUNT(DISTINCT ev.eve_Id) > 0 AND MAX(e.eve_Fecha) >= DATEADD(MONTH, -6, GETDATE())))
    ORDER BY EventosParticipados DESC, UltimaParticipacion DESC
END

GO


# [Rescate].[PR_Rescate_Ingresos_Delete]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_Ingresos_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_Ingresos_Delete]
    @ingr_Id INT
AS
BEGIN
    UPDATE [Rescate].[tbIngresos]
    SET ingr_EsEliminado = 1
    WHERE ingr_Id = @ingr_Id
END
GO


# [Rescate].[PR_Rescate_Ingresos_Detail]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_Ingresos_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_Ingresos_Detail]
AS
BEGIN
    SELECT  ingr.ingr_Id,
            ingr.repa_Id,
            ingr.refg_Id,
            refg.refg_Nombre AS NombreRefugio,
            ingr.ingr_FechaIngreso,
            ingr.ingr_LugarRescate,
            ingr.ingr_CondicionInicial,
            ingr.ingr_PersonaRescatista,
            ingr.ingr_MedioTransporte,
            ingr.ingr_Observaciones,
            ingr.ingr_EsEmergencia,
            -- Datos del reporte (si existe)
            repa.repa_UbicacionIncidente AS LugarReporte,
            repa.repa_DescripcionAnimal,
            repa.repa_EstadoAtencion,
            repa.repa_NombreReportante,
            repa.repa_TelefonoContacto AS TelefonoReportante,
            -- Auditoría
            usuarioCrea.usu_Nombre AS UsuarioCreacion,
            ingr.ingr_FechaCrea,
            usuarioModifica.usu_Nombre AS UsuarioModificacion,
            ingr.ingr_FechaModifica,
            -- Mascota asociada (si existe)
            (SELECT COUNT(*) FROM [Refugio].[tbMascotas] WHERE masc_IngresoId = ingr.ingr_Id AND masc_EsEliminado != 1) AS TieneMascota,
            (SELECT TOP 1 masc_Id FROM [Refugio].[tbMascotas] WHERE masc_IngresoId = ingr.ingr_Id AND masc_EsEliminado != 1) AS MascotaId,
            (SELECT TOP 1 masc_Nombre FROM [Refugio].[tbMascotas] WHERE masc_IngresoId = ingr.ingr_Id AND masc_EsEliminado != 1) AS MascotaNombre
    FROM [Rescate].[tbIngresos] AS ingr
    INNER JOIN [Refugio].[tbRefugios] AS refg
        ON ingr.refg_Id = refg.refg_Id
    LEFT JOIN [Rescate].[tbReportesAbandono] AS repa
        ON ingr.repa_Id = repa.repa_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON ingr.ingr_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON ingr.ingr_UsuarioModifica = usuarioModifica.usu_Id
    WHERE ingr.ingr_EsEliminado != 1
    ORDER BY ingr.ingr_FechaIngreso DESC
END
GO


# [Rescate].[PR_Rescate_Ingresos_Find]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_Ingresos_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_Ingresos_Find]
    @ingr_Id INT
AS
BEGIN
    SELECT  ingr.ingr_Id,
            ingr.repa_Id,
            ingr.refg_Id,
            refg.refg_Nombre AS NombreRefugio,
            ingr.ingr_FechaIngreso,
            ingr.ingr_LugarRescate,
            ingr.ingr_CondicionInicial,
            ingr.ingr_PersonaRescatista,
            ingr.ingr_MedioTransporte,
            ingr.ingr_Observaciones,
            ingr.ingr_EsEmergencia,
            -- Datos del reporte (si existe)
            repa.repa_UbicacionIncidente AS LugarReporte,
            repa.repa_DescripcionAnimal,
            repa.repa_EstadoAtencion,
            repa.repa_NombreReportante,
            repa.repa_TelefonoContacto AS TelefonoReportante,
            -- Auditoría
            ingr.ingr_UsuarioCrea,
            usuarioCrea.usu_Nombre AS usuarioCrea,
            ingr.ingr_FechaCrea,
            ingr.ingr_UsuarioModifica,
            usuarioModifica.usu_Nombre AS usuarioModifica,
            ingr.ingr_FechaModifica,
            -- Mascota asociada (si existe)
            (SELECT COUNT(*) FROM [Refugio].[tbMascotas] WHERE masc_IngresoId = ingr.ingr_Id AND masc_EsEliminado != 1) AS TieneMascota,
            (SELECT TOP 1 masc_Id FROM [Refugio].[tbMascotas] WHERE masc_IngresoId = ingr.ingr_Id AND masc_EsEliminado != 1) AS MascotaId,
            (SELECT TOP 1 masc_Nombre FROM [Refugio].[tbMascotas] WHERE masc_IngresoId = ingr.ingr_Id AND masc_EsEliminado != 1) AS MascotaNombre
    FROM [Rescate].[tbIngresos] AS ingr
    INNER JOIN [Refugio].[tbRefugios] AS refg
        ON ingr.refg_Id = refg.refg_Id
    LEFT JOIN [Rescate].[tbReportesAbandono] AS repa
        ON ingr.repa_Id = repa.repa_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON ingr.ingr_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON ingr.ingr_UsuarioModifica = usuarioModifica.usu_Id
    WHERE ingr.ingr_EsEliminado != 1
    AND ingr.ingr_Id = @ingr_Id
END
GO


# [Rescate].[PR_Rescate_Ingresos_Insert]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_Ingresos_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_Ingresos_Insert]
    @repa_Id INT,
    @refg_Id INT,
    @ingr_FechaIngreso DATETIME,
    @ingr_LugarRescate VARCHAR(200),
    @ingr_CondicionInicial VARCHAR(200),
    @ingr_PersonaRescatista VARCHAR(150),
    @ingr_MedioTransporte VARCHAR(100),
    @ingr_Observaciones VARCHAR(300),
    @ingr_EsEmergencia BIT,
    @ingr_UsuarioCrea INT
AS
BEGIN
    INSERT INTO [Rescate].[tbIngresos]
    (
        repa_Id,
        refg_Id,
        ingr_FechaIngreso,
        ingr_LugarRescate,
        ingr_CondicionInicial,
        ingr_PersonaRescatista,
        ingr_MedioTransporte,
        ingr_Observaciones,
        ingr_EsEmergencia,
        ingr_UsuarioCrea,
        ingr_FechaCrea
    )
    VALUES
    (
        @repa_Id,
        @refg_Id,
        @ingr_FechaIngreso,
        @ingr_LugarRescate,
        @ingr_CondicionInicial,
        @ingr_PersonaRescatista,
        @ingr_MedioTransporte,
        @ingr_Observaciones,
        @ingr_EsEmergencia,
        @ingr_UsuarioCrea,
        GETDATE()
    )

    -- Si viene de un reporte, actualizar estado a "En Proceso"
    IF @repa_Id IS NOT NULL
    BEGIN
        UPDATE [Rescate].[tbReportesAbandono]
        SET repa_EstadoAtencion = 'En Proceso'
        WHERE repa_Id = @repa_Id
    END
END
GO


# [Rescate].[PR_Rescate_Ingresos_List]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_Ingresos_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_Ingresos_List]
AS
BEGIN
    SELECT  ROW_NUMBER() OVER(ORDER BY ingr.ingr_FechaIngreso DESC) AS Fila,
            ingr.ingr_Id,
            ingr.repa_Id,
            ingr.refg_Id,
            refg.refg_Nombre,
            ingr.ingr_FechaIngreso,
            ingr.ingr_LugarRescate,
            ingr.ingr_CondicionInicial,
            ingr.ingr_PersonaRescatista,
            ingr.ingr_MedioTransporte,
            ingr.ingr_Observaciones,
            ingr.ingr_EsEmergencia,
            -- Datos del reporte (si existe)
            repa.repa_UbicacionIncidente AS LugarReporte,
            repa.repa_DescripcionAnimal,
            -- Verificar si ya tiene mascota asociada
            (SELECT COUNT(*) FROM [Refugio].[tbMascotas] WHERE masc_IngresoId = ingr.ingr_Id AND masc_EsEliminado != 1) AS TieneMascota
    FROM [Rescate].[tbIngresos] AS ingr
    INNER JOIN [Refugio].[tbRefugios] AS refg
        ON ingr.refg_Id = refg.refg_Id
    LEFT JOIN [Rescate].[tbReportesAbandono] AS repa
        ON ingr.repa_Id = repa.repa_Id
    WHERE ingr.ingr_EsEliminado != 1
    ORDER BY ingr.ingr_FechaIngreso DESC
END
GO


# [Rescate].[PR_Rescate_Ingresos_Update]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_Ingresos_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_Ingresos_Update]
    @ingr_Id INT,
    @repa_Id INT,
    @refg_Id INT,
    @ingr_FechaIngreso DATETIME,
    @ingr_LugarRescate VARCHAR(200),
    @ingr_CondicionInicial VARCHAR(200),
    @ingr_PersonaRescatista VARCHAR(150),
    @ingr_MedioTransporte VARCHAR(100),
    @ingr_Observaciones VARCHAR(300),
    @ingr_EsEmergencia BIT,
    @ingr_UsuarioModifica INT
AS
BEGIN
    UPDATE [Rescate].[tbIngresos]
    SET repa_Id = @repa_Id,
        refg_Id = @refg_Id,
        ingr_FechaIngreso = @ingr_FechaIngreso,
        ingr_LugarRescate = @ingr_LugarRescate,
        ingr_CondicionInicial = @ingr_CondicionInicial,
        ingr_PersonaRescatista = @ingr_PersonaRescatista,
        ingr_MedioTransporte = @ingr_MedioTransporte,
        ingr_Observaciones = @ingr_Observaciones,
        ingr_EsEmergencia = @ingr_EsEmergencia,
        ingr_UsuarioModifica = @ingr_UsuarioModifica,
        ingr_FechaModifica = GETDATE()
    WHERE ingr_Id = @ingr_Id
END
GO


# [Rescate].[PR_Rescate_ReportantesTipo_Delete]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportantesTipo_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportantesTipo_Delete]
    @reptip_Id INT
AS
BEGIN
    UPDATE [Rescate].[tbReportantesTipo]
    SET reptip_EsEliminado = 1
    WHERE reptip_Id = @reptip_Id
END
GO


# [Rescate].[PR_Rescate_ReportantesTipo_Detail]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportantesTipo_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportantesTipo_Detail]
AS
BEGIN
    SELECT  reptip_Id,
            reptip_Descripcion,
            reptip_EsActivo,
            usuarioCrea.usu_Nombre AS UsuarioCreacion,
            reptip_FechaCrea,
            usuarioModifica.usu_Nombre AS UsuarioModificacion,
            reptip_FechaModifica
    FROM [Rescate].[tbReportantesTipo] AS reptip
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON reptip.reptip_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON reptip.reptip_UsuarioModifica = usuarioModifica.usu_Id
    WHERE reptip_EsEliminado != 1
    ORDER BY reptip_Descripcion
END
GO


# [Rescate].[PR_Rescate_ReportantesTipo_Find]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportantesTipo_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportantesTipo_Find]
    @reptip_Id INT
AS
BEGIN
    SELECT  reptip_Id,
            reptip_Descripcion,
            reptip_EsActivo,
            reptip_UsuarioCrea,
            usuarioCrea.usu_Nombre AS usuarioCrea,
            reptip_FechaCrea,
            reptip_UsuarioModifica,
            usuarioModifica.usu_Nombre AS usuarioModifica,
            reptip_FechaModifica
    FROM [Rescate].[tbReportantesTipo] AS reptip
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON reptip.reptip_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON reptip.reptip_UsuarioModifica = usuarioModifica.usu_Id
    WHERE reptip_EsEliminado != 1
    AND reptip_Id = @reptip_Id
END
GO


# [Rescate].[PR_Rescate_ReportantesTipo_Insert]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportantesTipo_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportantesTipo_Insert]
    @reptip_Descripcion VARCHAR(100),
    @reptip_EsActivo BIT,
    @reptip_UsuarioCrea INT
AS
BEGIN
    INSERT INTO [Rescate].[tbReportantesTipo]
    (
        reptip_Descripcion,
        reptip_EsActivo,
        reptip_UsuarioCrea,
        reptip_FechaCrea
    )
    VALUES
    (
        @reptip_Descripcion,
        @reptip_EsActivo,
        @reptip_UsuarioCrea,
        GETDATE()
    )
END
GO


# [Rescate].[PR_Rescate_ReportantesTipo_List]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportantesTipo_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportantesTipo_List]
AS
BEGIN
    SELECT  ROW_NUMBER() OVER(ORDER BY reptip_Id ASC) AS Fila,
            reptip_Id,
            reptip_Descripcion,
            reptip_EsActivo
    FROM [Rescate].[tbReportantesTipo]
    WHERE reptip_EsEliminado != 1
    ORDER BY reptip_Descripcion
END
GO


# [Rescate].[PR_Rescate_ReportantesTipo_Update]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportantesTipo_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportantesTipo_Update]
    @reptip_Id INT,
    @reptip_Descripcion VARCHAR(100),
    @reptip_EsActivo BIT,
    @reptip_UsuarioModifica INT
AS
BEGIN
    UPDATE [Rescate].[tbReportantesTipo]
    SET reptip_Descripcion = @reptip_Descripcion,
        reptip_EsActivo = @reptip_EsActivo,
        reptip_UsuarioModifica = @reptip_UsuarioModifica,
        reptip_FechaModifica = GETDATE()
    WHERE reptip_Id = @reptip_Id
END
GO


# [Rescate].[PR_Rescate_ReportesAbandono_Delete]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportesAbandono_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportesAbandono_Delete]
    @repa_Id INT
AS
BEGIN
    UPDATE [Rescate].[tbReportesAbandono]
    SET repa_EsEliminado = 1
    WHERE repa_Id = @repa_Id
END
GO


# [Rescate].[PR_Rescate_ReportesAbandono_Detail]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportesAbandono_Detail]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportesAbandono_Detail]
AS
BEGIN
    SELECT  repa.repa_Id,
            repa.reptip_Id,
            reptip.reptip_Descripcion AS TipoReportante,
            repa.repa_NombreReportante,
            repa.repa_TelefonoContacto,
            repa.repa_Email,
            repa.repa_FechaReporte,
            repa.repa_UbicacionIncidente,
            repa.repa_DescripcionAnimal,
            repa.repa_EstadoAtencion,
            repa.repa_Observaciones,
            repa.repa_EsAnonimo,
            repa.refg_Id,
            refg.refg_Nombre AS NombreRefugio,
            usuarioCrea.usu_Nombre AS UsuarioCreacion,
            repa.repa_FechaCrea,
            usuarioModifica.usu_Nombre AS UsuarioModificacion,
            repa.repa_FechaModifica
    FROM [Rescate].[tbReportesAbandono] AS repa
    INNER JOIN [Rescate].[tbReportantesTipo] AS reptip
        ON repa.reptip_Id = reptip.reptip_Id
    INNER JOIN [Refugio].[tbRefugios] AS refg
        ON repa.refg_Id = refg.refg_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON repa.repa_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON repa.repa_UsuarioModifica = usuarioModifica.usu_Id
    WHERE repa.repa_EsEliminado != 1
    ORDER BY repa.repa_FechaReporte DESC
END
GO


# [Rescate].[PR_Rescate_ReportesAbandono_Find]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportesAbandono_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportesAbandono_Find]
    @repa_Id INT
AS
BEGIN
    SELECT  repa.repa_Id,
            repa.reptip_Id,
            reptip.reptip_Descripcion AS TipoReportante,
            repa.repa_NombreReportante,
            repa.repa_TelefonoContacto,
            repa.repa_Email,
            repa.repa_FechaReporte,
            repa.repa_UbicacionIncidente,
            repa.repa_DescripcionAnimal,
            repa.repa_EstadoAtencion,
            repa.repa_Observaciones,
            repa.repa_EsAnonimo,
            repa.refg_Id,
            refg.refg_Nombre AS NombreRefugio,
            repa.repa_UsuarioCrea,
            usuarioCrea.usu_Nombre AS usuarioCrea,
            repa.repa_FechaCrea,
            repa.repa_UsuarioModifica,
            usuarioModifica.usu_Nombre AS usuarioModifica,
            repa.repa_FechaModifica
    FROM [Rescate].[tbReportesAbandono] AS repa
    INNER JOIN [Rescate].[tbReportantesTipo] AS reptip
        ON repa.reptip_Id = reptip.reptip_Id
    INNER JOIN [Refugio].[tbRefugios] AS refg
        ON repa.refg_Id = refg.refg_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioCrea
        ON repa.repa_UsuarioCrea = usuarioCrea.usu_Id
    LEFT JOIN [Seguridad].[tbUsuarios] AS usuarioModifica
        ON repa.repa_UsuarioModifica = usuarioModifica.usu_Id
    WHERE repa.repa_EsEliminado != 1
    AND repa.repa_Id = @repa_Id
END
GO


# [Rescate].[PR_Rescate_ReportesAbandono_Insert]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportesAbandono_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportesAbandono_Insert]
    @reptip_Id INT,
    @repa_NombreReportante VARCHAR(150),
    @repa_TelefonoContacto VARCHAR(20),
    @repa_Email VARCHAR(100),
    @repa_FechaReporte DATETIME,
    @repa_UbicacionIncidente VARCHAR(200),
    @repa_DescripcionAnimal VARCHAR(300),
    @repa_EstadoAtencion VARCHAR(50),
    @repa_Observaciones VARCHAR(300),
    @repa_EsAnonimo BIT,
    @refg_Id INT,
    @repa_UsuarioCrea INT
AS
BEGIN
    INSERT INTO [Rescate].[tbReportesAbandono]
    (
        reptip_Id,
        repa_NombreReportante,
        repa_TelefonoContacto,
        repa_Email,
        repa_FechaReporte,
        repa_UbicacionIncidente,
        repa_DescripcionAnimal,
        repa_EstadoAtencion,
        repa_Observaciones,
        repa_EsAnonimo,
        refg_Id,
        repa_UsuarioCrea,
        repa_FechaCrea
    )
    VALUES
    (
        @reptip_Id,
        @repa_NombreReportante,
        @repa_TelefonoContacto,
        @repa_Email,
        @repa_FechaReporte,
        @repa_UbicacionIncidente,
        @repa_DescripcionAnimal,
        @repa_EstadoAtencion,
        @repa_Observaciones,
        @repa_EsAnonimo,
        @refg_Id,
        @repa_UsuarioCrea,
        GETDATE()
    )
END
GO


# [Rescate].[PR_Rescate_ReportesAbandono_List]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportesAbandono_List]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportesAbandono_List]
AS
BEGIN
    SELECT  ROW_NUMBER() OVER(ORDER BY repa.repa_FechaReporte DESC) AS Fila,
            repa.repa_Id,
            repa.reptip_Id,
            reptip.reptip_Descripcion AS TipoReportante,
            repa.repa_NombreReportante,
            repa.repa_TelefonoContacto,
            repa.repa_Email,
            repa.repa_FechaReporte,
            repa.repa_UbicacionIncidente,
            repa.repa_DescripcionAnimal,
            repa.repa_EstadoAtencion,
            repa.repa_Observaciones,
            repa.repa_EsAnonimo,
            repa.refg_Id,
            refg.refg_Nombre AS NombreRefugio
    FROM [Rescate].[tbReportesAbandono] AS repa
    INNER JOIN [Rescate].[tbReportantesTipo] AS reptip
        ON repa.reptip_Id = reptip.reptip_Id
    INNER JOIN [Refugio].[tbRefugios] AS refg
        ON repa.refg_Id = refg.refg_Id
    WHERE repa.repa_EsEliminado != 1
    ORDER BY repa.repa_FechaReporte DESC
END
GO


# [Rescate].[PR_Rescate_ReportesAbandono_Update]

In [0]:
/****** Object:  StoredProcedure [Rescate].[PR_Rescate_ReportesAbandono_Update]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Rescate].[PR_Rescate_ReportesAbandono_Update]
    @repa_Id INT,
    @reptip_Id INT,
    @repa_NombreReportante VARCHAR(150),
    @repa_TelefonoContacto VARCHAR(20),
    @repa_Email VARCHAR(100),
    @repa_FechaReporte DATETIME,
    @repa_UbicacionIncidente VARCHAR(200),
    @repa_DescripcionAnimal VARCHAR(300),
    @repa_EstadoAtencion VARCHAR(50),
    @repa_Observaciones VARCHAR(300),
    @repa_EsAnonimo BIT,
    @refg_Id INT,
    @repa_UsuarioModifica INT
AS
BEGIN
    UPDATE [Rescate].[tbReportesAbandono]
    SET reptip_Id = @reptip_Id,
        repa_NombreReportante = @repa_NombreReportante,
        repa_TelefonoContacto = @repa_TelefonoContacto,
        repa_Email = @repa_Email,
        repa_FechaReporte = @repa_FechaReporte,
        repa_UbicacionIncidente = @repa_UbicacionIncidente,
        repa_DescripcionAnimal = @repa_DescripcionAnimal,
        repa_EstadoAtencion = @repa_EstadoAtencion,
        repa_Observaciones = @repa_Observaciones,
        repa_EsAnonimo = @repa_EsAnonimo,
        refg_Id = @refg_Id,
        repa_UsuarioModifica = @repa_UsuarioModifica,
        repa_FechaModifica = GETDATE()
    WHERE repa_Id = @repa_Id
END
GO


# [Seguridad].[PR_CrearRolAdministrador]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_CrearRolAdministrador]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- PASO 10: PROCEDIMIENTO DE INICIALIZACIÓN
-- =============================================

-- SP: Crear rol de administrador con todos los permisos
CREATE   PROCEDURE [Seguridad].[PR_CrearRolAdministrador]
AS
BEGIN
    SET NOCOUNT ON;
    
    DECLARE @rol_Id INT;
    
    -- Crear o obtener rol de administrador
    IF NOT EXISTS (SELECT 1 FROM [Seguridad].[tbRoles] WHERE Rol_Descripcion = 'Administrador')
    BEGIN
        INSERT INTO [Seguridad].[tbRoles] (Rol_Descripcion, Rol_EsActivo, Rol_EsEliminado)
        VALUES ('Administrador', 1, 0);
        
        SET @rol_Id = SCOPE_IDENTITY();
    END
    ELSE
    BEGIN
        SELECT @rol_Id = Rol_Id FROM [Seguridad].[tbRoles] WHERE Rol_Descripcion = 'Administrador';
    END
    
    -- Asignar todas las pantallas al rol administrador
    INSERT INTO [Seguridad].[tbRolModulosPantallas] (modpt_Id, rol_Id)
    SELECT mp.modpt_Id, @rol_Id
    FROM [Seguridad].[tbModulosPantallas] mp
    WHERE NOT EXISTS (
        SELECT 1 FROM [Seguridad].[tbRolModulosPantallas] rmp 
        WHERE rmp.modpt_Id = mp.modpt_Id AND rmp.rol_Id = @rol_Id
    );
    
    -- Asignar todos los permisos al rol administrador
    INSERT INTO [Seguridad].[tbRolModuloPermisos] (Rol_Id, Mod_Id, Per_Id)
    SELECT @rol_Id, m.Mod_Id, p.Per_Id
    FROM [Seguridad].[tbModulos] m
    CROSS JOIN [Seguridad].[tbPermisos] p
    WHERE NOT EXISTS (
        SELECT 1 FROM [Seguridad].[tbRolModuloPermisos] rmp 
        WHERE rmp.Rol_Id = @rol_Id AND rmp.Mod_Id = m.Mod_Id AND rmp.Per_Id = p.Per_Id
    );
    
    PRINT 'Rol Administrador configurado con todos los permisos';
END
GO


# [Seguridad].[PR_MigrarPermisosExistentes]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_MigrarPermisosExistentes]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- PASO 9: PROCEDIMIENTO PARA MIGRAR DATOS EXISTENTES
-- =============================================

-- SP: Migrar permisos existentes al nuevo modelo
CREATE   PROCEDURE [Seguridad].[PR_MigrarPermisosExistentes]
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        BEGIN TRANSACTION;
        
        -- 1. Migrar relación directa Rol-Usuario (si no existe ya la tabla RolesUsuarios)
        IF EXISTS (SELECT * FROM sys.columns WHERE object_id = OBJECT_ID(N'[Seguridad].[tbUsuarios]') AND name = 'Rol_Id')
        BEGIN
            INSERT INTO [Seguridad].[tbRolesUsuarios] (rol_Id, usu_Id)
            SELECT DISTINCT Rol_Id, usu_Id
            FROM [Seguridad].[tbUsuarios]
            WHERE Rol_Id IS NOT NULL
                AND NOT EXISTS (
                    SELECT 1 FROM [Seguridad].[tbRolesUsuarios] ru 
                    WHERE ru.rol_Id = [Seguridad].[tbUsuarios].Rol_Id 
                        AND ru.usu_Id = [Seguridad].[tbUsuarios].usu_Id
                );
            
            PRINT 'Roles de usuarios migrados exitosamente';
        END
        
        -- 2. Limpiar tablas obsoletas si existen
        IF EXISTS (SELECT * FROM sys.objects WHERE object_id = OBJECT_ID(N'[Seguridad].[tbRolModulos]'))
        BEGIN
            -- Migrar datos de RolModulos a RolModulosPantallas si es necesario
            PRINT 'Tabla tbRolModulos encontrada - considerar migración manual si tiene datos importantes';
        END
        
        COMMIT TRANSACTION;
        PRINT 'Migración completada exitosamente';
        
    END TRY
    BEGIN CATCH
        ROLLBACK TRANSACTION;
        PRINT 'Error en la migración: ' + ERROR_MESSAGE();
        THROW;
    END CATCH
END
GO


# [Seguridad].[PR_Seguridad_PantallasPorRol]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_PantallasPorRol]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Obtener permisos por rol (pantallas accesibles)
CREATE   PROCEDURE [Seguridad].[PR_Seguridad_PantallasPorRol]
    @rol_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    -- Componentes accesibles
    SELECT DISTINCT
        c.comp_Id,
        c.comp_Descripcion
    FROM [Seguridad].[tbRolModulosPantallas] rmp
    INNER JOIN [Seguridad].[tbModulosPantallas] mp ON rmp.modpt_Id = mp.modpt_Id
    INNER JOIN [Seguridad].[tbModulos] m ON mp.mod_Id = m.Mod_Id
    INNER JOIN [Seguridad].[tbComponentes] c ON m.comp_Id = c.comp_Id
    WHERE rmp.rol_Id = @rol_Id AND mp.modpt_EsActivo = 1 AND m.Mod_EsActivo = 1;
    
    -- Módulos accesibles
    SELECT DISTINCT
        m.Mod_Id,
        m.Mod_Nombre,
        m.Mod_Descripcion,
        m.Mod_Icono,
        m.Mod_Orden,
        c.comp_Id,
        c.comp_Descripcion
    FROM [Seguridad].[tbRolModulosPantallas] rmp
    INNER JOIN [Seguridad].[tbModulosPantallas] mp ON rmp.modpt_Id = mp.modpt_Id
    INNER JOIN [Seguridad].[tbModulos] m ON mp.mod_Id = m.Mod_Id
    INNER JOIN [Seguridad].[tbComponentes] c ON m.comp_Id = c.comp_Id
    WHERE rmp.rol_Id = @rol_Id AND mp.modpt_EsActivo = 1 AND m.Mod_EsActivo = 1
    ORDER BY m.Mod_Orden, m.Mod_Nombre;
    
    -- Pantallas accesibles
    SELECT 
        mp.modpt_Id,
        mp.mod_Id,
        mp.modpt_Descripcion,
        mp.modpt_Url,
        mp.modpt_Icono,
        mp.modpt_Orden,
        m.Mod_Nombre,
        c.comp_Descripcion
    FROM [Seguridad].[tbRolModulosPantallas] rmp
    INNER JOIN [Seguridad].[tbModulosPantallas] mp ON rmp.modpt_Id = mp.modpt_Id
    INNER JOIN [Seguridad].[tbModulos] m ON mp.mod_Id = m.Mod_Id
    INNER JOIN [Seguridad].[tbComponentes] c ON m.comp_Id = c.comp_Id
    WHERE rmp.rol_Id = @rol_Id AND mp.modpt_EsActivo = 1 AND m.Mod_EsActivo = 1
    ORDER BY mp.modpt_Orden, mp.modpt_Descripcion;
END
GO


# [Seguridad].[PR_Seguridad_PantallasPorUsuario]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_PantallasPorUsuario]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Obtener permisos por usuario (considerando múltiples roles)
CREATE   PROCEDURE [Seguridad].[PR_Seguridad_PantallasPorUsuario]
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    -- Componentes accesibles
    SELECT DISTINCT
        c.comp_Id,
        c.comp_Descripcion
    FROM [Seguridad].[tbRolesUsuarios] ru
    INNER JOIN [Seguridad].[tbRolModulosPantallas] rmp ON ru.rol_Id = rmp.rol_Id
    INNER JOIN [Seguridad].[tbModulosPantallas] mp ON rmp.modpt_Id = mp.modpt_Id
    INNER JOIN [Seguridad].[tbModulos] m ON mp.mod_Id = m.Mod_Id
    INNER JOIN [Seguridad].[tbComponentes] c ON m.comp_Id = c.comp_Id
    WHERE ru.usu_Id = @usu_Id AND mp.modpt_EsActivo = 1 AND m.Mod_EsActivo = 1;
    
    -- Módulos accesibles
    SELECT DISTINCT
        m.Mod_Id,
        m.Mod_Nombre,
        m.Mod_Descripcion,
        m.Mod_Icono,
        m.Mod_Orden,
        c.comp_Id,
        c.comp_Descripcion
    FROM [Seguridad].[tbRolesUsuarios] ru
    INNER JOIN [Seguridad].[tbRolModulosPantallas] rmp ON ru.rol_Id = rmp.rol_Id
    INNER JOIN [Seguridad].[tbModulosPantallas] mp ON rmp.modpt_Id = mp.modpt_Id
    INNER JOIN [Seguridad].[tbModulos] m ON mp.mod_Id = m.Mod_Id
    INNER JOIN [Seguridad].[tbComponentes] c ON m.comp_Id = c.comp_Id
    WHERE ru.usu_Id = @usu_Id AND mp.modpt_EsActivo = 1 AND m.Mod_EsActivo = 1
    ORDER BY m.Mod_Orden, m.Mod_Nombre;
    
    -- Pantallas accesibles con permisos consolidados
    SELECT DISTINCT
        mp.modpt_Id,
        mp.mod_Id,
        mp.modpt_Descripcion,
        mp.modpt_Url,
        mp.modpt_Icono,
        mp.modpt_Orden,
        m.Mod_Nombre,
        c.comp_Descripcion,
        STRING_AGG(p.Per_Nombre, ',') AS Permisos
    FROM [Seguridad].[tbRolesUsuarios] ru
    INNER JOIN [Seguridad].[tbRolModulosPantallas] rmp ON ru.rol_Id = rmp.rol_Id
    INNER JOIN [Seguridad].[tbModulosPantallas] mp ON rmp.modpt_Id = mp.modpt_Id
    INNER JOIN [Seguridad].[tbModulos] m ON mp.mod_Id = m.Mod_Id
    INNER JOIN [Seguridad].[tbComponentes] c ON m.comp_Id = c.comp_Id
    LEFT JOIN [Seguridad].[tbRolModuloPermisos] rmpe ON rmp.rol_Id = rmpe.Rol_Id AND mp.mod_Id = rmpe.Mod_Id
    LEFT JOIN [Seguridad].[tbPermisos] p ON rmpe.Per_Id = p.Per_Id
    WHERE ru.usu_Id = @usu_Id AND mp.modpt_EsActivo = 1 AND m.Mod_EsActivo = 1
    GROUP BY mp.modpt_Id, mp.mod_Id, mp.modpt_Descripcion, mp.modpt_Url,
             mp.modpt_Icono, mp.modpt_Orden, m.Mod_Nombre, c.comp_Descripcion
    ORDER BY mp.modpt_Orden, mp.modpt_Descripcion;
END
GO


# [Seguridad].[PR_Seguridad_RegistrarAccesoPantalla]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_RegistrarAccesoPantalla]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Registrar acceso a pantalla
CREATE   PROCEDURE [Seguridad].[PR_Seguridad_RegistrarAccesoPantalla]
    @usu_Id INT,
    @modpt_Id INT,
    @UserAgent NVARCHAR(MAX) = NULL,
    @DireccionIP NVARCHAR(MAX) = NULL
AS
BEGIN
    SET NOCOUNT ON;
    
    DECLARE @pantalla NVARCHAR(100);
    DECLARE @modulo NVARCHAR(100);
    DECLARE @componente NVARCHAR(50);
    
    -- Obtener información de la pantalla
    SELECT 
        @pantalla = mp.modpt_Descripcion,
        @modulo = m.Mod_Nombre,
        @componente = c.comp_Descripcion
    FROM [Seguridad].[tbModulosPantallas] mp
    INNER JOIN [Seguridad].[tbModulos] m ON mp.mod_Id = m.Mod_Id
    INNER JOIN [Seguridad].[tbComponentes] c ON m.comp_Id = c.comp_Id
    WHERE mp.modpt_Id = @modpt_Id;
    
    -- Registrar evento
    INSERT INTO [Seguridad].[tbRegistroEventos]
    (Tpevt_Id, Evt_Usu_Id, Evt_Detalles, Evt_UserAgent, Evt_DireccionIP, 
     Evt_Pantalla, Evt_Modulo, Evt_Componente, Evt_FechaCreacion)
    VALUES (8, @usu_Id, 'Acceso a pantalla: ' + @pantalla, @UserAgent, @DireccionIP,
            @pantalla, @modulo, @componente, GETDATE());
END
GO


# [Seguridad].[PR_Seguridad_RegistroEventos_Find]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_RegistroEventos_Find]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Seguridad].[PR_Seguridad_RegistroEventos_Find] 
    @Evt_Id INT
AS
BEGIN
  SELECT 
         evt.Evt_Id,
         tpevt.Tpevt_Descripcion AS Tipo_Evento,
         evt.Evt_Usu_Id,
         evt.Evt_Detalles, 
         evt.Evt_UserAgent, 
         evt.Evt_DireccionIP,
         evt.Evt_EstadoAnterior,
         evt.Evt_NuevoEstado,
         evt.Evt_FechaCreacion
    FROM [Seguridad].[tbRegistroEventos] evt INNER JOIN  [Seguridad].[tbTipoEventos] tpevt
    ON   evt.Tpevt_Id=tpevt.Tpevt_Id
    WHERE evt.Evt_Id = @Evt_Id
END
GO


# [Seguridad].[PR_Seguridad_RegistroEventos_Insert]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_RegistroEventos_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Seguridad].[PR_Seguridad_RegistroEventos_Insert] 
    @Tpevt_Id             INT,
    @Evt_Usu_Id           INT,
    @Evt_Detalles         NVARCHAR(MAX), 
	@Evt_UserAgent        NVARCHAR(MAX), 
    @Evt_DireccionIP      NVARCHAR(MAX),
	@Evt_EstadoAnterior   NVARCHAR(MAX),
    @Evt_NuevoEstado      NVARCHAR(MAX),
    @Evt_FechaCreacion    DATETIME
AS
BEGIN
INSERT INTO [Seguridad].[tbRegistroEventos]
    (
        Tpevt_Id ,
        Evt_Usu_Id ,
        Evt_Detalles, 
		Evt_UserAgent, 
        Evt_DireccionIP,
		Evt_EstadoAnterior ,
        Evt_NuevoEstado,
        Evt_FechaCreacion
    )
    VALUES
    (
         @Tpevt_Id ,
		 @Evt_Usu_Id ,
		 @Evt_Detalles , 
		 @Evt_UserAgent,
		 @Evt_DireccionIP ,
		 @Evt_EstadoAnterior ,
         @Evt_NuevoEstado,
         @Evt_FechaCreacion
     )

END
GO


# [Seguridad].[PR_Seguridad_RegistroEventos_Select]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_RegistroEventos_Select]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

/*SECCIÓN #83*/
CREATE PROCEDURE [Seguridad].[PR_Seguridad_RegistroEventos_Select] 
AS
BEGIN
  SELECT 
		 Evt_Id,
		 TP.Tpevt_Descripcion,
		 Evt_Usu_Id,
		 Evt_Detalles, 
		 Evt_UserAgent, 
		 Evt_DireccionIP,
		 CASE
		 WHEN LEN(Evt_EstadoAnterior) >130  THEN (SUBSTRING(Evt_EstadoAnterior, 1, 130)+'...')
		 ELSE   Evt_EstadoAnterior
		 END AS Evt_EstadoAnterior,
		 CASE
		 WHEN LEN(Evt_NuevoEstado) >130  THEN (SUBSTRING(Evt_NuevoEstado, 1, 130)+'...')
		 ELSE   Evt_NuevoEstado
		 END AS Evt_NuevoEstado,
		 Evt_FechaCreacion
    FROM [Seguridad].[tbRegistroEventos] eve INNER JOIN [Seguridad].[tbTipoEventos] tp
    ON   eve.Tpevt_Id=TP.Tpevt_Id
END
GO


# [Seguridad].[PR_Seguridad_Roles_DependencyRoles]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Roles_DependencyRoles]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 1. GESTIÓN DE DEPENDENCIAS DE ROLES
-- =============================================

CREATE PROCEDURE [Seguridad].[PR_Seguridad_Roles_DependencyRoles]
    @Rol_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    -- Verificar si el rol tiene usuarios asignados
    SELECT ru.rol_Id, COUNT(*) as TotalUsuarios
    FROM [Seguridad].[tbRolesUsuarios] ru
    INNER JOIN [Seguridad].[tbUsuarios] u ON ru.usu_Id = u.usu_Id
    WHERE ru.rol_Id = @Rol_Id
        AND u.Usu_EsActivo = 1
        AND ISNULL(u.Usu_EsEliminado, 0) = 0
    GROUP BY ru.rol_Id;
END
GO


# [Seguridad].[PR_Seguridad_Roles_DependencyUsuarios]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Roles_DependencyUsuarios]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Seguridad].[PR_Seguridad_Roles_DependencyUsuarios]
    @Rol_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    -- Obtener usuarios que tienen este rol asignado
    SELECT 
        u.usu_Id,
        u.Usu_Nombre,
        p.per_PrimerNombre + ' ' + ISNULL(p.per_SegundoNombre, '') AS Nombres,
        p.per_ApellidoPaterno + ' ' + ISNULL(p.per_ApellidoMaterno, '') AS Apellidos,
        ru.rol_Id
    FROM [Seguridad].[tbRolesUsuarios] ru
    INNER JOIN [Seguridad].[tbUsuarios] u ON ru.usu_Id = u.usu_Id
    INNER JOIN [Refugio].[tbEmpleados] e ON u.Emp_Id = e.emp_Id
    INNER JOIN [General].[tbPersonas] p ON e.per_Id = p.per_Id
    WHERE ru.rol_Id = @Rol_Id 
        AND u.Usu_EsActivo = 1
        AND ISNULL(u.Usu_EsEliminado, 0) = 0;
END
GO


# [Seguridad].[PR_Seguridad_Roles_FindRol]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Roles_FindRol]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Seguridad].[PR_Seguridad_Roles_FindRol]
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        Rol_Id,
        Rol_Descripcion
    FROM [Seguridad].[tbRoles]
    WHERE Rol_EsActivo = 1
        AND ISNULL(Rol_EsEliminado, 0) = 0
    ORDER BY Rol_Descripcion;
END
GO


# [Seguridad].[PR_Seguridad_Roles_UpdateEstado]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Roles_UpdateEstado]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
-- 2. ACTUALIZACIÓN DE ESTADO DE ROLES
-- =============================================

CREATE PROCEDURE [Seguridad].[PR_Seguridad_Roles_UpdateEstado]
    @Rol_Id INT,
    @Rol_EsActivo BIT,
    @Usu_UsuarioModifica INT = NULL
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        -- Verificar dependencias antes de desactivar
        IF @Rol_EsActivo = 0
        BEGIN
            DECLARE @UsuariosAsignados INT;
            SELECT @UsuariosAsignados = COUNT(*)
            FROM [Seguridad].[tbRolesUsuarios] ru
            INNER JOIN [Seguridad].[tbUsuarios] u ON ru.usu_Id = u.usu_Id
            WHERE ru.rol_Id = @Rol_Id 
                AND u.Usu_EsActivo = 1
                AND ISNULL(u.Usu_EsEliminado, 0) = 0;
            
            IF @UsuariosAsignados > 0
            BEGIN
                SELECT 0 AS Success, 'No se puede desactivar el rol porque tiene usuarios asignados' AS Message;
                RETURN;
            END
        END
        
        UPDATE [Seguridad].[tbRoles]
        SET Rol_EsActivo = @Rol_EsActivo
            --Rol_UsuarioModifica = @Usu_UsuarioModifica,
            --Rol_FechaModifica = GETDATE()
        WHERE Rol_Id = @Rol_Id;
        
        SELECT 1 AS Success, 'Estado actualizado correctamente' AS Message;
    END TRY
    BEGIN CATCH
        SELECT 0 AS Success, ERROR_MESSAGE() AS Message;
    END CATCH
END
GO


# [Seguridad].[PR_Seguridad_Roles_ValidacionUnique]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Roles_ValidacionUnique]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
-- 3. VALIDACIONES ÚNICAS
-- =============================================

CREATE PROCEDURE [Seguridad].[PR_Seguridad_Roles_ValidacionUnique]
    @Rol_Descripcion NVARCHAR(50),
    @Rol_Id INT = NULL -- Para validar en actualizaciones
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT Rol_Id, Rol_Descripcion
    FROM [Seguridad].[tbRoles]
    WHERE Rol_Descripcion = @Rol_Descripcion
        AND Rol_EsActivo = 1
        AND ISNULL(Rol_EsEliminado, 0) = 0
        AND (@Rol_Id IS NULL OR Rol_Id <> @Rol_Id);
END
GO


# [Seguridad].[PR_Seguridad_RolesUsuarios_Delete]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_RolesUsuarios_Delete]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Remover rol de usuario
CREATE   PROCEDURE [Seguridad].[PR_Seguridad_RolesUsuarios_Delete]
    @rol_Id INT,
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    DELETE FROM [Seguridad].[tbRolesUsuarios]
    WHERE rol_Id = @rol_Id AND usu_Id = @usu_Id;
    
    -- Registrar evento
    INSERT INTO [Seguridad].[tbRegistroEventos]
    (Tpevt_Id, Evt_Usu_Id, Evt_Detalles, Evt_FechaCreacion)
    VALUES (7, @usu_Id, 'Remoción de rol ID: ' + CAST(@rol_Id AS NVARCHAR(10)), GETDATE());
    
    SELECT @@ROWCOUNT AS FilasAfectadas;
END
GO


# [Seguridad].[PR_Seguridad_RolesUsuarios_Insert]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_RolesUsuarios_Insert]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Asignar rol a usuario
CREATE   PROCEDURE [Seguridad].[PR_Seguridad_RolesUsuarios_Insert]
    @rol_Id INT,
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    -- Verificar si ya existe la asignación
    IF NOT EXISTS (SELECT 1 FROM [Seguridad].[tbRolesUsuarios] WHERE rol_Id = @rol_Id AND usu_Id = @usu_Id)
    BEGIN
        INSERT INTO [Seguridad].[tbRolesUsuarios] (rol_Id, usu_Id)
        VALUES (@rol_Id, @usu_Id);
        
        -- Registrar evento
        INSERT INTO [Seguridad].[tbRegistroEventos]
        (Tpevt_Id, Evt_Usu_Id, Evt_Detalles, Evt_FechaCreacion)
        VALUES (7, @usu_Id, 'Asignación de rol ID: ' + CAST(@rol_Id AS NVARCHAR(10)), GETDATE());
        
        SELECT SCOPE_IDENTITY() AS rol_usu_Id, 'Rol asignado exitosamente' AS Mensaje;
    END
    ELSE
    BEGIN
        SELECT 0 AS rol_usu_Id, 'El usuario ya tiene este rol asignado' AS Mensaje;
    END
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_FindNombreUsuario]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_FindNombreUsuario]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_FindNombreUsuario]
    @Usu_Nombre NVARCHAR(150)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        u.Usu_Nombre,
        u.usu_Id,
        u.Usu_EsActivo,
        u.Usu_Suspendido
    FROM [Seguridad].[tbUsuarios] u
    WHERE u.Usu_Nombre = @Usu_Nombre
        AND ISNULL(u.Usu_EsEliminado, 0) = 0;
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_Login]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_Login]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
-- Procedimiento para login de usuario con permisos de rol y módulo
CREATE   PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_Login] 
    @Usu_Nombre NVARCHAR(150),
    @Con_Hash NVARCHAR(255)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        u.usu_Id,
        u.Emp_Id,
        u.Usu_Nombre,
        p.per_PrimerNombre + ' ' + p.per_SegundoNombre Emp_Nombres,
        p.per_ApellidoPaterno Emp_Apellidos,
        u.Rol_Id,
        r.Rol_Descripcion, 
         STRING_AGG(p.per_PrimerNombre, ', ') AS Permisos,
        -- OR, if you prefer the permission names:
        -- STRING_AGG(DISTINCT perm.Per_Nombre, ',') WITHIN GROUP (ORDER BY perm.Per_Nombre) AS PermisosNombresAsignados,
		perm.Per_Nombre,
        u.Usu_EsActivo,
        u.Usu_Suspendido
        --c.Con_Hash,
        --c.Con_Salt
    FROM tbUsuarios u
    INNER JOIN refugio.tbEmpleados e ON u.Emp_Id = e.Emp_Id
    INNER JOIN seguridad.tbRoles r ON u.Rol_Id = r.Rol_Id
	INNER JOIN General.tbPersonas p ON e.per_Id = p.per_Id 
    LEFT JOIN Seguridad.tbRolModuloPermisos rmp ON u.Rol_Id = rmp.Rol_Id -- Join con la tabla de permisos
    -- If you want permission names, uncomment the next line and the corresponding STRING_AGG line above:
     LEFT JOIN Seguridad.tbPermisos perm ON rmp.Per_Id = perm.Per_Id
    WHERE u.Usu_Nombre = @Usu_Nombre 
        AND u.Usu_EsActivo = 1 
        AND u.Usu_Suspendido = 0
        AND u.Usu_EsEliminado = 0
        --AND c.Con_Hash = @Con_Hash
        --AND c.Con_EsActivo = 1;
    GROUP BY
        u.usu_Id,
        u.Emp_Id,
        u.Usu_Nombre,
        p.per_PrimerNombre,
        p.per_SegundoNombre,
        p.per_ApellidoPaterno,
        u.Rol_Id,
perm.Per_Nombre,
        r.Rol_Descripcion,
        u.Usu_EsActivo,
        u.Usu_Suspendido;
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_Login_V2]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_Login_V2]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- =============================================
-- PASO 4: PROCEDIMIENTOS ALMACENADOS
-- =============================================

-- SP: Login mejorado con registro de eventos
CREATE   PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_Login_V2]
    @Usu_Nombre NVARCHAR(150),
    @Con_Hash NVARCHAR(255),
    @UserAgent NVARCHAR(MAX) = NULL,
    @DireccionIP NVARCHAR(MAX) = NULL
AS
BEGIN
    SET NOCOUNT ON;
    
    DECLARE @usu_Id INT;
    DECLARE @intentosFallidos INT;
    DECLARE @fechaBloqueo DATETIME;
    DECLARE @resultado INT = 0;
    
    -- Verificar si el usuario existe
    SELECT @usu_Id = usu_Id, 
           @intentosFallidos = ISNULL(usu_IntentosFallidos, 0),
           @fechaBloqueo = usu_FechaBloqueo
    FROM [Seguridad].[tbUsuarios]
    WHERE Usu_Nombre = @Usu_Nombre;
    
    -- Si no existe el usuario
    IF @usu_Id IS NULL
    BEGIN
        -- Registrar intento fallido
        INSERT INTO [Seguridad].[tbRegistroEventos]
        (Tpevt_Id, Evt_Detalles, Evt_UserAgent, Evt_DireccionIP, Evt_FechaCreacion)
        VALUES (3, 'Intento de acceso con usuario inexistente: ' + @Usu_Nombre, 
                @UserAgent, @DireccionIP, GETDATE());
        
        SELECT @resultado AS Resultado, 'Usuario no encontrado' AS Mensaje;
        RETURN;
    END
    
    -- Verificar si está bloqueado
    IF @fechaBloqueo IS NOT NULL AND DATEDIFF(MINUTE, @fechaBloqueo, GETDATE()) < 30
    BEGIN
        SELECT @resultado AS Resultado, 'Usuario bloqueado temporalmente' AS Mensaje;
        RETURN;
    END
    
    -- Verificar credenciales
    IF EXISTS (
        SELECT 1
        FROM [Seguridad].[tbUsuarios] u
        INNER JOIN [Refugio].[tbEmpleados] e ON u.Emp_Id = e.emp_Id
        WHERE u.usu_Id = @usu_Id 
            AND u.Usu_PasswordHash = @Con_Hash
            AND u.Usu_EsActivo = 1 
            AND ISNULL(u.Usu_Suspendido, 0) = 0
            AND ISNULL(u.Usu_EsEliminado, 0) = 0
            AND e.emp_EsActivo = 1
    )
    BEGIN
        -- Login exitoso
        UPDATE [Seguridad].[tbUsuarios]
        SET usu_Logueado = 1,
            usu_UltimoAcceso = GETDATE(),
            usu_IntentosFallidos = 0,
            usu_FechaBloqueo = NULL
        WHERE usu_Id = @usu_Id;
        
        -- Registrar evento de login
        INSERT INTO [Seguridad].[tbRegistroEventos]
        (Tpevt_Id, Evt_Usu_Id, Evt_Detalles, Evt_UserAgent, Evt_DireccionIP, Evt_FechaCreacion)
        VALUES (1, @usu_Id, 'Inicio de sesión exitoso', @UserAgent, @DireccionIP, GETDATE());
        
        -- Retornar datos del usuario con sus permisos
        SELECT 
            u.usu_Id,
            u.Emp_Id,
            u.Usu_Nombre,
            p.per_PrimerNombre + ' ' + ISNULL(p.per_SegundoNombre, '') AS Emp_Nombres,
            p.per_ApellidoPaterno + ' ' + ISNULL(p.per_ApellidoMaterno, '') AS Emp_Apellidos,
            u.usu_ImagenPerfil,
            1 AS Resultado,
            'Login exitoso' AS Mensaje
        FROM [Seguridad].[tbUsuarios] u
        INNER JOIN [Refugio].[tbEmpleados] e ON u.Emp_Id = e.emp_Id
        INNER JOIN [General].[tbPersonas] p ON e.per_Id = p.per_Id 
        WHERE u.usu_Id = @usu_Id;
        
        -- Retornar roles del usuario
        SELECT DISTINCT
            ru.rol_Id,
            r.Rol_Descripcion
        FROM [Seguridad].[tbRolesUsuarios] ru
        INNER JOIN [Seguridad].[tbRoles] r ON ru.rol_Id = r.Rol_Id
        WHERE ru.usu_Id = @usu_Id AND r.Rol_EsActivo = 1;
        
    END
    ELSE
    BEGIN
        -- Login fallido
        UPDATE [Seguridad].[tbUsuarios]
        SET usu_IntentosFallidos = @intentosFallidos + 1,
            usu_FechaBloqueo = CASE WHEN @intentosFallidos + 1 >= 3 THEN GETDATE() ELSE NULL END
        WHERE usu_Id = @usu_Id;
        
        -- Registrar evento de intento fallido
        INSERT INTO [Seguridad].[tbRegistroEventos]
        (Tpevt_Id, Evt_Usu_Id, Evt_Detalles, Evt_UserAgent, Evt_DireccionIP, Evt_FechaCreacion)
        VALUES (3, @usu_Id, 'Intento de acceso fallido', @UserAgent, @DireccionIP, GETDATE());
        
        SELECT @resultado AS Resultado, 
               CASE WHEN @intentosFallidos + 1 >= 3 
                    THEN 'Usuario bloqueado por múltiples intentos fallidos' 
                    ELSE 'Credenciales incorrectas' END AS Mensaje;
    END
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_LoginIn]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_LoginIn]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_LoginIn]
    @usu_Id INT,
    @UserAgent NVARCHAR(MAX) = NULL,
    @DireccionIP NVARCHAR(MAX) = NULL
AS
BEGIN
    SET NOCOUNT ON;
    
    UPDATE [Seguridad].[tbUsuarios]
    SET usu_Logueado = 1,
        usu_UltimoAcceso = GETDATE()
    WHERE usu_Id = @usu_Id;
    
    -- Registrar evento de login
    INSERT INTO [Seguridad].[tbRegistroEventos]
    (Tpevt_Id, Evt_Usu_Id, Evt_Detalles, Evt_UserAgent, Evt_DireccionIP, Evt_FechaCreacion)
    VALUES (1, @usu_Id, 'Usuario marcado como logueado', @UserAgent, @DireccionIP, GETDATE());
    
    SELECT 'Usuario marcado como logueado exitosamente' AS Message;
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_Logout]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_Logout]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- SP: Logout de usuario
CREATE   PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_Logout]
    @usu_Id INT,
    @UserAgent NVARCHAR(MAX) = NULL,
    @DireccionIP NVARCHAR(MAX) = NULL
AS
BEGIN
    SET NOCOUNT ON;
    
    -- Actualizar estado del usuario
    UPDATE [Seguridad].[tbUsuarios]
    SET usu_Logueado = 0
    WHERE usu_Id = @usu_Id;
    
    -- Registrar evento
    INSERT INTO [Seguridad].[tbRegistroEventos]
    (Tpevt_Id, Evt_Usu_Id, Evt_Detalles, Evt_UserAgent, Evt_DireccionIP, Evt_FechaCreacion)
    VALUES (2, @usu_Id, 'Cierre de sesión', @UserAgent, @DireccionIP, GETDATE());
    
    SELECT 'Sesión cerrada exitosamente' AS Mensaje;
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_NameValidation]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_NameValidation]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_NameValidation]
    @Usu_Nombre NVARCHAR(150)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        u.Usu_Nombre,
        u.usu_Id
    FROM [Seguridad].[tbUsuarios] u
    WHERE u.Usu_Nombre = @Usu_Nombre
        AND u.Usu_EsActivo = 1
        AND ISNULL(u.Usu_EsEliminado, 0) = 0;
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_RecuperarContra]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_RecuperarContra]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_RecuperarContra]
    @Usu_Nombre NVARCHAR(150)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        u.usu_Id,
        u.Usu_Nombre,
        p.per_Correo,
        p.per_PrimerNombre,
        p.per_ApellidoPaterno,
        p.per_Id
    FROM [Seguridad].[tbUsuarios] u
    INNER JOIN [Refugio].[tbEmpleados] e ON u.Emp_Id = e.emp_Id
    INNER JOIN [General].[tbPersonas] p ON e.per_Id = p.per_Id
    WHERE u.Usu_Nombre = @Usu_Nombre
        AND u.Usu_EsActivo = 1
        AND ISNULL(u.Usu_Suspendido, 0) = 0
        AND ISNULL(u.Usu_EsEliminado, 0) = 0;
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_SelectByLogin]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_SelectByLogin]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
-- 7. BÚSQUEDA DE USUARIOS POR LOGIN
-- =============================================

CREATE PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_SelectByLogin]
    @Usu_Nombre NVARCHAR(150)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        u.usu_Id,
        u.Usu_Nombre,
        u.Usu_EsActivo,
        CASE WHEN u.Usu_EsActivo = 1 THEN 'Activo' ELSE 'Inactivo' END AS Estado,
        p.per_Id,
        p.per_PrimerNombre + ' ' + ISNULL(p.per_SegundoNombre, '') + ' ' + 
        p.per_ApellidoPaterno + ' ' + ISNULL(p.per_ApellidoMaterno, '') AS PersonaNombre,
        STRING_AGG(r.Rol_Descripcion, ', ') AS Roles
    FROM [Seguridad].[tbUsuarios] u
    INNER JOIN [Refugio].[tbEmpleados] e ON u.Emp_Id = e.emp_Id
    INNER JOIN [General].[tbPersonas] p ON e.per_Id = p.per_Id
    LEFT JOIN [Seguridad].[tbRolesUsuarios] ru ON u.usu_Id = ru.usu_Id
    LEFT JOIN [Seguridad].[tbRoles] r ON ru.rol_Id = r.Rol_Id AND r.Rol_EsActivo = 1
    WHERE u.Usu_Nombre = @Usu_Nombre
        AND ISNULL(u.Usu_EsEliminado, 0) = 0
    GROUP BY u.usu_Id, u.Usu_Nombre, u.Usu_EsActivo, p.per_Id,
             p.per_PrimerNombre, p.per_SegundoNombre, p.per_ApellidoPaterno, p.per_ApellidoMaterno;
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_SelectByUsuId]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_SelectByUsuId]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 8. OBTENER DATOS DE USUARIO POR ID
-- =============================================

CREATE PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_SelectByUsuId]
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        u.usu_Id,
        p.per_Id,
        p.per_Correo,
        p.per_PrimerNombre + ' ' + p.per_ApellidoPaterno AS NombreCompleto
    FROM [Seguridad].[tbUsuarios] u
    INNER JOIN [Refugio].[tbEmpleados] e ON u.Emp_Id = e.emp_Id
    INNER JOIN [General].[tbPersonas] p ON e.per_Id = p.per_Id
    WHERE u.usu_Id = @usu_Id
        AND u.Usu_EsActivo = 1
        AND ISNULL(u.Usu_EsEliminado, 0) = 0;
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_UpdateLog]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_UpdateLog]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_UpdateLog]
    @usu_Id INT,
    @log BIT,
    @UserAgent NVARCHAR(MAX) = NULL,
    @DireccionIP NVARCHAR(MAX) = NULL
AS
BEGIN
    SET NOCOUNT ON;
    
    UPDATE [Seguridad].[tbUsuarios]
    SET usu_Logueado = @log
    WHERE usu_Id = @usu_Id;
    
    -- Registrar evento
    DECLARE @detalle NVARCHAR(100) = CASE WHEN @log = 1 THEN 'Usuario logueado' ELSE 'Usuario deslogueado' END;
    DECLARE @tipoEvento INT = CASE WHEN @log = 1 THEN 1 ELSE 2 END;
    
    INSERT INTO [Seguridad].[tbRegistroEventos]
    (Tpevt_Id, Evt_Usu_Id, Evt_Detalles, Evt_UserAgent, Evt_DireccionIP, Evt_FechaCreacion)
    VALUES (@tipoEvento, @usu_Id, @detalle, @UserAgent, @DireccionIP, GETDATE());
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_UpdatePassword]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_UpdatePassword]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
-- 4. GESTIÓN DE CONTRASEÑAS
-- =============================================

CREATE PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_UpdatePassword]
    @usu_Id INT,
    @Usu_PasswordHash NVARCHAR(255),
    @Usu_PasswordSalt NVARCHAR(255) = NULL,
    @UserAgent NVARCHAR(MAX) = NULL,
    @DireccionIP NVARCHAR(MAX) = NULL
AS
BEGIN
    SET NOCOUNT ON;
    
    BEGIN TRY
        UPDATE [Seguridad].[tbUsuarios]
        SET Usu_PasswordHash = @Usu_PasswordHash,
            Usu_PasswordSalt = @Usu_PasswordSalt
            --usu_UltimoActualizacionPassword = GETDATE()
        WHERE usu_Id = @usu_Id;
        
        -- Registrar evento de cambio de contraseña
        INSERT INTO [Seguridad].[tbRegistroEventos]
        (Tpevt_Id, Evt_Usu_Id, Evt_Detalles, Evt_UserAgent, Evt_DireccionIP, Evt_FechaCreacion)
        VALUES (6, @usu_Id, 'Cambio de contraseña', @UserAgent, @DireccionIP, GETDATE());
        
        SELECT 1 AS Success, 'Contraseña actualizada correctamente' AS Message;
    END TRY
    BEGIN CATCH
        SELECT 0 AS Success, ERROR_MESSAGE() AS Message;
    END CATCH
END
GO


# [Seguridad].[PR_Seguridad_Usuarios_ValidationUnique]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[PR_Seguridad_Usuarios_ValidationUnique]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO
CREATE PROCEDURE [Seguridad].[PR_Seguridad_Usuarios_ValidationUnique]
    @Usu_Nombre NVARCHAR(150),
    @usu_Id INT = NULL -- Para validar en actualizaciones
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT u.usu_Id, u.Usu_Nombre, p.per_Identidad
    FROM [Seguridad].[tbUsuarios] u
    INNER JOIN [Refugio].[tbEmpleados] e ON u.Emp_Id = e.emp_Id
    INNER JOIN [General].[tbPersonas] p ON e.per_Id = p.per_Id
    WHERE u.Usu_Nombre = @Usu_Nombre
        AND u.Usu_EsActivo = 1
        AND ISNULL(u.Usu_EsEliminado, 0) = 0
        AND (@usu_Id IS NULL OR u.usu_Id <> @usu_Id);
END
GO


# [Seguridad].[UDP_Acce_PantallasXRol]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[UDP_Acce_PantallasXRol]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 3. Obtener pantallas por rol simplificado como AHM (UDP_Acce_PantallasXRol)
CREATE   PROCEDURE [Seguridad].[UDP_Acce_PantallasXRol]
    @rol_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT DISTINCT
        mp.modpt_Id,
        mp.modpt_Descripcion,
        mp.modpt_Url,
        mp.modpt_Icono,
        mp.modpt_Orden,
        mp.mod_Id,
        m.Mod_Nombre,
        m.Mod_Descripcion AS Mod_Descripcion,
        m.Mod_Icono AS Mod_Icono,
        m.Mod_Orden AS Mod_Orden
    FROM [Seguridad].[tbRolModulosPantallas] rmp
    INNER JOIN [Seguridad].[tbModulosPantallas] mp ON rmp.modpt_Id = mp.modpt_Id
    INNER JOIN [Seguridad].[tbModulos] m ON mp.mod_Id = m.Mod_Id
    WHERE rmp.rol_Id = @rol_Id 
        AND mp.modpt_EsActivo = 1 
        AND m.Mod_EsActivo = 1
    ORDER BY m.Mod_Orden, mp.modpt_Orden, mp.modpt_Descripcion;
END
GO


# [Seguridad].[UDP_Acce_PantallasXUsuario]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[UDP_Acce_PantallasXUsuario]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 4. Helpers para obtener pantallas por usuario (compatible con múltiples roles)
CREATE   PROCEDURE [Seguridad].[UDP_Acce_PantallasXUsuario]
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT DISTINCT
        mp.modpt_Id,
        mp.modpt_Descripcion,
        mp.modpt_Url,
        mp.modpt_Icono,
        mp.modpt_Orden,
        mp.mod_Id,
        m.Mod_Nombre,
        m.Mod_Descripcion AS Mod_Descripcion,
        m.Mod_Icono AS Mod_Icono,
        m.Mod_Orden AS Mod_Orden
    FROM [Seguridad].[tbRolesUsuarios] ru
    INNER JOIN [Seguridad].[tbRolModulosPantallas] rmp ON ru.rol_Id = rmp.rol_Id
    INNER JOIN [Seguridad].[tbModulosPantallas] mp ON rmp.modpt_Id = mp.modpt_Id
    INNER JOIN [Seguridad].[tbModulos] m ON mp.mod_Id = m.Mod_Id
    WHERE ru.usu_Id = @usu_Id 
        AND mp.modpt_EsActivo = 1 
        AND m.Mod_EsActivo = 1
    ORDER BY m.Mod_Orden, mp.modpt_Orden, mp.modpt_Descripcion;
END
GO


# [Seguridad].[UDP_Acce_tbUsuarios_FindDetalle]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[UDP_Acce_tbUsuarios_FindDetalle]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 6. Procedimiento para obtener detalle de usuario
CREATE   PROCEDURE [Seguridad].[UDP_Acce_tbUsuarios_FindDetalle]
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        u.usu_Id,
        u.Emp_Id,
        u.Usu_Nombre AS usu_NombreUsuario,
        CONCAT(p.per_PrimerNombre, ' ', ISNULL(p.per_SegundoNombre, ''), ' ', 
               p.per_ApellidoPaterno, ' ', ISNULL(p.per_ApellidoMaterno, '')) AS usu_NombreCompleto,
        p.per_PrimerNombre,
        p.per_ApellidoPaterno,
        -- Obtener el rol principal
        (SELECT TOP 1 ru.rol_Id 
         FROM [Seguridad].[tbRolesUsuarios] ru 
         WHERE ru.usu_Id = u.usu_Id 
         ORDER BY ru.rol_usu_FechaAsignacion ASC) AS rol_Id,
        (SELECT TOP 1 r.Rol_Descripcion 
         FROM [Seguridad].[tbRolesUsuarios] ru 
         INNER JOIN [Seguridad].[tbRoles] r ON ru.rol_Id = r.Rol_Id
         WHERE ru.usu_Id = u.usu_Id 
         ORDER BY ru.rol_usu_FechaAsignacion ASC) AS rol_Descripcion,
        u.Usu_EsActivo AS usu_Estado,
        u.usu_ImagenPerfil,
        u.usu_Logueado,
        u.usu_UltimoAcceso,
        u.Usu_FechaCreacion AS usu_FechaCreacion
    FROM [Seguridad].[tbUsuarios] u
    INNER JOIN [Refugio].[tbEmpleados] e ON u.Emp_Id = e.emp_Id
    INNER JOIN [General].[tbPersonas] p ON e.per_Id = p.per_Id
    WHERE u.usu_Id = @usu_Id 
        AND ISNULL(u.Usu_EsEliminado, 0) = 0;
END
GO


# [Seguridad].[UDP_Acce_tbUsuarios_Login]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[UDP_Acce_tbUsuarios_Login]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 1. Login simplificado como AHM (UDP_Acce_tbUsuarios_Login)
CREATE   PROCEDURE [Seguridad].[UDP_Acce_tbUsuarios_Login]
    @usu_NombreUsuario NVARCHAR(150),
    @contrasena NVARCHAR(255)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        u.usu_Id,
        u.Emp_Id,
        u.Usu_Nombre AS usu_NombreUsuario,
        CONCAT(p.per_PrimerNombre, ' ', ISNULL(p.per_SegundoNombre, ''), ' ', 
               p.per_ApellidoPaterno, ' ', ISNULL(p.per_ApellidoMaterno, '')) AS usu_NombreCompleto,
        p.per_PrimerNombre,
        p.per_ApellidoPaterno,
        -- Obtener el rol principal (el primero si tiene múltiples)
        (SELECT TOP 1 ru.rol_Id 
         FROM [Seguridad].[tbRolesUsuarios] ru 
         WHERE ru.usu_Id = u.usu_Id 
         ORDER BY ru.rol_usu_FechaAsignacion ASC) AS rol_Id,
        (SELECT TOP 1 r.Rol_Descripcion 
         FROM [Seguridad].[tbRolesUsuarios] ru 
         INNER JOIN [Seguridad].[tbRoles] r ON ru.rol_Id = r.Rol_Id
         WHERE ru.usu_Id = u.usu_Id 
         ORDER BY ru.rol_usu_FechaAsignacion ASC) AS rol_Descripcion,
        u.Usu_EsActivo AS usu_Estado,
        u.usu_ImagenPerfil,
        u.usu_Logueado
    FROM [Seguridad].[tbUsuarios] u
    INNER JOIN [Refugio].[tbEmpleados] e ON u.Emp_Id = e.emp_Id
    INNER JOIN [General].[tbPersonas] p ON e.per_Id = p.per_Id
    WHERE u.Usu_Nombre = @usu_NombreUsuario 
        AND u.Usu_PasswordHash = @contrasena
        AND u.Usu_EsActivo = 1 
        AND ISNULL(u.Usu_Suspendido, 0) = 0
        AND ISNULL(u.Usu_EsEliminado, 0) = 0
        AND e.emp_EsActivo = 1
        -- Verificar que el usuario tenga al menos un rol activo
        AND EXISTS (
            SELECT 1 FROM [Seguridad].[tbRolesUsuarios] ru 
            INNER JOIN [Seguridad].[tbRoles] r ON ru.rol_Id = r.Rol_Id
            WHERE ru.usu_Id = u.usu_Id AND r.Rol_EsActivo = 1
        );
END
GO


# [Seguridad].[UDP_Acce_tbUsuarios_LoginIn]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[UDP_Acce_tbUsuarios_LoginIn]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 2. Login/Logout State Management como AHM
CREATE   PROCEDURE [Seguridad].[UDP_Acce_tbUsuarios_LoginIn]
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    UPDATE [Seguridad].[tbUsuarios]
    SET usu_Logueado = 1,
        usu_UltimoAcceso = GETDATE(),
        usu_IntentosFallidos = 0,
        usu_FechaBloqueo = NULL
    WHERE usu_Id = @usu_Id;
    
    -- Registrar evento de login
    INSERT INTO [Seguridad].[tbRegistroEventos]
    (Tpevt_Id, Evt_Usu_Id, Evt_Detalles, Evt_FechaCreacion)
    VALUES (1, @usu_Id, 'Usuario inició sesión', GETDATE());
    
    SELECT 0 AS Resultado; -- 0 = éxito en AHM
END
GO


# [Seguridad].[UDP_Acce_tbUsuarios_Logout]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[UDP_Acce_tbUsuarios_Logout]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

CREATE   PROCEDURE [Seguridad].[UDP_Acce_tbUsuarios_Logout]
    @usu_Id INT
AS
BEGIN
    SET NOCOUNT ON;
    
    UPDATE [Seguridad].[tbUsuarios]
    SET usu_Logueado = 0
    WHERE usu_Id = @usu_Id;
    
    -- Registrar evento de logout
    INSERT INTO [Seguridad].[tbRegistroEventos]
    (Tpevt_Id, Evt_Usu_Id, Evt_Detalles, Evt_FechaCreacion)
    VALUES (2, @usu_Id, 'Usuario cerró sesión', GETDATE());
    
    SELECT 0 AS Resultado; -- 0 = éxito en AHM
END
GO


# [Seguridad].[UDP_Acce_tbUsuarios_NameValidation]

In [0]:
/****** Object:  StoredProcedure [Seguridad].[UDP_Acce_tbUsuarios_NameValidation]    Script Date: 2/21/2026 11:59:19 PM ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

-- 5. Procedimiento para validar usuario (compatible con AHM)
CREATE   PROCEDURE [Seguridad].[UDP_Acce_tbUsuarios_NameValidation]
    @usu_UsuarioNombre NVARCHAR(150)
AS
BEGIN
    SET NOCOUNT ON;
    
    SELECT 
        usu_Id,
        usu_Nombre AS usu_NombreUsuario,
        Emp_Id,
        Usu_EsActivo AS usu_Estado
    FROM [Seguridad].[tbUsuarios]
    WHERE Usu_Nombre = @usu_UsuarioNombre 
        AND ISNULL(Usu_EsEliminado, 0) = 0;
END
GO
